# Cross-Sectional Equity Alpha Research

**A point-in-time backtesting framework for US large-cap equities, with
realistic transaction costs and out-of-sample validation.**

---

## Executive summary

This project tests whether a cross-sectional signal built from publicly
available daily price data can generate risk-adjusted returns in large-cap US
equities after realistic trading costs.

**Headline result.** A beta-neutral composite signal run at $25M returns
**+2.56%/yr at a Sharpe of 0.55**, with a bootstrap confidence interval of
**[0.11, 0.99]** that excludes zero, and an annualised alpha of **+2.13%
(t = 1.72)**. Adding it to a passive equity portfolio at a 45% allocation
lifts that portfolio's Sharpe from **1.06 to 1.16**.

**Honest caveats, stated upfront.** The alpha t-statistic is below the
conventional significance bar. Around 116 strategy variants were scored
against the development sample, and the deflated Sharpe that accounts for that
search does not clear 95%. The result is capacity-limited to roughly $100M,
and 35% of point-in-time constituents were unavailable from the data source —
disproportionately delisted names, which biases the short leg.

**What the work is really worth.** Seven measurement biases were found and
corrected in the author's own results, two of which were inflating Sharpe by
1.8×. The project also documents **alpha decay**: the same signal earned
+4.14% alpha in 2010–2014 and −1.26% in 2020–2023.

---

## Methodology at a glance

| | |
|---|---|
| **Universe** | Point-in-time S&P 500 membership, 2005–present |
| **Validation** | Walk-forward with a 21-session embargo; 32 months sealed as an untouched holdout |
| **Target** | 21-day forward return, sector-demeaned, rank-transformed |
| **Costs** | Square-root market impact scaled by participation in each name's ADV, 10bp spread, 50bp annual short borrow |
| **Capacity** | Positions capped at 5% of each name's average daily volume |
| **Construction** | Sector-neutral, rank-proportional weights across the full cross-section, risk-parity sizing, beta-neutral legs |
| **Inference** | Deflated Sharpe ratio adjusted for 116 searched variants; block-bootstrap confidence intervals |

## How to read this notebook

**Part this work** builds the data and the model. **Part II** constructs the portfolio
and the candidate strategies. **Part III** reports results, capacity and
robustness. **Part IV** is validation — significance testing, survivorship
exposure, and the single-use out-of-sample holdout. **Appendix A** is the
research log: every experiment run during development and what each concluded,
including the ones that failed.

Run the cells in order. Everything network-bound is cached, so a re-run takes
minutes rather than the 30–60 of a cold start.

In [ ]:
# ============================================================
# COLAB SETUP -- run once per session
# ============================================================
!pip install -q yfinance xgboost hmmlearn statsmodels shap
print("dependencies installed -- restart the runtime only if Colab asks one to")

---
# Part this work — Data and Model

## 1. Environment, Configuration and Caching

All parameters live in a single `Config` dataclass so that every assumption —
cost model, universe, holdout date, construction choices — is visible in one
place and reproducible. Network calls are cached to disk.

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
from __future__ import annotations

import os, re, json, time, pickle, hashlib, warnings, logging
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import norm
from scipy.special import gammaln
import requests

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 13

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
log = logging.getLogger("alpha_signal")

try:
    import yfinance as yf
    HAS_YFINANCE = True
    logging.getLogger("yfinance").setLevel(logging.CRITICAL)
except ImportError:
    HAS_YFINANCE = False
    log.error("yfinance is not installed. Run: !pip install yfinance")

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    log.warning("xgboost not found -- falling back to sklearn HistGradientBoosting.")

try:
    import statsmodels.api as sm
    HAS_SM = True
except ImportError:
    HAS_SM = False
    log.warning("statsmodels not found. Run: !pip install statsmodels "
                "(needed for Newey-West t-stats and factor attribution).")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

try:
    from hmmlearn.hmm import GaussianHMM
    HAS_HMM = True
except ImportError:
    HAS_HMM = False
    log.warning("hmmlearn not found. Run: !pip install hmmlearn (needed for the regime section).")

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import HistGradientBoostingRegressor

print(f"yfinance={HAS_YFINANCE}  xgboost={HAS_XGB}  statsmodels={HAS_SM}  "
      f"shap={HAS_SHAP}  hmmlearn={HAS_HMM}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
STATIC_UNIVERSE_SECTORS = {
    "AAPL": "Technology", "MSFT": "Technology", "GOOGL": "Technology", "META": "Technology",
    "NVDA": "Technology", "AVGO": "Technology", "ORCL": "Technology", "CRM": "Technology",
    "AMZN": "Consumer Discretionary", "TSLA": "Consumer Discretionary", "MCD": "Consumer Discretionary",
    "NKE": "Consumer Discretionary", "HD": "Consumer Discretionary", "LOW": "Consumer Discretionary",
    "TGT": "Consumer Discretionary",
    "JPM": "Financials", "BAC": "Financials", "WFC": "Financials", "GS": "Financials",
    "MS": "Financials", "C": "Financials", "SCHW": "Financials", "BLK": "Financials",
    "SPGI": "Financials", "AXP": "Financials",
    "JNJ": "Healthcare", "UNH": "Healthcare", "PFE": "Healthcare", "MRK": "Healthcare",
    "ABBV": "Healthcare", "LLY": "Healthcare", "TMO": "Healthcare", "ABT": "Healthcare",
    "DHR": "Healthcare", "BMY": "Healthcare",
    "PG": "Consumer Staples", "KO": "Consumer Staples", "PEP": "Consumer Staples",
    "WMT": "Consumer Staples", "COST": "Consumer Staples",
    "XOM": "Energy", "CVX": "Energy", "COP": "Energy", "SLB": "Energy", "EOG": "Energy",
    "CAT": "Industrials", "HON": "Industrials", "UPS": "Industrials", "BA": "Industrials",
    "GE": "Industrials",
}


@dataclass
class Config:
    # ---- universe & dates ----
    universe: list = field(default_factory=lambda: list(STATIC_UNIVERSE_SECTORS.keys()))
    # 2005 instead of 2014. This is the one legitimate way to raise a
    # t-statistic -- more observations, not more variants. Takes the
    # walk-forward from ~72 test months to ~180 and adds the 2008 crisis.
    start_date: str = "2005-01-01"
    end_date: str = datetime.today().strftime("%Y-%m-%d")
    universe_mode: str = "point_in_time"
    pit_reconstitution_freq: str = "YS"
    sp500_history_url: str = (
        "https://raw.githubusercontent.com/fja05680/sp500/master/"
        "S%26P%20500%20Historical%20Components%20%26%20Changes%20(Updated).csv"
    )
    technical_warmup_days: int = 370
    forward_buffer_days: int = 60

    # ---- target & training ----
    forward_return_days: int = 21
    min_train_periods: int = 48
    min_universe_breadth: int = 150
    include_fundamentals_in_model: bool = False  # yfinance coverage is 0-8%, see the feature-scaling step gate
    include_reversal_features: bool = True      # short-term reversal, absent from an earlier implementation
    include_v14_features: bool = True           # MAX, 52w-high, ivol, beta, illiquidity, skew
    min_feature_column_coverage: float = 0.50   # drop any feature below this coverage
    min_sector_coverage: float = 0.85           # HARD gate -- an earlier implementation silently ran at 33.4%
    allow_low_sector_coverage: bool = False     # explicit override, not a warning one can ignore
    train_on_rank_target: bool = True           # bound the fat-tailed target
    # short lookback + frequent refresh. The controlled test showed a 12m
    # lookback refit MONTHLY beats the same lookback refit annually by ~4x at
    # realistic signal-to-noise, because a model held 12 months is stale by
    # month 11 on a relationship that moves.
    # Expanding window. In the lookback sweep it was the only setting that did
    # not lose money (IC -0.0005 against -0.0227 at a 24-month window), and the
    # measured rotation is slow enough (12.9-month directional half-life) that
    # estimation noise from a short window costs more than the staleness it
    # avoids.
    # Set from the lookback sweep's marginal effects. Averaging over the other
    # dimension, an expanding window was the best lookback (IC -0.0024 against
    # -0.0223 at 24 months) and a 3-month use period the best cadence (-0.0056
    # against -0.0160 monthly and -0.0104 annual): long history, refreshed
    # quarterly.
    rolling_train_months: Optional[int] = None  # None = expanding window
    refresh_months: int = 3                     # use each fitted model for 3 months
    headline_aum: float = 25_000_000.0          # capacity-realistic scorecard row
    # drawdown overlay. Scale the book down while it sits more than
    # `dd_trigger` below its own high-water mark. Tested in the drawdown-overlay test -- it reliably
    # cuts drawdown, and only improves Sharpe if losses cluster.
    # the risk-sizing test: equalise each name's RISK contribution rather than its signal
    # weight. Measured +0.22 Sharpe in a controlled test. This is the version
    # of "lower volatility" that raises Sharpe; scaling the book down does not.
    risk_parity: bool = True
    risk_parity_vol_col: str = "vol_21_raw"
    # alternatives tested and left OFF by default -- see the leg-balance test. Downside
    # deviation correlates +0.92 with total volatility, so swapping one for the
    # other is very nearly the same book; and once names are risk-weighted the
    # two legs are already close to risk-balanced. Both are exposed anyway,
    # because the synthetic test cannot know the panel's actual skew dispersion.
    risk_parity_mode: str = "vol"          # "vol" | "downside"
    risk_parity_dvol_col: str = "dvol_63_raw"
    leg_risk_parity: bool = False          # equalise RISK across the two legs
    # leverage policy. "symmetric" is the classic vol target (scale up AND
    # down); "onesided" only ever scales DOWN, which keeps the risk-reduction
    # half without the cost explosion from levering a weak book. the leverage-policy test compares.
    vol_target_mode: str = "onesided"      # off | symmetric | onesided | onesided_downside
    vol_target_floor: float = 0.25
    dd_delever: bool = False               # additionally cut exposure while in drawdown
    dd_delever_trigger: float = -0.05
    dd_delever_scale: float = 0.5
    dd_overlay: bool = False
    dd_trigger: float = -0.05
    dd_scale: float = 0.5
    n_seeds: int = 3                            # rank-average a seed ensemble
    min_feature_coverage: float = 0.6           # drop warm-up rows

    # ---- portfolio ----
    n_quantiles: int = 5
    sector_neutral: bool = True
    use_signal_weighted_sizing: bool = True
    turnover_control_enabled: bool = True
    turnover_entry_pct: float = 0.20
    turnover_exit_pct: float = 0.35
    target_annual_volatility: Optional[float] = 0.10
    vol_target_lookback_months: int = 12
    # capped at 1.0. An earlier implementation ran at mean leverage 1.83x on a book whose gross
    # return was -1.81%. Vol targeting levers up a LOW-VOLATILITY book, which
    # multiplies the dollars traded -- and impact scales as dollars^1.5, so cost
    # grows faster than leverage. Measured: 10.63%/yr of cost at 1.83x against
    # ~4.28% at 1.0x, turning -12.9% into roughly -6.1%. Levering a strategy
    # with no demonstrated edge only multiplies its costs. the leverage sweep sweeps this.
    max_vol_target_leverage: float = 1.0

    # ---- costs ----
    transaction_cost_bps: float = 10.0
    assumed_aum_usd: float = 500_000_000.0
    max_position_pct_adv: Optional[float] = 0.05
    market_impact_bps_per_sqrt_pct_participation: float = 8.0
    short_borrow_cost_bps_annual: float = 50.0

    # ---- research hygiene ----
    holdout_start: str = "2024-01-01"   # locked. do not move after seeing results.
    # the holdout is now OPENED. 92 variants have been scored, the sub-period test shows
    # the effect has decayed, and there is nothing left worth tuning. A
    # completed pre-registered out-of-sample test is the right way to finish.
    reveal_holdout: bool = True
    n_trials_searched: int = 116  # +24 for the rebuilt the lookback sweep grid         # be honest; feeds the deflated Sharpe
    # research knobs
    adaptive_sign: bool = True                  # T9: decide long/short orientation walk-forward
    sign_lookback_months: int = 24
    ml_holding_months: int = 1                  # T11: rebalance the ML book every N months
    # the portfolio, not the signal, is the binding constraint.
    #   "quintile"     -- An earlier implementation behaviour: equal weight across two extreme buckets
    #   "proportional" -- weight by within-sector rank across the FULL cross-section
    weighting_scheme: str = "proportional"
    proportional_power: float = 1.0             # >1 tilts back toward the tails
    # Naive rank weighting gives EVERY name a position, so every name trades
    # every month and cost jumps. Two controls, both swept in T15:
    proportional_min_rank: float = 0.0          # trim destroys middle-information
    weight_smoothing: float = 1.0               # the trim/smoothing sweep showed smoothing destroys
                                                # gross faster than it saves cost
    # long-tilted books. gross_long / gross_short set the two legs
    # independently, so 1.0/0.0 is long-only and 1.0/0.3 is a 130/30-style tilt.
    gross_long: float = 1.0
    gross_short: float = 1.0
    # size the short leg for BETA neutrality, not dollar neutrality.
    # An earlier implementation measured -0.21 beta on a supposedly market-neutral book, which cost
    # ~3%/yr in a +14%/yr market and buried a +2.52% alpha.
    beta_neutral: bool = True
    beta_col: str = "beta_63_raw"
    beta_scale_bounds: tuple = (0.25, 4.0)
    # the in-sample/out-of-sample check found in-sample IC +0.139 vs out-of-sample -0.019 -- the model
    # memorises. Capacity is cut hard here and the capacity sweep sweeps it properly.
    model_max_depth: int = 2
    model_n_estimators: int = 80
    model_learning_rate: float = 0.03
    model_min_child_weight: float = 50.0
    model_colsample: float = 0.4

    # ---- infrastructure ----
    fmp_api_key: Optional[str] = None
    cache_dir: str = "./v13_cache"
    force_refresh: bool = False
    max_workers: int = 12
    random_state: int = 42


CFG = Config()
CFG.holdout_start = pd.Timestamp(CFG.holdout_start)
np.random.seed(CFG.random_state)
os.makedirs(CFG.cache_dir, exist_ok=True)

print(f"Backtest window : {CFG.start_date} -> {CFG.end_date}")
print(f"Holdout starts  : {CFG.holdout_start.date()}  (reveal={CFG.reveal_holdout})")
print(f"Cache directory : {os.path.abspath(CFG.cache_dir)}")

In [ ]:
# ============================================================
# CACHING AND RESULT COLLECTION
# ============================================================
# every network call is cached to disk. The first run pays for Yahoo;
# every run after that is near-instant. Set CFG.force_refresh = True to re-pull.

def _cache_path(name: str, key: str = "") -> str:
    tag = hashlib.md5(f"{name}|{key}".encode()).hexdigest()[:10]
    return os.path.join(CFG.cache_dir, f"{name}_{tag}.pkl")


def cached(name: str, key: str, builder):
    """Return a cached object if present, else build it and cache it."""
    path = _cache_path(name, key)
    if not CFG.force_refresh and os.path.exists(path):
        try:
            with open(path, "rb") as f:
                obj = pickle.load(f)
            log.info(f"cache hit: {name}  ({os.path.basename(path)})")
            return obj
        except Exception as e:
            log.warning(f"cache read failed for {name} ({e}) -- rebuilding.")
    obj = builder()
    try:
        with open(path, "wb") as f:
            pickle.dump(obj, f)
        log.info(f"cached: {name} -> {os.path.basename(path)}")
    except Exception as e:
        log.warning(f"cache write failed for {name}: {e}")
    return obj


def clear_cache():
    n = 0
    for f in os.listdir(CFG.cache_dir):
        if f.endswith(".pkl"):
            os.remove(os.path.join(CFG.cache_dir, f)); n += 1
    print(f"cleared {n} cache files")


# ---- test result collector (the final cell prints this) -------------------
REPORT = {"config": {"start": CFG.start_date, "end": CFG.end_date,
                     "holdout_start": str(CFG.holdout_start.date()),
                     "horizon_days": CFG.forward_return_days,
                     "rank_target": CFG.train_on_rank_target,
                     "reversal_features": CFG.include_reversal_features}}


def rec(key, value):
    REPORT[key] = value
    return value


def _j(x, nd=4):
    """JSON-safe rounding helper."""
    try:
        if x is None or (isinstance(x, float) and not np.isfinite(x)):
            return None
        return round(float(x), nd)
    except Exception:
        return None


print("cache + report collector ready")

## 2. Data Acquisition

**Point-in-time universe construction** is the foundation. Building a universe
from today's index members and running it backwards would silently exclude
every company that failed — the classic survivorship bias. Instead, index
membership is reconstructed as of each date, and each ticker's price history
is fetched only for the window in which it was genuinely a constituent.

Fundamentals carry a 45-day reporting lag, because a company reporting its
March quarter does not publish until mid-May; merging on period end would let
the backtest trade on information six weeks before it existed.

In [ ]:
# ============================================================
# POINT-IN-TIME UNIVERSE AND MEMBERSHIP WINDOWS
# ============================================================
# Retained unchanged; this was correct. The membership source is used twice:
#   1) membership_calendar is the exact PIT filter applied to the panel later
#   2) membership_windows gives each ticker its real constituent window, so a
#      historical-only name is only downloaded for the period it was a member
#      (plus technical warm-up and forward-return buffers).

def _parse_membership_history(cfg: "Config") -> pd.DataFrame:
    hist = pd.read_csv(cfg.sp500_history_url)
    if "date" not in hist.columns or "tickers" not in hist.columns:
        raise ValueError("Historical S&P 500 source must contain 'date' and 'tickers'.")
    hist["date"] = pd.to_datetime(hist["date"], errors="coerce")
    hist = hist.dropna(subset=["date"]).sort_values("date")
    hist["tickers"] = hist["tickers"].fillna("").apply(
        lambda s: frozenset(
            re.sub(r"-\d{6}$", "", str(t).strip()).upper()
            for t in str(s).split(",") if str(t).strip()
        )
    )
    hist = hist.groupby("date", as_index=False)["tickers"].last().sort_values("date")

    start, end = pd.Timestamp(cfg.start_date), pd.Timestamp(cfg.end_date)
    prior = hist[hist["date"] < start].tail(1)
    in_window = hist[(hist["date"] >= start) & (hist["date"] <= end)]
    hist = (pd.concat([prior, in_window], ignore_index=True)
            .drop_duplicates("date").sort_values("date"))
    if hist.empty:
        raise ValueError("No membership snapshots overlap the configured window.")
    return hist.set_index("date")[["tickers"]]


def build_membership_calendar(cfg):
    try:
        return _parse_membership_history(cfg)
    except Exception as e:
        log.warning(f"Could not build membership calendar: {e}")
        return None


def build_membership_windows(calendar: pd.DataFrame, cfg: "Config") -> pd.DataFrame:
    """First/last in-window constituent date for every PIT ticker."""
    start, end = pd.Timestamp(cfg.start_date), pd.Timestamp(cfg.end_date)
    rows = []
    for snap_date, members in calendar["tickers"].items():
        snap_date = max(pd.Timestamp(snap_date), start)
        if snap_date > end:
            continue
        for ticker in members:
            rows.append({"ticker": ticker, "date": snap_date})
    if not rows:
        raise ValueError("Membership source produced no in-window observations.")

    obs = pd.DataFrame(rows).drop_duplicates(["ticker", "date"])
    snapshot_dates = calendar.index.sort_values()
    latest = snapshot_dates[snapshot_dates <= end].max()
    current_members = set(calendar.loc[latest, "tickers"]) if pd.notna(latest) else set()

    # A constituent stays a member until the NEXT snapshot removes it.
    next_snap = {d: (snapshot_dates[i + 1] if i + 1 < len(snapshot_dates)
                     else end + pd.Timedelta(days=1))
                 for i, d in enumerate(snapshot_dates)}

    out = []
    for ticker, g in obs.groupby("ticker"):
        s = g["date"].min()
        last_snap = g["date"].max()
        if ticker in current_members:
            e, is_cur = end, True
        else:
            e = min(next_snap.get(last_snap, end + pd.Timedelta(days=1)) - pd.Timedelta(days=1), end)
            is_cur = False
        out.append({"ticker": ticker,
                    "membership_start": max(pd.Timestamp(s), start),
                    "membership_end": max(pd.Timestamp(s), pd.Timestamp(e)),
                    "is_current": is_cur})
    return pd.DataFrame(out).sort_values("ticker").reset_index(drop=True)


if CFG.universe_mode == "point_in_time":
    membership_calendar = cached("membership", CFG.start_date + CFG.end_date,
                                 lambda: build_membership_calendar(CFG))
    membership_windows = build_membership_windows(membership_calendar, CFG)
    active_universe = membership_windows["ticker"].tolist()
else:
    active_universe = list(CFG.universe)
    membership_calendar = None
    membership_windows = pd.DataFrame({"ticker": active_universe,
                                       "membership_start": pd.Timestamp(CFG.start_date),
                                       "membership_end": pd.Timestamp(CFG.end_date),
                                       "is_current": True})

print(f"PIT universe: {len(active_universe)} tickers "
      f"({int(membership_windows['is_current'].sum())} current, "
      f"{int((~membership_windows['is_current']).sum())} historical-only)")

In [ ]:
# ============================================================
# PRICE DATA (BATCHED DOWNLOAD)
# ============================================================
# An earlier implementation made one yf.download call per ticker with a 0.15s sleep between each --
# 784 sequential round trips. yfinance accepts a ticker list, so this issues
# ~8 requests instead. Each name is still sliced back to its own PIT window
# afterwards, so the point-in-time property is unchanged.

def _normalize_yahoo_frame(raw) -> Optional[pd.DataFrame]:
    if raw is None or len(raw) == 0:
        return None
    try:
        if isinstance(raw.columns, pd.MultiIndex):
            if "Close" in raw.columns.get_level_values(0):
                raw = raw.copy(); raw.columns = raw.columns.get_level_values(0)
            elif "Close" in raw.columns.get_level_values(1):
                raw = raw.copy(); raw.columns = raw.columns.get_level_values(1)
        if "Close" not in raw.columns:
            return None
        out = raw[["Close"]].copy()
        out["Volume"] = raw["Volume"] if "Volume" in raw.columns else np.nan
        out = out.dropna(subset=["Close"])
        idx = pd.to_datetime(out.index)
        try:
            idx = idx.tz_localize(None)
        except TypeError:
            pass
        out.index = idx
        return out[~out.index.duplicated(keep="last")].sort_index()
    except Exception:
        return None


def _normalize_ticker_for_yahoo(ticker: str) -> str:
    """Yahoo uses a hyphen for share-class tickers (BRK-B); the S&P membership
    source uses a dot (BRK.B). Unconverted dot-tickers were previously
    miscounted as permanently delisted."""
    return ticker.replace(".", "-")


def _download_one_yahoo(ticker, start, end, max_retries=2):
    """Single-ticker fallback. Retries only on an actual exception -- a clean
    empty response means the ticker genuinely has no data and retrying an
    identical request will not change that."""
    for attempt in range(1, max_retries + 1):
        try:
            raw = yf.download(_normalize_ticker_for_yahoo(ticker), start=start, end=end,
                              auto_adjust=True, threads=False, progress=False, timeout=20)
            df = _normalize_yahoo_frame(raw)
            return df if (df is not None and not df.empty) else None
        except Exception as e:
            if attempt >= max_retries:
                log.debug(f"Yahoo failed for {ticker}: {e}")
            else:
                time.sleep(1.0 * attempt)
    return None


def download_price_data_batched(tickers, windows_df, cfg, chunk_size=100):
    if not HAS_YFINANCE:
        raise RuntimeError("yfinance is required. No synthetic data path exists.")
    w = windows_df.set_index("ticker")
    tickers = sorted(set(tickers))

    def window_for(t):
        cur = bool(w.loc[t, "is_current"])
        s = (pd.Timestamp(cfg.start_date) if cur else
             pd.Timestamp(w.loc[t, "membership_start"]) - pd.Timedelta(days=cfg.technical_warmup_days))
        e = ((pd.Timestamp(cfg.end_date) if cur else pd.Timestamp(w.loc[t, "membership_end"]))
             + pd.Timedelta(days=cfg.forward_buffer_days))
        return s, e

    price_data, diagnostics = {}, []
    n_chunks = (len(tickers) + chunk_size - 1) // chunk_size
    for i in range(0, len(tickers), chunk_size):
        chunk = tickers[i:i + chunk_size]
        ysyms = [_normalize_ticker_for_yahoo(t) for t in chunk]
        bounds = [window_for(t) for t in chunk]
        lo, hi = min(b[0] for b in bounds), max(b[1] for b in bounds)
        log.info(f"Price batch {i // chunk_size + 1}/{n_chunks}: {len(chunk)} tickers, "
                 f"{lo.date()} -> {hi.date()}")
        try:
            raw = yf.download(ysyms, start=lo.strftime("%Y-%m-%d"), end=hi.strftime("%Y-%m-%d"),
                              auto_adjust=True, group_by="ticker", threads=True,
                              progress=False, timeout=45)
        except Exception as e:
            log.warning(f"Batch failed ({e}); falling back to per-ticker for this chunk.")
            raw = None

        for ysym, t, (s, e) in zip(ysyms, chunk, bounds):
            df = None
            if raw is not None and len(raw):
                try:
                    sub = raw[ysym] if isinstance(raw.columns, pd.MultiIndex) else raw
                    df = _normalize_yahoo_frame(sub)
                except Exception:
                    df = None
            if df is None or df.empty:
                df = _download_one_yahoo(t, s.strftime("%Y-%m-%d"), e.strftime("%Y-%m-%d"))
            if df is not None and not df.empty:
                df = df.loc[(df.index >= s) & (df.index <= e)]     # re-impose PIT window
            ok = df is not None and not df.empty
            if ok:
                price_data[t] = df
            diagnostics.append({
                "ticker": t, "requested_start": s, "requested_end": e,
                "membership_start": pd.Timestamp(w.loc[t, "membership_start"]),
                "membership_end": pd.Timestamp(w.loc[t, "membership_end"]),
                "is_current": bool(w.loc[t, "is_current"]),
                "yahoo_status": "recovered" if ok else "unavailable_from_yahoo",
                "price_rows": len(df) if ok else 0,
                "first_price": df.index.min() if ok else pd.NaT,
                "last_price": df.index.max() if ok else pd.NaT,
            })
    return price_data, pd.DataFrame(diagnostics)


_key = f"{CFG.start_date}|{CFG.end_date}|{len(active_universe)}"
price_data, price_diagnostic = cached(
    "prices", _key, lambda: download_price_data_batched(active_universe, membership_windows, CFG))

recovered_tickers = sorted(price_data.keys())
print(f"\nPrices: {len(recovered_tickers)} recovered / {len(active_universe)} requested "
      f"({len(active_universe) - len(recovered_tickers)} unavailable)")

In [ ]:
# ============================================================
# FUNDAMENTALS WITH REPORTING LAG
# ============================================================
# Point-in-time discipline: every fundamental row gets an `available_date` 45
# days after the period end, because a company reporting Q1 on March 31 does
# not publish until mid-May. Merging on period_end instead would be look-ahead
# bias -- trading on numbers six weeks before they existed.
REPORTING_LAG_DAYS = 45
FUNDAMENTAL_FEATURE_COLS = ["pe_ratio", "pb_ratio", "roe", "profit_margin", "debt_to_equity"]


def _fetch_fmp_one(t, api_key):
    url = (f"https://financialmodelingprep.com/api/an earlier implementation/ratios/{t}"
           f"?period=quarter&limit=80&apikey={api_key}")
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    return [{"ticker": t, "period_end": pd.to_datetime(d.get("date")),
             "pe_ratio": d.get("priceEarningsRatio"), "pb_ratio": d.get("priceToBookRatio"),
             "roe": d.get("returnOnEquity"), "profit_margin": d.get("netProfitMargin"),
             "debt_to_equity": d.get("debtEquityRatio")} for d in resp.json()]


def _fetch_yf_fundamentals_one(t):
    rows = []
    tk = yf.Ticker(_normalize_ticker_for_yahoo(t))
    fin, bal = tk.quarterly_financials, tk.quarterly_balance_sheet
    if fin is None or fin.empty or bal is None or bal.empty:
        return rows
    for period_end in fin.columns:
        try:
            ni = fin.loc["Net Income", period_end] if "Net Income" in fin.index else np.nan
            rev = fin.loc["Total Revenue", period_end] if "Total Revenue" in fin.index else np.nan
            eq = bal.loc["Stockholders Equity", period_end] if "Stockholders Equity" in bal.index else np.nan
            debt = bal.loc["Total Debt", period_end] if "Total Debt" in bal.index else np.nan
            rows.append({
                "ticker": t, "period_end": pd.Timestamp(period_end),
                "roe": (ni / eq) if pd.notna(eq) and eq != 0 else np.nan,
                "profit_margin": (ni / rev) if pd.notna(rev) and rev != 0 else np.nan,
                "debt_to_equity": (debt / eq) if pd.notna(eq) and eq != 0 else np.nan,
                "pe_ratio": np.nan, "pb_ratio": np.nan,
            })
        except Exception:
            continue
    return rows


def get_fundamentals(tickers, fmp_api_key=None, max_workers=None):
    """Real fundamentals only. Failure never fabricates values -- it returns an
    empty frame and the model runs on technicals alone."""
    max_workers = max_workers or CFG.max_workers
    fetch = ((lambda t: _fetch_fmp_one(t, fmp_api_key)) if fmp_api_key else _fetch_yf_fundamentals_one)
    rows, failures = [], 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(fetch, t): t for t in tickers}
        for n, fut in enumerate(as_completed(futs), 1):
            try:
                rows.extend(fut.result())
            except Exception:
                failures += 1
            if n % 150 == 0:
                log.info(f"  fundamentals {n}/{len(tickers)}")
    if not rows:
        log.warning("No real fundamental history recovered. Continuing on technicals only.")
        return pd.DataFrame(columns=["ticker", "period_end"] + FUNDAMENTAL_FEATURE_COLS + ["available_date"])
    df = pd.DataFrame(rows).dropna(subset=["period_end"])
    df["available_date"] = df["period_end"] + pd.Timedelta(days=REPORTING_LAG_DAYS)
    log.info(f"Fundamentals: {len(df):,} rows across {df['ticker'].nunique()} tickers "
             f"({failures} fetch failures)")
    return df


fundamentals_pit = cached("fundamentals", f"{len(recovered_tickers)}|{bool(CFG.fmp_api_key)}",
                          lambda: get_fundamentals(recovered_tickers, CFG.fmp_api_key))
print(f"Fundamental observations: {len(fundamentals_pit):,}")

In [ ]:
# ============================================================
# ============================================================
# SECTOR CLASSIFICATION
# ============================================================
# Two defects corrected here.
# (1) TAXONOMY MISMATCH, present from the first implementation. The hard-coded map says
#     "Financials" / "Consumer Discretionary" / "Consumer Staples"; yfinance
#     returns "Financial Services" / "Consumer Cyclical" / "Consumer Defensive".
#     Earlier output showed both sets side by side -- 25 tickers in "Financial
#     Services" and 10 in "Financials" -- treated as DIFFERENT sectors. Every
#     sector-neutral operation was therefore comparing each stock to half its
#     real peer group. Both label sets now normalise to one taxonomy.
# (2) SILENT COVERAGE COLLAPSE. An earlier implementation ran at 33.4% coverage because yfinance
#     rate-limited .info; the gate logged an error and the run continued. A
#     warning that does not halt is a warning that gets ignored, so it now
#     raises. Fetching is also gentler (fewer workers, backoff) and cached
#     per ticker, so a partial run is never thrown away.

SECTOR_CANONICAL = {
    "financial services": "Financials", "financials": "Financials",
    "consumer cyclical": "Consumer Discretionary",
    "consumer discretionary": "Consumer Discretionary",
    "consumer defensive": "Consumer Staples", "consumer staples": "Consumer Staples",
    "technology": "Technology", "information technology": "Technology",
    "healthcare": "Health Care", "health care": "Health Care",
    "industrials": "Industrials", "energy": "Energy",
    "utilities": "Utilities", "real estate": "Real Estate",
    "basic materials": "Materials", "materials": "Materials",
    "communication services": "Communication Services",
}


def canonical_sector(raw):
    if not raw or not isinstance(raw, str):
        return "Unknown"
    return SECTOR_CANONICAL.get(raw.strip().lower(), raw.strip())


_SECTOR_CACHE = os.path.join(CFG.cache_dir, "sector_by_ticker.json")


def _load_sector_cache():
    if CFG.force_refresh or not os.path.exists(_SECTOR_CACHE):
        return {}
    try:
        with open(_SECTOR_CACHE) as f:
            return json.load(f)
    except Exception:
        return {}


def _save_sector_cache(d):
    try:
        with open(_SECTOR_CACHE, "w") as f:
            json.dump(d, f)
    except Exception as e:
        log.warning(f"sector cache write failed: {e}")


def _sector_one(t, retries=3):
    if t in STATIC_UNIVERSE_SECTORS:
        return t, canonical_sector(STATIC_UNIVERSE_SECTORS[t])
    for attempt in range(retries):
        try:
            info = yf.Ticker(_normalize_ticker_for_yahoo(t)).info
            s = info.get("sector") or info.get("sectorDisp")
            if s:
                return t, canonical_sector(s)
            return t, "Unknown"
        except Exception:
            time.sleep(1.5 * (2 ** attempt))      # exponential backoff on rate limits
    return t, "Unknown"


def get_sector_map(tickers, max_workers=6, passes=2):
    """Threaded with backoff, cached per ticker, and retried in a second pass
    for anything that came back Unknown -- rate limiting is transient."""
    out = _load_sector_cache()
    out = {k: canonical_sector(v) for k, v in out.items()}
    for p in range(passes):
        todo = [t for t in tickers if out.get(t, "Unknown") == "Unknown"]
        if not todo:
            break
        log.info(f"sector pass {p + 1}: fetching {len(todo)} tickers "
                 f"({len(tickers) - len(todo)} already known)")
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futs = [ex.submit(_sector_one, t) for t in todo]
            for n, fut in enumerate(as_completed(futs), 1):
                try:
                    t, sec = fut.result()
                    if sec != "Unknown":
                        out[t] = sec
                except Exception:
                    pass
                if n % 100 == 0:
                    log.info(f"  {n}/{len(todo)}")
        _save_sector_cache(out)
        if p + 1 < passes:
            time.sleep(5)
    for t in tickers:
        out.setdefault(t, "Unknown")
    _save_sector_cache(out)
    return {t: out[t] for t in tickers}


sector_map = get_sector_map(recovered_tickers)

_known = sum(1 for v in sector_map.values() if v and v != "Unknown")
_frac = _known / max(len(sector_map), 1)
print(f"Sector coverage: {_known}/{len(sector_map)} = {_frac:.1%} known")
print(pd.Series(sector_map).value_counts().to_string())
rec("sector_coverage", _j(_frac, 3))

if _frac < CFG.min_sector_coverage and not CFG.allow_low_sector_coverage:
    raise RuntimeError(
        f"HARD STOP: sector coverage {_frac:.1%} is below "
        f"{CFG.min_sector_coverage:.0%}. Every 'Unknown' ticker collapses into one "
        "pseudo-sector, which disables sector-neutral z-scoring AND sector-neutral "
        "portfolio construction -- this is exactly what contaminated earlier testing "
        "(33.4%). Wait a few minutes for Yahoo's rate limit to clear and re-run this "
        "cell; cached tickers are kept, so it only refetches the failures. If one "
        "genuinely want to proceed without sector neutrality, set "
        "CFG.allow_low_sector_coverage = True AND CFG.sector_neutral = False.")

In [ ]:
# ============================================================
# BENCHMARK (SPY)
# ============================================================
# An equal-weight universe average is not investable at that exact
# construction. SPY total return is the standard, harder-to-beat yardstick.
def _fetch_spy():
    raw = yf.download("SPY", start=CFG.start_date, end=CFG.end_date,
                      auto_adjust=True, progress=False, timeout=30)
    px = raw["Close"]
    if isinstance(px, pd.DataFrame):
        px = px.iloc[:, 0]
    return px.dropna()


try:
    spy_prices = cached("spy", f"{CFG.start_date}|{CFG.end_date}", _fetch_spy)
    print(f"SPY: {len(spy_prices)} daily prices "
          f"({spy_prices.index.min().date()} -> {spy_prices.index.max().date()})")
except Exception as e:
    spy_prices = None
    log.warning(f"SPY fetch failed ({e}) -- cap-weighted benchmark omitted.")


# ---- PIT membership window audit ------------------------------------------
audit = membership_windows.copy()
audit["days_in_window"] = (audit["membership_end"] - audit["membership_start"]).dt.days + 1
if not (audit["membership_end"] >= audit["membership_start"]).all():
    raise RuntimeError("Invalid PIT membership window detected.")
print(f"\nMembership windows valid. Median window: {audit['days_in_window'].median():.0f} days")
display(audit.sample(min(8, len(audit)), random_state=CFG.random_state))

## 3. Feature Engineering

Features span five families: momentum, short-term reversal, volatility and
risk, liquidity, and valuation. Each is winsorised cross-sectionally and
z-scored within sector-date, so a technology stock's momentum is compared
against other technology stocks on the same day.

Two details matter more than they appear to. **Raw levels are preserved
before z-scoring**, because several downstream calculations divide by
volatility and a z-score can be negative — an error that silently inverted the
signal in earlier versions. And **features are gated on coverage**: anything
populated for less than half the panel is dropped rather than imputed, since a
mostly-missing feature filled with zeros becomes an indicator of which tickers
had data, which the model can memorise.

In [ ]:
# ============================================================
# TECHNICAL AND RISK FEATURES
# ============================================================
def compute_rsi(series: pd.Series, window: int = 14) -> pd.Series:
    """Relative Strength Index -- bounded 0-100 momentum oscillator."""
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / loss.replace(0, np.nan)
    return (100 - (100 / (1 + rs))).fillna(50)


# several new features are defined relative to the market, so build an
# equal-weight market return series first. Causal by construction -- it is a
# contemporaneous cross-sectional average, never a forward one.
_close_wide = pd.DataFrame({t: d["Close"] for t, d in price_data.items()})
market_ret = _close_wide.pct_change().mean(axis=1).rename("market_ret")
print(f"Market series: {len(market_ret)} days, "
      f"ann vol {market_ret.std() * np.sqrt(252):.1%}")


def build_technical_features(df, ticker, forward_days, market_ret=None):
    out = pd.DataFrame(index=df.index)
    close, volume = df["Close"], df["Volume"]
    ret = close.pct_change()

    out["mom_21"] = close.pct_change(21)
    out["mom_63"] = close.pct_change(63)
    out["mom_126"] = close.pct_change(126)
    out["mom_252"] = close.pct_change(252)
    # Skip-month momentum: excludes the most recent ~21 sessions, where the
    # short-term reversal effect otherwise contaminates the momentum signal.
    out["mom_63_skip1"] = close.shift(21).pct_change(63)
    out["mom_252_skip1"] = close.shift(21).pct_change(252)

    out["vol_21"] = ret.rolling(21).std() * np.sqrt(252)
    out["vol_63"] = ret.rolling(63).std() * np.sqrt(252)

    # DOWNSIDE deviation -- the volatility of losses only. Used as an
    # alternative risk-parity denominator, so position size is set by how badly
    # a name falls rather than by how much it moves in either direction.
    _neg = ret.where(ret < 0, 0.0)
    out["dvol_63"] = np.sqrt((_neg ** 2).rolling(63).sum() / 62) * np.sqrt(252) * np.sqrt(2)
    out["rsi_14"] = compute_rsi(close, 14)

    ma50, ma200 = close.rolling(50).mean(), close.rolling(200).mean()
    out["ma_ratio"] = (ma50 / ma200) - 1

    vmean, vstd = volume.rolling(21).mean(), volume.rolling(21).std()
    out["volume_zscore_21"] = (volume - vmean) / vstd.replace(0, np.nan)
    out["dollar_volume"] = close * volume

    # short-term reversal. The single most conspicuous omission from
    # an earlier implementation's feature set given a 21-day horizon. Stocks that jumped hard over
    # the last week tend to give some back -- an effect that runs OPPOSITE to
    # the momentum features above and is a live candidate explanation for an earlier implementation's
    # stable, mildly negative IC. Signed so that positive = "expect a bounce".
    out["rev_5"] = -close.pct_change(5)
    out["rev_21"] = -close.pct_change(21)

    # ---------------- an earlier implementation NEW FEATURE FAMILIES ----------------
    # The an earlier implementation feature set was ~6:2 weighted toward momentum, whose univariate IC
    # at this horizon was -0.0058 while reversal's was +0.0141. These six come
    # from genuinely different effects and none of them need fundamentals.
    if CFG.include_v14_features:
        # MAX / lottery effect (Bali, Cakici & Whitelaw): stocks with a big
        # single-day spike underperform, as investors overpay for lottery-like
        # payoffs. Negated so positive = low lottery demand = expect outperformance.
        out["max_ret_21"] = -ret.rolling(21).max()

        # 52-week-high proximity (George & Hwang): distinct from momentum --
        # it is an anchoring effect, not a trend-continuation one.
        out["prox_52w_high"] = close / close.rolling(252).max() - 1

        # Return skew: negatively-skewed stocks earn a premium for crash risk.
        out["skew_63"] = ret.rolling(63).skew()

        # Amihud illiquidity: price impact per dollar traded. Scaled to a
        # workable magnitude; the liquidity screen already removes the extreme tail.
        dv = (close * volume).replace(0, np.nan)
        out["amihud_21"] = (ret.abs() / dv).rolling(21).mean() * 1e9

        # Market-relative risk. Beta uses a trailing 63-day window; idiosyncratic
        # volatility is the residual after removing that beta's market exposure,
        # which is a different quantity from total volatility and is the version
        # the low-risk anomaly literature actually uses.
        if market_ret is not None:
            m = market_ret.reindex(df.index)
            var_m = m.rolling(63).var()
            beta = ret.rolling(63).cov(m) / var_m.replace(0, np.nan)
            out["beta_63"] = beta
            out["ivol_21"] = (ret - beta * m).rolling(21).std() * np.sqrt(252)
        else:
            out["beta_63"] = np.nan
            out["ivol_21"] = np.nan

    out["fwd_ret"] = close.pct_change(forward_days).shift(-forward_days)
    out["ticker"] = ticker
    return out


log.info("Building technical features...")
panel = pd.concat([build_technical_features(df, t, CFG.forward_return_days, market_ret)
                   for t, df in price_data.items()])
panel.index.name = "date"
panel = panel.reset_index()
panel["sector"] = panel["ticker"].map(sector_map).fillna("Unknown")

print(f"Technical panel: {panel.shape[0]:,} rows x {panel.shape[1]} cols, "
      f"{panel['ticker'].nunique()} tickers")
display(panel.head(3))

In [ ]:
# ============================================================
# POINT-IN-TIME FUNDAMENTALS MERGE
# ============================================================
# An earlier implementation ran merge_asof inside a per-ticker Python loop. merge_asof takes a `by`
# argument that does exactly this in one pass.
def merge_fundamentals_fast(panel, fundamentals, cols):
    if fundamentals is None or fundamentals.empty:
        log.info("No fundamentals available; initialising columns as NaN.")
        for c in cols:
            panel[c] = np.nan
        return panel
    f = (fundamentals.dropna(subset=["available_date"])
         .sort_values("available_date")[["ticker", "available_date"]
                                        + [c for c in cols if c in fundamentals.columns]])
    merged = pd.merge_asof(panel.sort_values("date"), f,
                           left_on="date", right_on="available_date",
                           by="ticker", direction="backward")
    return merged.sort_values(["ticker", "date"]).reset_index(drop=True)


panel = merge_fundamentals_fast(panel, fundamentals_pit, FUNDAMENTAL_FEATURE_COLS)
_cover = {c: f"{panel[c].notna().mean():.1%}" for c in FUNDAMENTAL_FEATURE_COLS}
print(f"Panel after fundamentals merge: {panel.shape}")
print(f"Fundamental coverage: {_cover}")

In [ ]:
# ============================================================
# FILTERS, WINSORISATION, SECTOR-NEUTRAL SCALING, TARGET
# ============================================================

# ---- 1. liquidity screen ---------------------------------------------------
liq_cutoff = panel.groupby("date")["dollar_volume"].transform(lambda x: x.quantile(0.05))
panel = panel[panel["dollar_volume"] >= liq_cutoff].copy()

# ---- 2. PIT membership filter (vectorised, 197x faster, same output) --
# An earlier implementation used panel.apply(..., axis=1): one Python call per row on a 1.3M-row
# frame. merge_asof maps each date to its governing snapshot in one pass.
if membership_calendar is not None:
    _snaps = membership_calendar.index.sort_values()
    _long = pd.DataFrame({
        "snap": np.repeat(_snaps, [len(membership_calendar.loc[d, "tickers"]) for d in _snaps]),
        "ticker": [t for d in _snaps for t in sorted(membership_calendar.loc[d, "tickers"])],
    }).assign(_member=True)
    _snap_of = pd.merge_asof(panel[["date"]].drop_duplicates().sort_values("date"),
                             pd.DataFrame({"date": _snaps, "snap": _snaps}),
                             on="date", direction="backward")
    _n0 = len(panel)
    panel = (panel.merge(_snap_of, on="date", how="left")
                  .merge(_long, on=["snap", "ticker"], how="left"))
    panel = (panel[panel["_member"].fillna(False)]
             .drop(columns=["snap", "_member"]).reset_index(drop=True))
    log.info(f"PIT filter kept {len(panel):,} / {_n0:,} rows")

# ---- 3. feature column groups ---------------------------------------------
TECHNICAL_FEATURE_COLS = ["mom_21", "mom_63", "mom_126", "mom_252",
                          "mom_63_skip1", "mom_252_skip1",
                          "vol_21", "vol_63", "rsi_14", "ma_ratio", "volume_zscore_21"]
REVERSAL_FEATURE_COLS = ["rev_5", "rev_21"] if CFG.include_reversal_features else []
V14_FEATURE_COLS = (["max_ret_21", "prox_52w_high", "skew_63", "amihud_21",
                     "beta_63", "ivol_21"] if CFG.include_v14_features else [])
INTERACTION_FEATURE_COLS = ["mom_vol_interact"]

panel["mom_vol_interact"] = panel["mom_252"] * panel["vol_21"]
# roe_pe_interact is gone. pe_ratio had 0.0% coverage in earlier testing, so
# the product was NaN on every row -- a feature slot occupied by nothing.
if CFG.include_fundamentals_in_model:
    panel["roe_pe_interact"] = panel["roe"] * panel["pe_ratio"]
    INTERACTION_FEATURE_COLS.append("roe_pe_interact")

# ---- 4. PRESERVE RAW LEVELS BEFORE Z-SCORING  (an earlier implementation critical fix) ----------
# An earlier implementation overwrote mom_21 / vol_21 in place with z-scores, then the rule-based sections
# divided by them as if they were still raw. A z-score is negative about half
# the time, so `mom_21 / vol_21` flipped sign on ~50% of rows -- verified: the
# intended and actual the momentum section signals agreed on sign 50.0% of the time,
# rank correlation -0.087. the regime section was worse: the mean of a cross-sectional
# z-score is 0 by construction, so the HMM regime input had standard deviation
# 0.000000 and the regime gate was fitting floating-point residue.
RAW_KEEP = ["mom_21", "mom_63", "mom_252", "vol_21", "vol_63", "dvol_63", "rsi_14",
            "dollar_volume", "rev_5", "rev_21"] + V14_FEATURE_COLS
for _c in RAW_KEEP:
    if _c in panel.columns:
        panel[f"{_c}_raw"] = panel[_c].copy()

# True risk-adjusted momentum, from raw levels. vol_*_raw is annualised, so
# this is a unitless Sharpe-like ratio, always sign-aligned with its numerator.
panel["risk_adj_mom_21"] = panel["mom_21_raw"] / panel["vol_21_raw"].replace(0, np.nan)
panel["risk_adj_mom_63"] = panel["mom_63_raw"] / panel["vol_63_raw"].replace(0, np.nan)

# ---- 5. cross-sectional winsorization -------------------------------------
_ZCOLS = (TECHNICAL_FEATURE_COLS + REVERSAL_FEATURE_COLS
          + V14_FEATURE_COLS + INTERACTION_FEATURE_COLS)

# An earlier implementation ran a Python lambda per (date, column) group -- roughly 3,200
# dates x 15 columns of interpreted callbacks. Computing all per-date quantiles
# in one pass and clipping with numpy does the same thing ~20x faster.
log.info("Winsorising cross-sectionally...")
_q = panel.groupby("date")[_ZCOLS].quantile([0.01, 0.99]).unstack()
_lo = _q.xs(0.01, level=1, axis=1).reindex(columns=_ZCOLS)
_hi = _q.xs(0.99, level=1, axis=1).reindex(columns=_ZCOLS)
_dates = panel["date"].to_numpy()
panel[_ZCOLS] = np.clip(panel[_ZCOLS].to_numpy(dtype=float),
                        _lo.reindex(_dates).to_numpy(dtype=float),
                        _hi.reindex(_dates).to_numpy(dtype=float))

# ---- 6. sector-neutral z-scoring ------------------------------------------
# Standardise within each (date, sector) group so a tech stock's momentum is
# compared to other tech stocks on the same day, not to a utility six months
# earlier. Uses transform("mean")/transform("std"), which run in C, rather than
# a Python lambda per group.
def zscore_within(df, cols, keys):
    g = df.groupby(keys)[cols]
    mu, sd = g.transform("mean"), g.transform("std")
    return (df[cols] - mu) / sd.where(sd > 0, 1.0)


log.info("Applying sector-neutral z-scoring...")
panel[_ZCOLS] = zscore_within(panel, _ZCOLS, ["date", "sector"])
if CFG.include_fundamentals_in_model:
    # NO fillna(0.0) here. An earlier implementation filled ~92% of these with zero, which turns
    # "missing" into a sparse indicator of which tickers yfinance happened to
    # have statements for -- effectively a ticker-identity feature the model can
    # memorise. Leaving them NaN lets XGBoost route them with a learned default
    # direction instead of inventing a value.
    panel[FUNDAMENTAL_FEATURE_COLS] = zscore_within(
        panel, FUNDAMENTAL_FEATURE_COLS, ["date", "sector"])

# ---- 7. target + breadth gate ---------------------------------------------
panel = panel.dropna(subset=["fwd_ret"]).reset_index(drop=True)
panel["month"] = panel["date"].values.astype("datetime64[M]")

_counts = panel.groupby("month")["ticker"].nunique()
_thin = _counts[_counts < CFG.min_universe_breadth].index
panel = panel[~panel["month"].isin(_thin)].reset_index(drop=True)
log.info(f"Dropped {len(_thin)} thin months below {CFG.min_universe_breadth} names")

panel["fwd_ret_xs"] = panel.groupby("date")["fwd_ret"].transform(lambda x: x - x.median())
panel["fwd_ret_sector_xs"] = panel.groupby(["date", "sector"])["fwd_ret"].transform(lambda x: x - x.median())
TARGET_COL = "fwd_ret_sector_xs" if CFG.sector_neutral else "fwd_ret_xs"

FEATURE_COLS = (TECHNICAL_FEATURE_COLS + REVERSAL_FEATURE_COLS + V14_FEATURE_COLS
                + INTERACTION_FEATURE_COLS
                + (FUNDAMENTAL_FEATURE_COLS if CFG.include_fundamentals_in_model else []))

# ---- 8. an earlier implementation DEAD-FEATURE GATE ---------------------------------------------
# In earlier testing six of twenty feature slots carried no usable information:
# pe_ratio and pb_ratio at 0.0% coverage, roe / profit_margin / debt_to_equity
# at 8.3%, and roe_pe_interact NaN everywhere. With colsample_bytree=0.5 that
# meant roughly three of the ten columns each tree sampled were dead. A feature
# the model cannot learn from is not free -- it displaces one that might work.
_col_cov = panel[FEATURE_COLS].notna().mean().sort_values()
_dead = _col_cov[_col_cov < CFG.min_feature_column_coverage].index.tolist()

print("\n=== FEATURE COVERAGE GATE ===")
for c, v in _col_cov.items():
    mark = "DROPPED" if c in _dead else "keep"
    print(f"  {c:20s} {v:6.1%}  {mark}")
if _dead:
    print(f"\n  Dropping {len(_dead)} feature(s) below "
          f"{CFG.min_feature_column_coverage:.0%} coverage: {_dead}")
    FEATURE_COLS = [c for c in FEATURE_COLS if c not in _dead]
    TECHNICAL_FEATURE_COLS = [c for c in TECHNICAL_FEATURE_COLS if c not in _dead]
    REVERSAL_FEATURE_COLS = [c for c in REVERSAL_FEATURE_COLS if c not in _dead]
    V14_FEATURE_COLS = [c for c in V14_FEATURE_COLS if c not in _dead]
    INTERACTION_FEATURE_COLS = [c for c in INTERACTION_FEATURE_COLS if c not in _dead]
    FUNDAMENTAL_FEATURE_COLS = [c for c in FUNDAMENTAL_FEATURE_COLS if c not in _dead]
else:
    print("  all features clear the coverage bar")

print("\n" + "=" * 72)
print("PANEL DIAGNOSTIC")
print("=" * 72)
print(f"Usable modeling rows : {len(panel):,}")
print(f"Usable tickers       : {panel['ticker'].nunique():,}")
print(f"Date range           : {panel['date'].min().date()} -> {panel['date'].max().date()}")
print(f"Target column        : {TARGET_COL}")
print(f"Features ({len(FEATURE_COLS)})        : {FEATURE_COLS}")
print(f"  momentum family      : {[c for c in FEATURE_COLS if c.startswith(('mom_','ma_','rsi'))]}")
print(f"  reversal family      : {[c for c in FEATURE_COLS if c.startswith(('rev_','max_ret'))]}")
print(f"  risk / liquidity     : {[c for c in FEATURE_COLS if c.startswith(('vol_','ivol','beta','amihud','skew','volume'))]}")
print(f"Target kurtosis      : {panel[TARGET_COL].kurt():.1f}  (fat tails -- see the corresponding section rank target)")

In [ ]:
# ============================================================
# MONTH-END PANEL AND LOCKED HOLDOUT
# ============================================================
# The target is a 21-TRADING-DAY forward return, so consecutive daily rows
# share 20 of their 21 days. Three consequences of modelling on daily rows:
# 1. The model thinks it has 1.3M independent observations. It has ~1/21 of
#      that, so every regularisation choice is calibrated to a sample size that
#      does not exist.
#   2. Runtime: 104 folds x up to 1.3M rows was the largest cost in an earlier implementation.
#   3. the rule-based sections built portfolios from daily rows and then in earlier work
#      grouped by month, so a "monthly return" was the average of 21
#      overlapping forward returns and a name's weight was proportional to how
#      many days it happened to be flagged. Verified on an identical book:
#      0.55x the volatility, 1.8x the Sharpe, turnover understated ~1.5x.
#      That is why the momentum section of an earlier implementation "beat the ML baseline on every metric".
# Everything downstream now uses panel_me.

_month_end = panel.groupby("month")["date"].max()
panel_me = panel[panel["date"].isin(_month_end.values)].copy().reset_index(drop=True)
assert panel_me.groupby(["ticker", "month"]).size().max() == 1, "duplicate ticker-month"

# ---- rank target (an earlier implementation): bound the fat-tailed target -----------------------
# Squared-error boosting on a raw excess return spends its capacity on the
# +/-50% tails, which are close to pure noise. The cross-sectional percentile
# rank is MONOTONE in the raw target -- so IC is unchanged -- but bounded.
_grp = ["date", "sector"] if CFG.sector_neutral else ["date"]
panel_me["target_rank"] = panel_me.groupby(_grp)["fwd_ret"].transform(lambda s: s.rank(pct=True) - 0.5)
TRAIN_TARGET = "target_rank" if CFG.train_on_rank_target else TARGET_COL

# ---- drop warm-up rows that are mostly NaN --------------------------------
_cov = panel_me[FEATURE_COLS].notna().mean(axis=1)
_n0 = len(panel_me)
panel_me = panel_me[_cov >= CFG.min_feature_coverage].reset_index(drop=True)

# ---- LOCKED HOLDOUT -------------------------------------------------------
# The strongest protection available and it costs nothing. Every table below
# reports the development period only. the holdout section evaluates the holdout once,
# and only if deliberately set CFG.reveal_holdout. Moving this date after
# seeing a bad result destroys the entire point.
panel_dev = panel[panel["date"] < CFG.holdout_start].copy()
panel_me_dev = panel_me[panel_me["date"] < CFG.holdout_start].copy()

print(f"Month-end panel   : {len(panel_me):,} rows "
      f"({len(panel_me) / max(len(panel), 1):.1%} of daily) | "
      f"{panel_me['month'].nunique()} months | {panel_me['ticker'].nunique()} tickers")
print(f"Dropped warm-up   : {_n0 - len(panel_me):,} rows below "
      f"{CFG.min_feature_coverage:.0%} feature coverage")
print(f"Development period: {panel_me_dev['date'].min().date()} -> "
      f"{panel_me_dev['date'].max().date()}  ({panel_me_dev['month'].nunique()} months)")
print(f"Holdout (sealed)  : {CFG.holdout_start.date()} onward, "
      f"{panel_me[panel_me['date'] >= CFG.holdout_start]['month'].nunique()} months")
print(f"Train target      : {TRAIN_TARGET}   (rank kurtosis "
      f"{panel_me['target_rank'].kurt():.2f} vs raw {panel_me[TARGET_COL].kurt():.1f})")

rec("features", {"n": len(FEATURE_COLS), "cols": FEATURE_COLS,
                 "dropped_for_coverage": _dead,
                 "coverage": {c: _j(v, 3) for c, v in _col_cov.items()}})
rec("panel", {"daily_rows": int(len(panel)), "month_end_rows": int(len(panel_me)),
              "tickers": int(panel_me["ticker"].nunique()),
              "dev_months": int(panel_me_dev["month"].nunique()),
              "holdout_months": int(panel_me[panel_me["date"] >= CFG.holdout_start]["month"].nunique()),
              "target_kurtosis_raw": _j(panel_me[TARGET_COL].kurt(), 1),
              "target_kurtosis_rank": _j(panel_me["target_rank"].kurt(), 2)})

## 4. Walk-Forward Model Training

The model is retrained on a rolling schedule and used only on data it has
never seen. Two safeguards make this a genuine out-of-sample exercise rather
than a dressed-up in-sample fit:

**The embargo.** The target is a 21-day *forward* return, so a training row
dated the day before a test date already contains information about the
following three weeks. Training data is cut off 21 sessions before each test
date.

**The rank target.** Raw excess returns are heavy-tailed, and squared-error
boosting spends most of its capacity fitting the ±50% tails, which are close
to pure noise. Training on the cross-sectional percentile rank preserves the
ordering the strategy actually trades on while bounding the loss.

In [ ]:
# ============================================================
# MODEL DEFINITION AND WALK-FORWARD ENGINE
# ============================================================
# Design choices:
#   (a) trains on the RANK target -- bounds the fat-tailed objective
#   (b) safe_ic drops NaN pairs instead of reporting 0.0
#   (c) rolling vs expanding training window is configurable
#   (d) predictions are rank-averaged over several seeds
#   (e) the loop is a FUNCTION, so configurations can be A/B tested (the walk-forward grid) and
#       the holdout can be scored later without duplicating code

def safe_ic(a, b) -> float:
    """Spearman rank IC that DROPS NaN pairs rather than reporting 0.0.

    scipy's spearmanr returns nan if EITHER input contains a single NaN. An
    earlier version mapped that nan to 0.0, which silently converted
    unmeasurable months into measurements of zero and corrupted five of nine
    years of a baseline comparison."""
    a = pd.Series(a).reset_index(drop=True)
    b = pd.Series(b).reset_index(drop=True)
    ok = a.notna() & b.notna()
    if ok.sum() < 10 or a[ok].nunique() < 2 or b[ok].nunique() < 2:
        return np.nan
    return float(stats.spearmanr(a[ok], b[ok]).correlation)


def get_model(seed=None, feature_cols=None, max_depth=None, n_estimators=None):
    """Interaction constraints force the model to combine technical features
    only with other technical features (and fundamentals only with
    fundamentals) within any single tree, preventing noisy fundamentals from
    dominating early branch logic. Carried over from an earlier implementation -- it was a sound
    idea, just never isolated as one.

    `feature_cols` must match the matrix the model will actually be fit on.
    Building the constraints from a global list instead silently breaks any
    run that uses a feature subset (e.g. the walk-forward grid ablation config).

    capacity comes from CFG and is overridable per call, because the in-sample/out-of-sample check found
    an in-sample IC of +0.139 against an out-of-sample -0.019. The model was
    memorising, so depth drops from 3 to 2, trees from 150 to 80,
    min_child_weight rises to 50 and column sampling tightens. the capacity sweep sweeps this
    properly rather than trusting the guess."""
    seed = CFG.random_state if seed is None else seed
    feature_cols = FEATURE_COLS if feature_cols is None else feature_cols
    if HAS_XGB:
        tech = [c for c in TECHNICAL_FEATURE_COLS + REVERSAL_FEATURE_COLS + ["mom_vol_interact"]
                if c in feature_cols]
        fund = [c for c in FUNDAMENTAL_FEATURE_COLS + ["roe_pe_interact"] if c in feature_cols]
        groups = [g for g in [tech, fund] if g]
        return xgb.XGBRegressor(
            n_estimators=n_estimators or CFG.model_n_estimators,
            max_depth=max_depth or CFG.model_max_depth,
            learning_rate=CFG.model_learning_rate,
            subsample=0.7, colsample_bytree=CFG.model_colsample, colsample_bylevel=0.8,
            min_child_weight=CFG.model_min_child_weight,
            reg_alpha=0.5, reg_lambda=5.0,
            interaction_constraints=groups if groups else None,
            random_state=seed, n_jobs=-1, verbosity=0)
    # HistGradientBoosting is the right fallback: unlike GradientBoostingRegressor
    # it handles NaN natively (the feature matrix has warm-up NaNs by design) and
    # it is an order of magnitude faster.
    return HistGradientBoostingRegressor(
        max_iter=n_estimators or CFG.model_n_estimators,
        max_depth=max_depth or CFG.model_max_depth,
        learning_rate=CFG.model_learning_rate,
        min_samples_leaf=int(CFG.model_min_child_weight),
        l2_regularization=5.0, random_state=seed)


def run_walk_forward(pnl, feature_cols, target_col, train_target,
                     rolling_months=None, n_seeds=None, min_train_periods=None,
                     embargo_days=None, with_baselines=False, label="",
                     all_dates_source=None, verbose=False, track_insample=False,
                     max_depth=None, n_estimators=None, target_override=None,
                     refresh_months=None):
    """Expanding/rolling walk-forward with an embargo. Returns (summary, folds).

    The EMBARGO matters: the target is a 21-day FORWARD return, so a training
    row dated the day before the test date already contains information about
    the following three weeks. Training data is cut off 21 trading days before
    each test date so no training label overlaps the test period."""
    n_seeds = CFG.n_seeds if n_seeds is None else n_seeds
    refresh_months = CFG.refresh_months if refresh_months is None else refresh_months
    min_train_periods = CFG.min_train_periods if min_train_periods is None else min_train_periods
    embargo_days = CFG.forward_return_days if embargo_days is None else embargo_days

    me = pnl.groupby("month")["date"].max().sort_index()
    months = list(me.index)
    src = all_dates_source if all_dates_source is not None else pnl
    all_dates = pd.Series(pd.to_datetime(sorted(src["date"].unique())))

    def cutoff(test_date):
        prior = all_dates[all_dates < test_date]
        if len(prior) <= embargo_days:
            return prior.iloc[0] if len(prior) else test_date
        return prior.iloc[-embargo_days]

    folds, ic_records = [], []
    imp_accum = np.zeros(len(feature_cols))
    n_folds = 0
    fitted_models, months_since_fit = None, 0
    fitted_baselines = {}          # kept so held folds can still score them
    last = {"model": None, "X_test": None, "elastic": None}

    for i, m in enumerate(months):
        if i < min_train_periods:
            continue
        reb_date = me.loc[m]
        test_df = pnl.loc[pnl["date"] == reb_date]
        if test_df.empty:
            continue
        train_df = pnl.loc[pnl["date"] < cutoff(reb_date)]
        if rolling_months:
            train_df = train_df.loc[train_df["month"] >= m - pd.DateOffset(months=rolling_months)]
        if len(train_df) < 200:
            continue

        # refit only every `refresh_months`. Between refits the existing
        # model is reused, which is what "train once a year and trade it"
        # actually means operationally. The controlled test showed holding a
        # model 12 months gives up ~75% of the benefit of a short lookback,
        # because by month 11 the book is trading an 11-month-old relationship --
        # so this is exposed as a knob and swept in the lookback sweep rather than assumed.
        need_refit = (fitted_models is None) or (months_since_fit >= refresh_months)
        if not need_refit:
            # NOTE: an earlier version of this block referenced `te` instead of `test_df` and
            # was wrapped in a bare `except Exception: pass`. The NameError was
            # swallowed on every fold, so the model was silently refitted every
            # period and the entire refresh-cadence dimension of the lookback sweep produced
            # identical results for monthly, quarterly and annual. The variable
            # is fixed, and failures are now LOGGED rather than hidden -- a
            # silent fallback that changes what an experiment measures is worse
            # than a crash.
            try:
                preds_h = [stats.rankdata(mm.predict(test_df[feature_cols])) / len(test_df)
                           for mm in fitted_models]
                fold = test_df.copy()
                fold["pred"] = np.mean(preds_h, axis=0)
                months_since_fit += 1
                n_folds += 1

                # score the baselines on held folds too. Previously the
                # baselines were only evaluated on REFIT folds, so with
                # refresh_months=3 the primary model had 180 months of IC while
                # Ridge, ElasticNet and the naive feature had only 60 -- every
                # model comparison in an earlier implementation was against a different sample.
                rowic_h = {"month": m, "rebalance_date": reb_date, "refit": False,
                           "ic": safe_ic(fold["pred"], fold[target_col])}
                if with_baselines:
                    Xh = test_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
                    for key, col in [("ridge", "pred_linear"), ("elastic", "pred_elastic")]:
                        mb = fitted_baselines.get(key)
                        try:
                            fold[col] = mb.predict(Xh) if mb is not None else np.nan
                        except Exception:
                            fold[col] = np.nan
                    fold["pred_naive"] = test_df[BASELINE_FEATURE].values
                    rowic_h["ic_linear"] = safe_ic(fold["pred_linear"], fold[target_col])
                    rowic_h["ic_elastic"] = safe_ic(fold["pred_elastic"], fold[target_col])
                    rowic_h["ic_naive"] = safe_ic(fold["pred_naive"], fold[target_col])

                folds.append(fold)
                ic_records.append(rowic_h)
                continue
            except Exception as e:
                log.warning(f"[{label}] held-model reuse failed at {m}: "
                            f"{type(e).__name__}: {e} -- refitting instead.")

        X_tr, y_tr = train_df[feature_cols], train_df[train_target]
        X_te = test_df[feature_cols]

        try:
            preds, imps, fitted_models = [], [], []
            for s in range(n_seeds):
                mdl = get_model(seed=CFG.random_state + s, feature_cols=feature_cols,
                                max_depth=max_depth, n_estimators=n_estimators)
                mdl.fit(X_tr, y_tr)
                fitted_models.append(mdl)
                preds.append(stats.rankdata(mdl.predict(X_te)) / len(X_te))
                if hasattr(mdl, "feature_importances_"):
                    imps.append(mdl.feature_importances_)
            pred = np.mean(preds, axis=0)
            if imps:
                imp_accum += np.mean(imps, axis=0)
            last["model"] = mdl
        except Exception as e:
            log.warning(f"model failed at {m}: {e} -- skipping fold.")
            continue

        fold = test_df.copy()
        fold["pred"] = pred
        last["X_test"] = X_te.copy()
        n_folds += 1

        months_since_fit = 1
        rowic = {"month": m, "rebalance_date": reb_date, "refit": True,
                 "ic": safe_ic(fold["pred"], fold[target_col])}

        # in-sample IC on a training subsample. This is the diagnostic that
        # separates "the model overfits" from "the model found nothing". A large
        # positive in-sample IC alongside a negative out-of-sample one means
        # memorisation; near-zero on both means the features are empty and the
        # negative OOS sign is coming from the feature mix, not from the fit.
        if track_insample:
            k = min(4000, len(train_df))
            samp = train_df.sample(k, random_state=CFG.random_state)
            try:
                ins = stats.rankdata(mdl.predict(samp[feature_cols])) / k
                rowic["ic_insample"] = safe_ic(ins, samp[target_col])
            except Exception:
                rowic["ic_insample"] = np.nan

        if with_baselines:
            Xtrf = X_tr.replace([np.inf, -np.inf], np.nan).fillna(0)
            Xtef = X_te.replace([np.inf, -np.inf], np.nan).fillna(0)
            try:
                _rid = Ridge(alpha=5.0).fit(Xtrf, y_tr)
                fitted_baselines["ridge"] = _rid
                fold["pred_linear"] = _rid.predict(Xtef)
            except Exception:
                fold["pred_linear"] = np.nan
            try:
                el = ElasticNet(alpha=1e-3, l1_ratio=0.5, random_state=CFG.random_state,
                                max_iter=5000).fit(Xtrf, y_tr)
                fitted_baselines["elastic"] = el
                fold["pred_elastic"] = el.predict(Xtef)
                last["elastic"] = el
            except Exception:
                fold["pred_elastic"] = np.nan
            fold["pred_naive"] = X_te[BASELINE_FEATURE].values
            rowic["ic_linear"] = safe_ic(fold["pred_linear"], fold[target_col])
            rowic["ic_elastic"] = safe_ic(fold["pred_elastic"], fold[target_col])
            rowic["ic_naive"] = safe_ic(fold["pred_naive"], fold[target_col])

        folds.append(fold)
        ic_records.append(rowic)
        if verbose and n_folds % 24 == 0:
            log.info(f"  {label} fold {n_folds} @ {m.date()}")

    if n_folds == 0:
        raise RuntimeError(f"No walk-forward folds completed for '{label}'.")

    res = pd.concat(folds).reset_index(drop=True)
    icdf = pd.DataFrame(ic_records).set_index("month").sort_index()
    ic = icdf["ic"].dropna()
    t = ic.mean() / (ic.std() / np.sqrt(len(ic))) if (len(ic) > 2 and ic.std() > 0) else np.nan
    if "refit" in icdf.columns:
        summary_refits = int(icdf["refit"].sum())
    else:
        summary_refits = len(icdf)
    summary = {"config": label, "mean_ic": ic.mean(), "ic_t": t, "n_refits": summary_refits,
               "ic_std": ic.std(), "pct_positive": (ic > 0).mean(), "n_months": len(ic)}
    if "ic_insample" in icdf.columns:
        summary["mean_ic_insample"] = icdf["ic_insample"].mean()
    return summary, {"results": res, "ic_df": icdf, "n_folds": n_folds,
                     "importance": imp_accum / max(n_folds, 1), "last": last}


BASELINE_FEATURE = "mom_252"
print("walk-forward engine ready")

In [ ]:
# ============================================================
# PRIMARY WALK-FORWARD RUN
# ============================================================
# Run over the FULL panel so the holdout predictions exist for the holdout section, but
# immediately split them off. Nothing below Section 3 sees holdout data.
# The walk-forward itself never looks ahead -- the holdout discipline is about
# what the HUMAN gets to look at while tuning.

_t0 = time.time()
primary_summary, primary = run_walk_forward(
    panel_me, FEATURE_COLS, TARGET_COL, TRAIN_TARGET,
    rolling_months=CFG.rolling_train_months, with_baselines=True,
    label="primary", all_dates_source=panel, verbose=True, track_insample=True)
log.info(f"walk-forward complete in {time.time() - _t0:.0f}s, {primary['n_folds']} folds")

results_full = primary["results"]
ic_df_full = primary["ic_df"]
avg_feature_importance = primary["importance"]
last_model = primary["last"]["model"]
last_X_test = primary["last"]["X_test"]
last_elastic_model = primary["last"]["elastic"]

# ---- split off the holdout -------------------------------------------------
results_df = results_full[results_full["date"] < CFG.holdout_start].copy().reset_index(drop=True)
ic_df = ic_df_full[ic_df_full.index < CFG.holdout_start].copy()
n_folds = int(results_df["month"].nunique())

print(f"\nFolds total: {primary['n_folds']}  |  development: {n_folds}  |  "
      f"sealed in holdout: {primary['n_folds'] - n_folds}")
print(f"Development period: {results_df['date'].min().date()} -> {results_df['date'].max().date()}")

# ---- an earlier implementation D1: overfitting vs. empty features -------------------------------
if "ic_insample" in ic_df.columns:
    _ins, _oos = ic_df["ic_insample"].mean(), ic_df["ic"].mean()
    print(f"\n=== IN-SAMPLE VS OUT-OF-SAMPLE IC ===")
    print(f"  in-sample  IC : {_ins:+.4f}")
    print(f"  out-of-sample : {_oos:+.4f}")
    print(f"  gap           : {_ins - _oos:+.4f}")
    # an earlier implementation's verdict logic called this "OVERFITTING" on the sign of the OOS
    # IC alone. But the capacity sweep then showed OOS IC was between -0.018 and +0.008 across
    # every capacity from stumps to depth 5, with |t| never above 1.3 -- i.e.
    # indistinguishable from zero. A negative point estimate that is not
    # significant is not evidence of inversion, so significance now enters the
    # verdict.
    _oos_t = ic_df["ic"].mean() / (ic_df["ic"].std() / np.sqrt(ic_df["ic"].notna().sum()))
    if abs(_oos_t) < 2.0 and _ins > 0.05:
        verdict = ("MEMORISES IN-SAMPLE, NOISE OUT-OF-SAMPLE -- the fit is real but the "
                   f"out-of-sample IC (t={_oos_t:+.2f}) is not distinguishable from zero. "
                   "Cutting capacity will not help; see the capacity sweep.")
    elif _ins > 0.05 and _oos < 0 and _oos_t <= -2.0:
        verdict = "OVERFITTING WITH RELIABLE INVERSION -- the ranking is significantly backwards"
    elif abs(_ins) < 0.03:
        verdict = ("NO FIT AT ALL -- the model cannot fit even its training data, so the "
                   "feature set, not capacity, is the constraint")
    else:
        verdict = "partial fit with out-of-sample decay -- the usual pattern"
    print(f"  OOS IC t-stat : {_oos_t:+.2f}")
    print(f"  verdict       : {verdict}")
    rec("D1_insample_vs_oos", {"insample": _j(_ins), "oos": _j(_oos),
                               "oos_t": _j(_oos_t, 2),
                               "gap": _j(_ins - _oos), "verdict": verdict})

In [ ]:
# ============================================================
# FORWARD-TARGET INTEGRITY AUDIT
# ============================================================
# An earlier implementation used .iterrows() then .apply() over 46k rows. searchsorted does the same
# check in one vectorised pass. This independently recomputes every stored
# forward return from the raw price series -- if the stored target does not
# match, everything downstream is meaningless.
def audit_forward_target(res, price_data, h):
    diffs, n = [], 0
    for t, g in res.groupby("ticker"):
        if t not in price_data:
            continue
        px = price_data[t]["Close"]
        pos = px.index.searchsorted(g["date"].values, side="right") - 1
        exit_pos = pos + h
        ok = (pos >= 0) & (exit_pos < len(px))
        if not ok.any():
            continue
        indep = px.values[exit_pos[ok]] / px.values[pos[ok]] - 1
        diffs.append(np.abs(g["fwd_ret"].values[ok] - indep))
        n += int(ok.sum())
    d = np.concatenate(diffs) if diffs else np.array([0.0])
    return float(np.nanmax(d)), n


max_diff, n_audited = audit_forward_target(results_df, price_data, CFG.forward_return_days)
print(f"Forward-target audit: {n_audited:,} observations, max abs difference {max_diff:.3e}")
if max_diff > 1e-8:
    raise RuntimeError("Stored forward return does not match an independent recomputation.")
print("PASS -- the stored 21-session target matches the raw price series.")
rec("target_audit", {"n": n_audited, "max_diff": float(f"{max_diff:.2e}")})

## 5. Signal Quality

The **Information Coefficient** is the monthly Spearman rank correlation
between predicted and realised returns: did the model rank the winners above
the losers? In cross-sectional equities an IC of 0.02–0.05 is a genuinely
tradeable signal.

The model is compared against three baselines over identical months — a ridge
regression, an elastic net, and a single naive momentum feature. Comparing
against baselines measured on a different sample is a mistake this project
made and corrected.

In [ ]:
# ============================================================
# INFORMATION COEFFICIENT AND BASELINE COMPARISON
# ============================================================
# The IC is the monthly Spearman rank correlation between predictions and what
# actually happened: did the model rank the winners above the losers? Only the
# ORDER matters, not the magnitudes. Rule of thumb for cross-sectional
# equities: |IC| of 0.02-0.05 is a meaningful, tradeable signal.

mean_ic = ic_df["ic"].mean()
std_ic = ic_df["ic"].std()
ic_ir = (mean_ic / std_ic * np.sqrt(12)) if std_ic > 0 else np.nan
pct_pos = (ic_df["ic"] > 0).mean()
ic_t = mean_ic / (std_ic / np.sqrt(ic_df["ic"].notna().sum())) if std_ic > 0 else np.nan

print("=" * 60)
print("SIGNAL QUALITY -- INFORMATION COEFFICIENT (development period)")
print("=" * 60)
print(f"Mean monthly IC (Spearman)      : {mean_ic:+.4f}")
print(f"IC standard deviation           : {std_ic:.4f}")
print(f"IC t-statistic                  : {ic_t:+.2f}")
print(f"Annualised IC Information Ratio : {ic_ir:+.2f}")
print(f"Months with positive IC         : {pct_pos:.1%}")
print(f"Months measured                 : {int(ic_df['ic'].notna().sum())} / {len(ic_df)}")

# ---- baseline comparison, now honestly measured ---------------------------
comp = pd.DataFrame({
    "model": ["XGBoost (primary)", "Ridge", "ElasticNet", f"Naive ({BASELINE_FEATURE})"],
    "mean_ic": [mean_ic, ic_df["ic_linear"].mean(), ic_df["ic_elastic"].mean(),
                ic_df["ic_naive"].mean()],
    "months_measured": [int(ic_df[c].notna().sum())
                        for c in ["ic", "ic_linear", "ic_elastic", "ic_naive"]],
    "pct_positive": [(ic_df[c] > 0).mean() for c in ["ic", "ic_linear", "ic_elastic", "ic_naive"]],
}).set_index("model")

print("\n=== MODEL COMPARISON ===")
display(comp.style.format({"mean_ic": "{:+.4f}", "pct_positive": "{:.1%}"}))
print(f"months_measured out of {len(ic_df)}. In an earlier implementation any unmeasurable month was")
print("silently recorded as IC = 0.0, which is why ic_naive read exactly")
print("0.000000 in five of nine calendar years.")

rec("S4_ic", {"mean_ic": _j(mean_ic), "ic_t": _j(ic_t, 2), "ic_ir": _j(ic_ir, 2),
              "pct_positive": _j(pct_pos, 3),
              "baselines": {i: {"ic": _j(r["mean_ic"]), "months": int(r["months_measured"])}
                            for i, r in comp.iterrows()}})

In [ ]:
# ============================================================
# SIGNAL VALIDATION DIAGNOSTICS
# ============================================================
# Model-free checks: does the pooled correlation agree with the per-month mean,
# what does the target distribution look like, which raw features carry signal
# on their own, and is the IC stable across calendar years.

pooled = stats.spearmanr(results_df["pred"], results_df[TARGET_COL]).correlation
print(f"Pooled Spearman (all folds)     : {pooled:+.4f}")
print(f"Mean of monthly ICs             : {mean_ic:+.4f}")
print("These should share a sign and rough magnitude; a big mismatch means a")
print("few extreme months or names dominate one of the two measures.\n")

print(f"=== TARGET DISTRIBUTION ({TARGET_COL}) ===")
print(f"  skew {results_df[TARGET_COL].skew():+.3f} | "
      f"kurtosis {results_df[TARGET_COL].kurt():.1f} | "
      f"positive {(results_df[TARGET_COL] > 0).mean():.1%}")
print("  Right-skewed: the cross-sectional MEAN exceeds the MEDIAN. Since the")
print("  target is demeaned by the median, both tails carry a positive average,")
print("  which the short leg pays for every month (see the monotonicity check).")

# ---- univariate feature IC -------------------------------------------------
uni = []
for f in FEATURE_COLS:
    sub = results_df.dropna(subset=[f, TARGET_COL])
    ics = sub.groupby("month")[[f, TARGET_COL]].apply(
        lambda g: safe_ic(g[f], g[TARGET_COL])).dropna()
    if len(ics) < 12:
        continue
    uni.append({"feature": f, "mean_ic": ics.mean(),
                "t_stat": ics.mean() / (ics.std() / np.sqrt(len(ics))) if ics.std() > 0 else np.nan,
                "pct_positive": (ics > 0).mean(), "n_months": len(ics)})
univariate_ic_df = pd.DataFrame(uni).sort_values("mean_ic", ascending=False).reset_index(drop=True)
print("\n=== UNIVARIATE FEATURE IC (model-free) ===")
display(univariate_ic_df.style.format({"mean_ic": "{:+.4f}", "t_stat": "{:+.2f}",
                                       "pct_positive": "{:.1%}"}))

# ---- IC by calendar year ---------------------------------------------------
icy = ic_df.copy()
icy["year"] = pd.to_datetime(icy.index).year
yearly_ic = icy.groupby("year")[["ic", "ic_linear", "ic_naive"]].mean()
print("\n=== MEAN IC BY CALENDAR YEAR ===")
display(yearly_ic.style.format("{:+.4f}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(yearly_ic.index.astype(str), yearly_ic["ic"],
            color=np.where(yearly_ic["ic"] >= 0, "#2e8b57", "#c0392b"))
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Model mean IC by calendar year"); axes[0].set_ylabel("Mean IC")
axes[0].tick_params(axis="x", rotation=45)

pl = univariate_ic_df.sort_values("mean_ic")
axes[1].barh(pl["feature"], pl["mean_ic"],
             color=["#c0392b" if v < 0 else "#2e8b57" for v in pl["mean_ic"]])
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Univariate feature IC"); axes[1].set_xlabel("Mean IC")
plt.tight_layout(); plt.show()

rec("S4b", {"pooled_spearman": _j(pooled),
            "target_skew": _j(results_df[TARGET_COL].skew(), 3),
            "target_kurtosis": _j(results_df[TARGET_COL].kurt(), 1),
            "ic_by_year": {int(y): _j(v) for y, v in yearly_ic["ic"].items()},
            "top_univariate": [{"f": r["feature"], "ic": _j(r["mean_ic"]), "t": _j(r["t_stat"], 2)}
                               for _, r in univariate_ic_df.head(5).iterrows()]})

In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================
importance_df = (pd.DataFrame({"feature": FEATURE_COLS, "importance": avg_feature_importance})
                 .sort_values("importance", ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(9, 6))
s = importance_df.sort_values("importance")
ax.barh(s["feature"], s["importance"], color="#4472C4")
ax.set_title("Average feature importance across walk-forward folds")
ax.set_xlabel("Importance (gain)")
plt.tight_layout(); plt.show()
display(importance_df.head(10))

if HAS_SHAP and last_model is not None:
    try:
        sample_X = last_X_test.sample(min(500, len(last_X_test)), random_state=CFG.random_state)
        shap_values = shap.TreeExplainer(last_model).shap_values(sample_X)
        log.info("SHAP computed on the final fold's test set.")
    except Exception as e:
        shap_values = None
        log.warning(f"SHAP failed ({e}) -- model-native importances only.")

rec("S5_top_features", [{"f": r["feature"], "imp": _j(r["importance"])}
                        for _, r in importance_df.head(6).iterrows()])

---
# Part II — Portfolio Construction

## 6. From Signal to Positions

A ranking is not a portfolio. Four construction decisions turned out to matter
more than any modelling choice:

**Proportional weighting.** Weighting every name by its within-sector rank,
rather than equal-weighting two extreme quintiles, uses the whole
cross-section. A quintile book ignores 60% of the ranking, and in this data
most of the signal sat in the middle. Worth +0.45 of Sharpe on its own.

**Risk-parity sizing.** Dividing each weight by the name's volatility
equalises risk contribution. Without it, volatile names dominate the variance
while carrying the same expected return.

**Beta-neutral legs.** One dollar long against one dollar short is not zero
market exposure when the legs hold different kinds of stock. Sizing the legs so
their betas offset removed an uncompensated −0.21 beta that was costing roughly
3%/yr.

**Cost realism.** Market impact scales with the square root of participation
in each name's own daily volume; short borrow is charged monthly as a holding
cost, not a trading cost. Costs turned out to be the binding constraint on
everything.

In [ ]:
# ============================================================
# PORTFOLIO UTILITIES AND COST MODEL
# ============================================================
# An earlier implementation used THREE different transaction-cost conventions in tables that sat next
# to each other: `turnover * c` for the ML book, `turnover * c * 2` for the
# trend book, `turnover * c` again for the long-only leg. One-way turnover
# measured against gross notional costs 2 * turnover * c on a 1x long / 1x
# short book, so the ML baseline's costs were understated 2x against the
# strategy it was being compared to. One helper now, used everywhere.

def cost_of(turnover, bps=None, gross=2.0):
    """One-way turnover is measured against gross notional, so a round trip on
    a 1x long / 1x short book costs 2 * turnover * rate."""
    bps = CFG.transaction_cost_bps if bps is None else bps
    return turnover * (bps / 10_000) * gross


def one_way_turnover(df, weight_col, month_col="month", ticker_col="ticker"):
    """Half the L1 change in signed weights between consecutive rebalances."""
    w = df.pivot_table(index=month_col, columns=ticker_col, values=weight_col,
                       aggfunc="sum", fill_value=0.0).sort_index()
    return (w - w.shift(1).fillna(0.0)).abs().sum(axis=1) / 2.0


def performance_summary(returns, freq=12, label="Strategy"):
    r = pd.Series(returns).dropna()
    if len(r) == 0:
        return {"label": label, "ann_return": np.nan, "total_return": np.nan,
                "ann_vol": np.nan, "sharpe": np.nan, "max_drawdown": np.nan,
                "calmar": np.nan, "hit_rate": np.nan, "n_months": 0}
    cum = (1 + r).cumprod()
    years = len(r) / freq
    terminal = cum.iloc[-1]
    # a negative terminal value makes the CAGR formula return NaN or a
    # complex number. Report total_return alongside so the row is still legible.
    cagr = (terminal ** (1 / years) - 1) if (years > 0 and terminal > 0) else np.nan
    ann_vol = r.std(ddof=1) * np.sqrt(freq)
    sharpe = (r.mean() * freq) / ann_vol if ann_vol > 0 else np.nan
    max_dd = (cum / cum.cummax() - 1).min()
    return {"label": label, "ann_return": cagr, "total_return": terminal - 1,
            "ann_vol": ann_vol, "sharpe": sharpe, "max_drawdown": max_dd,
            "calmar": (cagr / abs(max_dd)) if (max_dd and pd.notna(cagr) and max_dd != 0) else np.nan,
            "hit_rate": (r > 0).mean(), "n_months": len(r)}


FMT = {"ann_return": "{:.2%}", "total_return": "{:.1%}", "ann_vol": "{:.2%}",
       "sharpe": "{:.2f}", "max_drawdown": "{:.2%}", "calmar": "{:.2f}",
       "hit_rate": "{:.1%}", "turnover": "{:.2%}"}
print("portfolio utilities ready")

In [ ]:
# ============================================================
# SECTOR-NEUTRAL CONSTRUCTION AND CAPACITY CAPS
# ============================================================
def build_month_end_portfolio(df, n_quantiles=5, sector_neutral=True):
    """Exactly one observation per ticker-month; signal frozen at month end;
    long/short legs selected WITHIN sector when sector_neutral is on."""
    df = df.copy()
    if df.duplicated(["ticker", "month"]).any():
        raise RuntimeError("Duplicate ticker-month rows before portfolio construction.")
    out = []
    for month, g in df.groupby("month", sort=True):
        g = g.copy()
        if sector_neutral:
            g["pred_adj"] = g["pred"] - g.groupby("sector")["pred"].transform("mean")
            g["pct_rank"] = g.groupby("sector")["pred_adj"].rank(pct=True, method="first")
        else:
            g["pred_adj"] = g["pred"]
            g["pct_rank"] = g["pred_adj"].rank(pct=True, method="first")
        g["long_flag"] = g["pct_rank"] >= (1 - 1 / n_quantiles)
        g["short_flag"] = g["pct_rank"] <= (1 / n_quantiles)
        out.append(g)
    return pd.concat(out, ignore_index=True)


def apply_turnover_control(df, entry_pct, exit_pct):
    """No-trade band / hysteresis. A name already held stays while it remains
    inside the WIDER exit band; it only enters by crossing the NARROWER entry
    band. Standard buffering practice against turnover-driven costs."""
    df = df.sort_values("month").copy()
    prev_long, prev_short = set(), set()
    out = []
    for m in sorted(df["month"].unique()):
        g = df[df["month"] == m].copy()
        entry_long = set(g.loc[g["pct_rank"] >= (1 - entry_pct), "ticker"])
        entry_short = set(g.loc[g["pct_rank"] <= entry_pct, "ticker"])
        hold_long = set(g.loc[g["pct_rank"] >= (1 - exit_pct), "ticker"]) & prev_long
        hold_short = set(g.loc[g["pct_rank"] <= exit_pct, "ticker"]) & prev_short
        this_long, this_short = entry_long | hold_long, entry_short | hold_short
        g["long_flag"] = g["ticker"].isin(this_long)
        g["short_flag"] = g["ticker"].isin(this_short)
        out.append(g)
        prev_long, prev_short = this_long, this_short
    return pd.concat(out, ignore_index=True)


def assign_proportional_weights(df, power=1.0, min_rank=None, smoothing=None,
                                risk_parity=None, vol_col=None, mode=None):
    """weight every name by its within-sector rank, across the FULL
    cross-section, instead of splitting capital equally across two extreme
    buckets.

    Why this matters here. The an earlier implementation ladder was monotone but its spread sat almost
    entirely in the Q1->Q2 step -- the middle of the cross-section. A quintile
    book holds only Q0 and Q4, so it discards exactly the part of the ranking
    that carried the signal while concentrating capital in the tails, where
    estimation error is largest. Weighting by rank is also what Grinold's
    fundamental law assumes when it says IR is roughly IC x sqrt(breadth):
    breadth means every name one hold, not just the extremes.

    Centred rank in [-0.5, +0.5]; positives form the long leg, negatives the
    short leg, each normalised to 1.0 gross so the book stays comparable to the
    quintile version. `power` > 1 tilts back toward the tails.

    Two turnover controls, both necessary. Left naive, rank weighting gives
    EVERY name a position, so every name trades every month -- in testing that
    raised cost by ~40% against the quintile book and wiped out the benefit of
    the wider capture.
      `min_rank`  drops names inside the middle of the ranking, where the
                  weight is tiny and the only reliable thing it generates is
                  turnover. 0.15 keeps roughly the top and bottom third, still
                  far wider than a quintile's top and bottom fifth.
      `smoothing` blends this month's target weights with last month's actual
                  ones. 1.0 trades all the way to target; 0.5 halves the
                  distance, roughly halving turnover for a small tracking cost."""
    min_rank = CFG.proportional_min_rank if min_rank is None else min_rank
    smoothing = CFG.weight_smoothing if smoothing is None else smoothing
    risk_parity = CFG.risk_parity if risk_parity is None else risk_parity
    vol_col = CFG.risk_parity_vol_col if vol_col is None else vol_col

    df = df.copy()
    df["long_weight"] = 0.0
    df["short_weight"] = 0.0
    key = ["month", "sector"] if CFG.sector_neutral else ["month"]
    r = df.groupby(key)["pred"].transform(lambda s: s.rank(pct=True) - 0.5)
    r = r.where(r.abs() >= min_rank, 0.0)                     # trim the middle
    df["prop_score"] = np.sign(r) * (r.abs() ** power)

    # the risk-sizing test: RISK-PARITY SIZING. Weighting by signal rank alone means a volatile
    # name and a calm name with the same rank take very different amounts of
    # RISK while carrying the same expected return, so the volatile names
    # dominate the book's variance without earning more. Dividing by each
    # name's volatility equalises risk contribution.
    # This is the version of "reduce volatility" that actually helps. Scaling
    # the whole book down changes nothing -- it halves vol AND halves alpha,
    # leaving Sharpe identical. Reallocating risk between names cuts vol while
    # leaving expected return roughly intact: measured +0.22 Sharpe in a
    # controlled test, and slightly better here because the feature IC comparison found high-ivol
    # names underperform (ivol_21 IC -0.006).
    mode = CFG.risk_parity_mode if mode is None else mode
    if mode == "downside" and CFG.risk_parity_dvol_col in df.columns:
        vol_col = CFG.risk_parity_dvol_col
    if risk_parity and vol_col in df.columns:
        v = df[vol_col].astype(float)
        v = v.where(np.isfinite(v) & (v > 0))
        # cross-sectional median fallback keeps a missing vol from becoming a
        # huge position rather than dropping the name
        v = v.fillna(v.groupby(df["month"]).transform("median")).fillna(v.median())
        v = v.clip(lower=v.quantile(0.05), upper=v.quantile(0.95))   # tame the tails
        df["prop_score"] = df["prop_score"] / v

    months = sorted(df["month"].unique())
    groups = df.groupby("month").groups
    prev = {}
    for m in months:
        idx = groups[m]
        g = df.loc[idx]
        pos, neg = g["prop_score"].clip(lower=0), (-g["prop_score"]).clip(lower=0)
        tgt_l = (pos / pos.sum()) if pos.sum() > 0 else pos * 0.0
        tgt_s = (neg / neg.sum()) if neg.sum() > 0 else neg * 0.0
        tgt = pd.Series(tgt_l.values - tgt_s.values, index=g["ticker"].values)

        if smoothing < 1.0 and prev:
            p = pd.Series(prev).reindex(tgt.index).fillna(0.0)
            blended = p + smoothing * (tgt - p)
            # renormalise each side back to 1.0 gross so exposure stays constant
            bl, bs = blended.clip(lower=0), (-blended).clip(lower=0)
            blended = (bl / bl.sum() if bl.sum() > 0 else bl) - \
                      (bs / bs.sum() if bs.sum() > 0 else bs)
            tgt = blended

        df.loc[idx, "long_weight"] = tgt.clip(lower=0).values
        df.loc[idx, "short_weight"] = (-tgt).clip(lower=0).values
        prev = tgt[tgt.abs() > 1e-12].to_dict()

    df["long_flag"] = df["long_weight"] > 0
    df["short_flag"] = df["short_weight"] > 0
    df["signed_weight"] = df["long_weight"] - df["short_weight"]
    return df


def assign_weights(df, signal_weighted=True):
    """Equal capital per sector within each leg, then either equal-weight or
    conviction-weight (|pred_adj|) within each sector-leg."""
    df = df.copy()
    df["long_weight"] = 0.0
    df["short_weight"] = 0.0
    for month, idx in df.groupby("month").groups.items():
        g = df.loc[idx]
        for flag_col, wcol in [("long_flag", "long_weight"), ("short_flag", "short_weight")]:
            sel = g[g[flag_col]]
            sectors = sel["sector"].dropna().unique()
            if len(sectors) == 0:
                continue
            sw = 1.0 / len(sectors)
            for sec in sectors:
                names = sel[sel["sector"] == sec].index
                if signal_weighted:
                    conv = df.loc[names, "pred_adj"].abs()
                    if conv.sum() > 0:
                        w = sw * (conv / conv.sum())
                        df.loc[names, wcol] = w * (sw / w.sum())
                    else:
                        df.loc[names, wcol] = sw / len(names)
                else:
                    df.loc[names, wcol] = sw / len(names)
    df["signed_weight"] = df["long_weight"] - df["short_weight"]
    return df


def cap_leg_to_adv(df, weight_col, max_pct_adv, assumed_aum, max_passes=6):
    """Water-filling ADV cap, per sector-leg per month. Names that breach their
    cap are LOCKED at the cap and never rescaled again; only still-free names
    absorb the remaining budget.

    This can legitimately leave a leg UNDER-deployed if the combined liquidity
    of its names cannot absorb the intended size. That shortfall is real and is
    reported rather than papered over -- a uniform final renormalisation would
    push already-capped names back above their cap, defeating the point."""
    df = df.copy()
    for month, idx in df.groupby("month").groups.items():
        g = df.loc[idx]
        for sector in g["sector"].dropna().unique():
            sel = g[(g["sector"] == sector) & (g[weight_col] > 0)].index
            if len(sel) == 0:
                continue
            w = df.loc[sel, weight_col].copy()
            target_total = w.sum()
            adv = (pd.to_numeric(df.loc[sel, "dollar_volume_raw"], errors="coerce")
                   .replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(lower=0.0))
            cap = (max_pct_adv * adv / assumed_aum).clip(lower=0.0)
            capped = pd.Series(False, index=w.index)
            budget = target_total
            for _ in range(max_passes):
                free = ~capped
                if free.sum() == 0:
                    break
                fw = w[free]
                if fw.sum() > 0:
                    w[free] = fw * (budget / fw.sum())
                breach = free & (w > cap)
                if not breach.any():
                    break
                w[breach] = cap[breach].fillna(w[breach])
                capped = capped | breach
                budget = target_total - w[capped].sum()
            df.loc[w.index, weight_col] = w.values
    return df


print("construction functions ready")

In [ ]:
# ============================================================
# PORTFOLIO CONSTRUCTION AND BACKTEST
# ============================================================
def build_and_backtest(res, cfg, label="ML L/S", verbose=True,
                       entry_pct=None, exit_pct=None, holding_months=None,
                       weighting_scheme=None, max_leverage=None,
                       prop_min_rank=None, prop_smoothing=None,
                       gross_long=None, gross_short=None, aum=None,
                       apply_vol_target=None, beta_neutral=None, risk_parity=None,
                       rp_mode=None, leg_risk_parity=None,
                       vol_target_mode=None, dd_delever=None):
    """End-to-end: rank -> select -> weight -> ADV cap -> vol target ->
    turnover -> realistic costs. Returns a dict of monthly series plus
    diagnostics. Packaged as a function so the holdout section can score the holdout
    with identical mechanics."""
    entry_pct = cfg.turnover_entry_pct if entry_pct is None else entry_pct
    exit_pct = cfg.turnover_exit_pct if exit_pct is None else exit_pct
    holding_months = cfg.ml_holding_months if holding_months is None else holding_months

    scheme = weighting_scheme or cfg.weighting_scheme
    g_long = cfg.gross_long if gross_long is None else gross_long
    g_short = cfg.gross_short if gross_short is None else gross_short
    df = build_month_end_portfolio(res, cfg.n_quantiles, cfg.sector_neutral)
    if scheme == "proportional":
        # no-trade bands are a quintile-membership device; rank weights move
        # continuously, so turnover is controlled by trimming and smoothing
        df = assign_proportional_weights(df, cfg.proportional_power,
                                         min_rank=prop_min_rank, smoothing=prop_smoothing,
                                         risk_parity=risk_parity, mode=rp_mode)
    else:
        if cfg.turnover_control_enabled:
            df = apply_turnover_control(df, entry_pct, exit_pct)
        df = assign_weights(df, cfg.use_signal_weighted_sizing)

    # hold positions for N months instead of re-deciding every month.
    # Costs were 7.44%/yr on 123.5% monthly turnover in earlier testing -- 59% of
    # the total damage -- and cutting turnover pays off whichever way the
    # signal points, so it is the one lever with a sign-independent payoff.
    if holding_months > 1:
        act = sorted(df["month"].unique())[::holding_months]
        w = df.pivot_table(index="month", columns="ticker", values="signed_weight",
                           aggfunc="sum", fill_value=0.0).sort_index()
        held = w.copy()
        held.loc[~held.index.isin(act)] = np.nan       # only act on rebalance months
        held = held.ffill().fillna(0.0)                # carry the book forward in between
        df = df.merge(held.stack().rename("held_weight").reset_index(),
                      on=["month", "ticker"], how="left")
        df["signed_weight"] = df["held_weight"].fillna(0.0)
        df["long_weight"] = df["signed_weight"].clip(lower=0)
        df["short_weight"] = (-df["signed_weight"]).clip(lower=0)
        df["long_flag"] = df["signed_weight"] > 0
        df["short_flag"] = df["signed_weight"] < 0

    # scale the two legs independently. gross_short = 0 gives a long-only
    # tilt, which pays trading cost on gross 1.0 instead of 2.0 and avoids the
    # borrow fee entirely. an earlier implementation's leg decomposition showed both legs returning
    # 10-14%/yr -- i.e. both were essentially the market -- so the book was
    # paying double freight to net out a spread of about one percent.
    if (g_long, g_short) != (1.0, 1.0):
        df["long_weight"] *= g_long
        df["short_weight"] *= g_short
        df["signed_weight"] = df["long_weight"] - df["short_weight"]

    # ---- LEG RISK PARITY ----------------------------------------------
    # Equalise the RISK each leg contributes rather than the dollars. Tested in
    # the leg-balance test and left off by default: once names are weighted by inverse
    # volatility the two legs are already close to risk-balanced, so this
    # moved Sharpe 0.90 -> 0.91 on synthetic data. Exposed because the real
    # cross-section may be less symmetric than the simulation.
    leg_risk_parity = CFG.leg_risk_parity if leg_risk_parity is None else leg_risk_parity
    if leg_risk_parity and g_short > 0 and cfg.risk_parity_vol_col in df.columns:
        vv = df[cfg.risk_parity_vol_col].astype(float)
        vv = vv.where(np.isfinite(vv) & (vv > 0)).fillna(vv.median()).fillna(0.3)
        for _m, _idx in df.groupby("month").groups.items():
            rl = float(np.sqrt(((df.loc[_idx, "long_weight"] * vv.loc[_idx]) ** 2).sum()))
            rs = float(np.sqrt(((df.loc[_idx, "short_weight"] * vv.loc[_idx]) ** 2).sum()))
            if rl > 0 and rs > 0:
                k = float(np.clip(rl / rs, 0.25, 4.0))
                den = g_long + k * g_short
                if den > 1e-12:
                    sL = (g_long + g_short) / den
                    df.loc[_idx, "long_weight"] *= sL
                    df.loc[_idx, "short_weight"] *= k * sL
        df["signed_weight"] = df["long_weight"] - df["short_weight"]

    # ---- BETA NEUTRALISATION ------------------------------------------
    # Dollar-neutral is not market-neutral. An earlier implementation measured a beta of -0.21 on the
    # composite's "market-neutral" book; in a market returning +14.13%/yr that
    # cost about 3 points a year and buried a +2.52% alpha. The tilt is
    # structural, not noise -- the composite's -ivol_21 and +max_ret_21
    # components put low-volatility (and therefore low-beta) names on the long
    # side and high-beta names on the short side, so the book runs an implicit
    # betting-against-beta trade on top of its stock selection.
    # Scaling the short leg so that sum(w_long x beta) == sum(w_short x beta)
    # removes that exposure. The book is then no longer exactly 1-for-1 in
    # dollars, which is the intended trade-off: it is neutral on the thing that
    # actually moves the P&L. This uses a TRAILING beta estimate, so ex-ante
    # neutrality does not guarantee ex-post -- the beta-neutrality test verifies the realised beta
    # rather than trusting the construction.
    beta_neutral = cfg.beta_neutral if beta_neutral is None else beta_neutral
    beta_report = {}
    if beta_neutral and g_short > 0 and cfg.beta_col in df.columns:
        b = df[cfg.beta_col].astype(float)
        b = b.where(np.isfinite(b)).fillna(1.0).clip(0.0, 3.0)
        lo, hi = cfg.beta_scale_bounds
        pre, post = [], []
        for month, idx in df.groupby("month").groups.items():
            bl = float((df.loc[idx, "long_weight"] * b.loc[idx]).sum())
            bs = float((df.loc[idx, "short_weight"] * b.loc[idx]).sum())
            if bs > 1e-9 and bl > 0:
                pre.append(bl - bs)
                k = float(np.clip(bl / bs, lo, hi))
                # scale BOTH legs, preserving total gross exposure.
                # An earlier implementation scaled only the short leg (short *= k). Whenever the long
                # leg held higher-beta names, k > 1 pushed short gross above
                # 1.0, which tripped the exposure assertion -- and, worse, it
                # silently increased total gross so costs and risk crept up
                # without appearing anywhere. Solving
                #     s_L * beta_L == s_S * beta_S            (beta nets to 0)
                #     s_L*g_long + s_S*g_short == g_long+g_short   (gross fixed)
                # gives s_L = (g_long+g_short)/(g_long + k*g_short), s_S = k*s_L.
                denom = g_long + k * g_short
                if denom > 1e-12:
                    s_L = (g_long + g_short) / denom
                    s_S = k * s_L
                    df.loc[idx, "long_weight"] *= s_L
                    df.loc[idx, "short_weight"] *= s_S
                post.append(float((df.loc[idx, "long_weight"] * b.loc[idx]).sum())
                            - float((df.loc[idx, "short_weight"] * b.loc[idx]).sum()))
        df["signed_weight"] = df["long_weight"] - df["short_weight"]
        beta_report = {"ex_ante_beta_before": float(np.mean(pre)) if pre else np.nan,
                       "ex_ante_beta_after": float(np.mean(post)) if post else np.nan}

    # ---- capacity: no position may exceed a share of its own daily volume --
    cap_report = {}
    if cfg.max_position_pct_adv is not None:
        before_l = df.groupby("month")["long_weight"].sum().mean()
        before_s = df.groupby("month")["short_weight"].sum().mean()
        df = cap_leg_to_adv(df, "long_weight", cfg.max_position_pct_adv, cfg.assumed_aum_usd)
        df = cap_leg_to_adv(df, "short_weight", cfg.max_position_pct_adv, cfg.assumed_aum_usd)
        df["signed_weight"] = df["long_weight"] - df["short_weight"]
        cap_report = {"long_gross_before": before_l,
                      "long_gross_after": df.groupby("month")["long_weight"].sum().mean(),
                      "short_gross_before": before_s,
                      "short_gross_after": df.groupby("month")["short_weight"].sum().mean()}

    # ---- exposure audit: a non-empty leg may fall short of 1.0 but never exceed it
    exp = df.groupby("month").agg(long_gross=("long_weight", "sum"),
                                  short_gross=("short_weight", "sum"),
                                  net=("signed_weight", "sum"),
                                  gross=("signed_weight", lambda x: x.abs().sum()),
                                  n_long=("long_flag", "sum"), n_short=("short_flag", "sum"))
    # with gross-preserving beta neutralisation the two legs no longer
    # each sit at their nominal target -- one is scaled up and the other down
    # by offsetting amounts. The invariant that still holds, and the one that
    # actually matters for risk and cost, is TOTAL gross. Checking per-leg
    # bounds here was what raised "Short-leg gross exceeds its target".
    TOL = 1e-6
    valid = (exp["n_long"] > 0) & (exp["n_short"] > 0)
    target_gross = g_long + g_short
    if not (exp.loc[valid, "long_gross"] <= target_gross + TOL).all():
        raise RuntimeError(
            f"Long-leg gross exceeds total target {target_gross:.2f} after capacity controls "
            f"(max {exp.loc[valid, 'long_gross'].max():.4f}).")
    if valid.any() and not (exp.loc[valid, "gross"] <= target_gross + TOL).all():
        raise RuntimeError(
            f"Total gross exposure exceeds its target of {target_gross:.2f} "
            f"(max {exp.loc[valid, 'gross'].max():.4f}). Capacity controls should only "
            "ever reduce exposure, so this means a sizing step scaled the book up.")
    if not beta_neutral:
        # without beta neutralisation each leg should still respect its own cap
        if not (exp.loc[valid, "long_gross"] <= g_long + TOL).all():
            raise RuntimeError("Long-leg gross exceeds its target after capacity controls.")
        if valid.any() and g_short > 0 and not (exp.loc[valid, "short_gross"] <= g_short + TOL).all():
            raise RuntimeError("Short-leg gross exceeds its target after capacity controls.")

    # ---- leg returns --------------------------------------------------------
    gross_ret = df.assign(_c=df["signed_weight"] * df["fwd_ret"]).groupby("month")["_c"].sum()
    long_leg = df.assign(_c=df["long_weight"] * df["fwd_ret"]).groupby("month")["_c"].sum()
    # track the short leg separately. the break-even analysis's intercept implies the book
    # loses ~1.2%/yr gross at ZERO skill, and the prime suspect is the short leg
    # paying the cross-sectional skew premium -- the target is demeaned by the
    # MEDIAN while stock returns are right-skewed, so the average name the short
    # leg is betting against has a positive expected excess return.
    short_leg = df.assign(_c=df["short_weight"] * df["fwd_ret"]).groupby("month")["_c"].sum()

    # ---- volatility targeting (leak-free: .shift(1) excludes the current month)
    # a TILTED book is not vol-targeted by default. Scaling a long-only
    # equity portfolio to 10% annual volatility roughly halves its market
    # exposure, which makes any comparison against SPY incoherent -- one end up
    # measuring a half-weight index fund. The market-neutral book, whose whole
    # point is to have no market exposure, is unaffected.
    is_tilted = (g_long != g_short)
    do_vt = (not is_tilted) if apply_vol_target is None else apply_vol_target
    vt_mode = cfg.vol_target_mode if vol_target_mode is None else vol_target_mode
    if vt_mode == "off":
        do_vt = False
    if do_vt and cfg.target_annual_volatility is not None and cfg.vol_target_lookback_months > 0:
        tgt_m = cfg.target_annual_volatility / np.sqrt(12)
        realized = gross_ret.shift(1).rolling(cfg.vol_target_lookback_months, min_periods=1).std()
        lev_cap = cfg.max_vol_target_leverage if max_leverage is None else max_leverage
        # ONE-SIDED targeting. Volatility is predictable (autocorrelation of
        # |returns| ~0.50 at realistic persistence) while direction is not
        # (~-0.03), so scaling by volatility is defensible and timing direction
        # is not. But levering UP a weak book multiplies cost superlinearly --
        # impact goes as dollars^1.5 -- which is what made symmetric targeting
        # the worst rule tested (net Sharpe -1.32 against -0.88 for constant
        # exposure). One-sided keeps the de-risking and drops the levering up.
        if vt_mode.startswith("onesided"):
            lev_cap = min(lev_cap, 1.0)
        if vt_mode == "onesided_downside":
            neg = gross_ret.where(gross_ret < 0, 0.0)
            realized = (neg.pow(2).shift(1)
                        .rolling(cfg.vol_target_lookback_months, min_periods=1).mean()
                        .pow(0.5) * np.sqrt(2))
        scale = (tgt_m / realized.replace(0, np.nan)).clip(
            lower=cfg.vol_target_floor, upper=lev_cap).fillna(1.0)

        # optional: additionally cut exposure while below the high-water mark.
        # CAUSAL -- the level for month t uses the drawdown through t-1.
        if (cfg.dd_delever if dd_delever is None else dd_delever):
            lv, cum, peak, seq = 1.0, 1.0, 1.0, []
            for x in (gross_ret * scale).values:
                seq.append(lv)
                cum *= (1 + lv * x); peak = max(peak, cum)
                lv = cfg.dd_delever_scale if (cum / peak - 1) < cfg.dd_delever_trigger else 1.0
            scale = scale * pd.Series(seq, index=gross_ret.index)
    else:
        realized = pd.Series(np.nan, index=gross_ret.index)
        scale = pd.Series(1.0, index=gross_ret.index)
    vt_ret = gross_ret * scale
    df["scaled_signed_weight"] = df["signed_weight"] * df["month"].map(scale).fillna(1.0)

    # ---- turnover -----------------------------------------------------------
    turnover = one_way_turnover(df, "signed_weight")
    turnover_scaled = one_way_turnover(df, "scaled_signed_weight")
    flat_cost = cost_of(turnover_scaled, cfg.transaction_cost_bps)

    # ---- realistic costs: sqrt market impact + short borrow ----------------
    # A flat bps number does not distinguish a large trade in a thin name from
    # the same dollars in a mega-cap. Impact scales with participation (trade
    # size relative to that name's own average daily volume) under a
    # square-root law -- the practitioner standard; linear overweights
    # high-participation trades. Short borrow is a HOLDING cost, charged every
    # month on gross short exposure whether or not traded.
    aum = cfg.assumed_aum_usd if aum is None else aum
    w_mat = df.pivot_table(index="month", columns="ticker", values="scaled_signed_weight",
                           aggfunc="sum", fill_value=0.0).sort_index()
    adv_mat = df.pivot_table(index="month", columns="ticker", values="dollar_volume_raw",
                             aggfunc="first")
    delta = (w_mat - w_mat.shift(1).fillna(0.0)).abs()
    dollars = delta * aum
    participation = ((dollars / adv_mat.replace(0, np.nan)) * 100).clip(upper=100)
    impact_bps = cfg.market_impact_bps_per_sqrt_pct_participation * np.sqrt(participation.fillna(0))
    impact = (dollars * impact_bps / 10_000).sum(axis=1) / aum
    short_gross = df.groupby("month")["short_weight"].sum()
    borrow = short_gross * (cfg.short_borrow_cost_bps_annual / 10_000 / 12)
    realistic_cost = impact.reindex(gross_ret.index).fillna(0) + borrow.reindex(gross_ret.index).fillna(0)

    net = vt_ret - realistic_cost
    long_only_net = long_leg - cost_of(turnover, cfg.transaction_cost_bps, gross=1.0)
    book_gross = g_long + g_short

    if verbose:
        print(f"=== {label} ===")
        if cap_report:
            print(f"  ADV cap: long gross {cap_report['long_gross_before']:.4f} -> "
                  f"{cap_report['long_gross_after']:.4f} | short "
                  f"{cap_report['short_gross_before']:.4f} -> {cap_report['short_gross_after']:.4f}")
        print(f"  scheme: {scheme} | names/month: {exp['n_long'].mean():.0f} long, "
              f"{exp['n_short'].mean():.0f} short")
        if beta_report:
            print(f"  ex-ante beta: {beta_report['ex_ante_beta_before']:+.3f} -> "
                  f"{beta_report['ex_ante_beta_after']:+.3f} after neutralisation")
        print(f"  leg returns: long {long_leg.mean() * 12:+.2%}/yr, "
              f"short-leg holdings {short_leg.mean() * 12:+.2%}/yr "
              f"(the book earns the negative of this)")
        print(f"  one-way turnover: {turnover.mean():.2%} unscaled | {turnover_scaled.mean():.2%} scaled")
        print(f"  mean vol-target leverage: {scale.mean():.2f}x")
        print(f"  cost: flat-bps {flat_cost.mean():.4%}/mo | realistic {realistic_cost.mean():.4%}/mo "
              f"(impact {impact.mean():.4%}, borrow {borrow.mean():.4%})")
        print(f"  participation: mean {participation.stack().mean():.2f}% of ADV, "
              f"95th pct {participation.stack().quantile(0.95):.2f}%")

    return {"df": df, "gross": gross_ret, "vol_targeted": vt_ret, "net": net,
            "long_only_net": long_only_net, "long_leg": long_leg, "short_leg": short_leg,
            "scheme": scheme,
            "turnover": turnover, "turnover_scaled": turnover_scaled,
            "book_gross": book_gross, "gross_long": g_long, "gross_short": g_short,
            "is_tilted": is_tilted, "vol_targeted_applied": do_vt,
            "vol_target_mode": vt_mode,
            "beta_report": beta_report, "beta_neutral_applied": bool(beta_neutral and g_short > 0),
            "flat_cost": flat_cost, "realistic_cost": realistic_cost,
            "impact": impact, "borrow": borrow, "scale": scale, "realized_vol": realized,
            "exposure": exp, "participation": participation, "cap_report": cap_report}


BT = build_and_backtest(results_df, CFG, label="ML L/S (development period)")

long_short_gross = BT["gross"]
long_short_gross_vol_targeted = BT["vol_targeted"]
long_short_net = BT["net"]
long_only_net = BT["long_only_net"]
turnover_series = BT["turnover"]
transaction_cost_series = BT["realistic_cost"]
portfolio_df = BT["df"]

raw_universe_ret = (panel_me_dev[panel_me_dev["date"].isin(results_df["date"].unique())]
                    .groupby("month")["fwd_ret"].mean())

cum_ls_net = (1 + long_short_net).cumprod()
cum_long_only_net = (1 + long_only_net).cumprod()
cum_universe = (1 + raw_universe_ret.reindex(long_short_net.index)).cumprod()

rec("the corresponding section", {"turnover_mean": _j(turnover_series.mean(), 4),
           "cost_realistic_monthly": _j(transaction_cost_series.mean(), 5),
           "cost_flat_monthly": _j(BT["flat_cost"].mean(), 5),
           "mean_leverage": _j(BT["scale"].mean(), 3)})

In [ ]:
# ============================================================
# GROSS VERSUS NET DECOMPOSITION
# ============================================================
gvn = pd.DataFrame({"gross (pre-cost, pre-vol-target)": long_short_gross,
                    "vol-targeted (pre-cost)": long_short_gross_vol_targeted,
                    "net (post realistic cost)": long_short_net})
cum_gvn = (1 + gvn).cumprod()

print("=== GROSS VS NET ===")
print(f"  mean monthly gross        : {long_short_gross.mean():+.4%}")
print(f"  mean monthly vol-targeted : {long_short_gross_vol_targeted.mean():+.4%}")
print(f"  mean monthly net          : {long_short_net.mean():+.4%}")
print(f"  mean monthly cost drag    : {transaction_cost_series.mean():.4%} "
      f"({transaction_cost_series.mean() * 12:.2%} annualised)")
denom = long_short_gross_vol_targeted.abs().sum()
print(f"  costs as a share of gross P&L magnitude: "
      f"{(transaction_cost_series.sum() / denom if denom else np.nan):.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for c in cum_gvn.columns:
    axes[0].plot(cum_gvn.index, cum_gvn[c], label=c, linewidth=1.8)
axes[0].axhline(1.0, color="gray", linewidth=0.8, linestyle=":")
axes[0].set_title("Cumulative growth of $1: gross vs vol-targeted vs net")
axes[0].legend(loc="upper left", fontsize=8)

axes[1].plot(turnover_series.index, turnover_series, label="one-way turnover", color="#9467bd")
axes[1].plot(transaction_cost_series.index, transaction_cost_series,
             label="realistic cost drag", color="#c0392b")
axes[1].set_title("Turnover and resulting cost drag")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

## 7. Performance Measurement

Every candidate is measured with the same machinery: Newey–West t-statistics
that correct for serial dependence, deflated Sharpe ratios that account for
the number of variants searched, and factor attribution against the market,
momentum and low-volatility so that "alpha" means residual alpha rather than a
repackaged factor tilt.

In [ ]:
# ============================================================
# SIGNIFICANCE AND ATTRIBUTION MACHINERY
# ============================================================
# Roughly thirty strategy variants have now been scored against this sample
# across development. Past that point the best Sharpe found is largely the
# maximum of thirty noisy estimates, not a discovery. These three tools say how
# much of an apparent edge survives that.

def newey_west_tstat(returns, lags=None):
    """t-stat on the mean monthly return, corrected for the fact that
    consecutive months of a strategy's returns are not fully independent.
    Without the correction, t-stats come out too big."""
    r = pd.Series(returns).dropna()
    if len(r) < 12:
        return np.nan, np.nan
    if not HAS_SM:
        t = r.mean() / (r.std(ddof=1) / np.sqrt(len(r)))
        return float(t), float(2 * (1 - norm.cdf(abs(t))))
    lags = lags if lags is not None else max(1, int(4 * (len(r) / 100) ** (2 / 9)))
    m = sm.OLS(r.values, np.ones(len(r))).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return float(m.tvalues[0]), float(m.pvalues[0])


def deflated_sharpe(returns, n_trials, freq=12):
    """Bailey & Lopez de Prado. P(true Sharpe > 0) AFTER accounting for how
    many variants were searched over the same data.

    Calibration on 104 months: a realised Sharpe of +0.83 scores 96.9% at one
    trial and 2.8% at thirty. That gap is the entire point."""
    r = pd.Series(returns).dropna()
    n = len(r)
    if n < 12 or r.std(ddof=1) == 0:
        return np.nan
    sr = r.mean() / r.std(ddof=1)
    g3, g4 = stats.skew(r), stats.kurtosis(r, fisher=False)
    e, k = 0.5772156649, max(n_trials, 2)
    sr0 = np.sqrt(1 / (n - 1)) * ((1 - e) * norm.ppf(1 - 1 / k) + e * norm.ppf(1 - 1 / (k * np.e)))
    den = np.sqrt(1 - g3 * sr + (g4 - 1) / 4 * sr ** 2)
    if den <= 0:
        return np.nan
    return float(norm.cdf((sr - sr0) * np.sqrt(n - 1) / den))


def full_report(returns, label, n_trials=None):
    n_trials = CFG.n_trials_searched if n_trials is None else n_trials
    s = performance_summary(returns, label=label)
    t, p = newey_west_tstat(returns)
    s.update({"nw_tstat": t, "nw_pvalue": p,
              "deflated_sharpe_prob": deflated_sharpe(returns, n_trials)})
    return s


def _ls_factor(panel_src, col, name, ascending=False):
    """A generic long/short factor portfolio: top quintile minus bottom
    quintile, sorted on `col`. Used to check whether a strategy's return is
    just a repackaged factor tilt one could buy for a few basis points."""
    d = panel_src.dropna(subset=[col, "fwd_ret"]).copy()
    q = d.groupby("month")[col].transform(lambda s: s.rank(pct=True))
    hi = d[q >= 0.8].groupby("month")["fwd_ret"].mean()
    lo = d[q <= 0.2].groupby("month")["fwd_ret"].mean()
    return ((lo - hi) if ascending else (hi - lo)).rename(name)


def build_factors(panel_src):
    mkt = panel_src.groupby("month")["fwd_ret"].mean().rename("MKT")
    mom_c = "mom_252_raw" if "mom_252_raw" in panel_src.columns else "mom_252"
    vol_c = "vol_21_raw" if "vol_21_raw" in panel_src.columns else "vol_21"
    rev_c = "rev_5_raw" if "rev_5_raw" in panel_src.columns else None
    cols = [mkt, _ls_factor(panel_src, mom_c, "MOM"),
            _ls_factor(panel_src, vol_c, "LOWVOL", ascending=True)]
    if rev_c:
        cols.append(_ls_factor(panel_src, rev_c, "REVERSAL"))
    return pd.concat(cols, axis=1).dropna()


FACTORS = build_factors(panel_me_dev)


def factor_attribution(strategy_ret, label, factors=None, show=True):
    """Regress the strategy on the market and the tilts it is most likely to
    carry accidentally. The intercept is ALPHA -- what is left after everything
    one could have bought cheaply. A high R-squared with zero alpha means the
    'strategy' is a factor bet in disguise."""
    factors = FACTORS if factors is None else factors
    if not HAS_SM:
        return None
    y = pd.Series(strategy_ret).dropna()
    ix = factors.index.intersection(y.index)
    if len(ix) < 24:
        return None
    m = sm.OLS(y.loc[ix].values, sm.add_constant(factors.loc[ix].values)).fit(
        cov_type="HAC", cov_kwds={"maxlags": 3})
    names = ["alpha"] + list(factors.columns)
    tbl = pd.DataFrame({"beta": m.params, "tstat": m.tvalues, "pvalue": m.pvalues}, index=names)
    if show:
        print(f"\n=== FACTOR ATTRIBUTION: {label}   (R^2 = {m.rsquared:.3f}) ===")
        print(f"  monthly alpha {m.params[0]:+.4%}  (annualised {m.params[0] * 12:+.2%}), "
              f"t = {m.tvalues[0]:+.2f}")
        display(tbl.style.format({"beta": "{:+.4f}", "tstat": "{:+.2f}", "pvalue": "{:.3f}"}))
    return {"ann_alpha": m.params[0] * 12, "alpha_t": m.tvalues[0], "r2": m.rsquared,
            "betas": dict(zip(factors.columns, m.params[1:]))}


# ============================================================
# FORWARD-ALIGNED BENCHMARK + BETA INVARIANT
# ============================================================
# The strategy's return at month M is fwd_ret: measured FORWARD from month-end
# M, so it covers month M+1. An earlier implementation built the SPY series with pct_change() on the
# same index, which at month M covers month M. The two were offset by one
# period, the regression compared non-overlapping windows, and beta collapsed
# to ~0 -- which made a long-only equity book look market-neutral and turned
# its raw return into "alpha". SPY is now measured over exactly the same
# forward window as the strategy.

def spy_forward_returns(month_end_map, h=None):
    """SPY's return over the SAME forward window the strategy's fwd_ret uses."""
    if spy_prices is None or len(spy_prices) == 0:
        return None
    h = CFG.forward_return_days if h is None else h
    px = spy_prices.sort_index()
    out = {}
    for m, d in month_end_map.items():
        pos = px.index.searchsorted(pd.Timestamp(d), side="right") - 1
        if pos < 0 or pos + h >= len(px):
            continue
        out[m] = px.values[pos + h] / px.values[pos] - 1.0
    return pd.Series(out).sort_index() if out else None


BENCH_FWD = spy_forward_returns(panel_me_dev.groupby("month")["date"].max().to_dict())
# Full-sample benchmark, used only where holdout months must be priced (the
# holdout section). Built separately so the development-period tables above
# cannot accidentally reach past the holdout boundary.
BENCH_FWD_ALL = spy_forward_returns(panel_me.groupby("month")["date"].max().to_dict())
if BENCH_FWD is not None:
    print(f"Forward-aligned SPY benchmark: {len(BENCH_FWD)} months, "
          f"mean {BENCH_FWD.mean() * 12:+.2%}/yr")


def beta_vs_bench(strategy_ret, bench=None):
    """OLS beta and alpha against the forward-aligned benchmark."""
    bench = BENCH_FWD if bench is None else bench
    if bench is None:
        return {}
    y = pd.Series(strategy_ret).dropna()
    ix = y.index.intersection(bench.index)
    if len(ix) < 24:
        return {}
    x, yy = bench.loc[ix].values, y.loc[ix].values
    beta, icpt = (float(v) for v in np.polyfit(x, yy, 1))
    resid = yy - (beta * x + icpt)
    se = resid.std(ddof=2) / np.sqrt(len(ix))
    return {"beta_spy": beta, "alpha_ann": icpt * 12,
            "alpha_t": float(icpt / se) if se > 0 else np.nan,
            "n_obs": len(ix)}


def bootstrap_ci(returns, stat="sharpe", n_boot=2000, alpha=0.05, freq=12, seed=None):
    """stationary-block bootstrap confidence interval.

    A single Sharpe number says nothing about how confident one should be in
    it. Resampling in BLOCKS rather than individual months preserves the
    autocorrelation a strategy's returns actually have -- resampling month by
    month would pretend they are independent and produce intervals that are
    far too narrow."""
    r = pd.Series(returns).dropna().values
    n = len(r)
    if n < 24:
        return (np.nan, np.nan)
    rng_ = np.random.default_rng(CFG.random_state if seed is None else seed)
    block = max(3, int(round(n ** (1 / 3))))
    n_blocks = int(np.ceil(n / block))
    out = []
    for _ in range(n_boot):
        starts = rng_.integers(0, n, n_blocks)
        idx = np.concatenate([np.arange(s, s + block) % n for s in starts])[:n]
        samp = r[idx]
        sd = samp.std(ddof=1)
        if sd <= 0:
            continue
        out.append((samp.mean() * freq) / (sd * np.sqrt(freq)) if stat == "sharpe"
                   else samp.mean() * freq)
    if not out:
        return (np.nan, np.nan)
    return (float(np.percentile(out, 100 * alpha / 2)),
            float(np.percentile(out, 100 * (1 - alpha / 2))))


def with_ci(returns, label):
    """Headline stats reported next to how uncertain they are."""
    s = performance_summary(returns, label=label)
    lo, hi = bootstrap_ci(returns, "sharpe")
    rlo, rhi = bootstrap_ci(returns, "mean")
    s.update({"sharpe_lo": lo, "sharpe_hi": hi, "ret_lo": rlo, "ret_hi": rhi,
              "ci_excludes_zero": bool(np.isfinite(lo) and lo > 0)})
    s.update(decompose_return(returns))
    return s


def decompose_return(strategy_ret, label=""):
    """report every book as `return = alpha + beta x market`, so an
    uncompensated market exposure can never hide inside a headline number
    again. an earlier implementation's composite market-neutral book had +2.52% alpha and a -0.93%
    return; the difference was entirely a -0.21 beta."""
    d = beta_vs_bench(strategy_ret)
    if not d or BENCH_FWD is None:
        return {}
    mkt = float(BENCH_FWD.mean() * 12)
    d["market_ann"] = mkt
    d["beta_contribution"] = d["beta_spy"] * mkt
    d["realised_ann"] = float(pd.Series(strategy_ret).dropna().mean() * 12)
    return d


def check_beta_invariant(measured_beta, g_long, g_short, label, tol=0.40):
    """a book's measured market beta should be close to its stated net
    exposure -- a long-only equity book is ~1.0, a 130/30 is ~1.0, a
    dollar-neutral book is ~0. When the measured value is far from that, the
    benchmark is misaligned or the exposure is not what the config says. This
    is the check that would have caught the long-tilt test immediately in earlier work."""
    if measured_beta is None or not np.isfinite(measured_beta):
        return True
    expected = g_long - g_short
    ok = abs(measured_beta - expected) <= tol
    if not ok:
        print(f"    !! BETA INVARIANT FAILED for {label}: expected ~{expected:+.2f} "
              f"(gross_long {g_long} - gross_short {g_short}), measured "
              f"{measured_beta:+.2f}. Treat this row's alpha as unreliable.")
    return ok


print(f"significance machinery ready (statsmodels={HAS_SM}, "
      f"n_trials_searched={CFG.n_trials_searched})")

In [ ]:
# ============================================================
# PRIMARY STRATEGY PERFORMANCE
# ============================================================
ls_stats = full_report(long_short_net, "ML L/S (net)")
lo_stats = full_report(long_only_net, "Long-only leg (net)")
# report the gross book too. Without it it is impossible to tell a bad signal from
# an expensive one -- in earlier testing costs were 59% of the total damage.
gross_stats = full_report(long_short_gross, "ML L/S (gross, pre-cost)")
summary_table = pd.DataFrame([gross_stats, ls_stats, lo_stats]).set_index("label")
print("=== ML STRATEGY PERFORMANCE (development period) ===")
display(summary_table.style.format({**FMT, "nw_tstat": "{:.2f}", "nw_pvalue": "{:.3f}",
                                    "deflated_sharpe_prob": "{:.1%}"}))

# ---- quantile monotonicity ------------------------------------------------
qs = (results_df.assign(q=lambda x: x.groupby("month")["pred"].transform(
        lambda s: pd.qcut(s.rank(method="first"), CFG.n_quantiles, labels=False, duplicates="drop")))
      .groupby("q")[TARGET_COL].agg(["mean", "std", "count"]))
print("\n=== QUANTILE MONOTONICITY ===")
display(qs.style.format({"mean": "{:+.5f}", "std": "{:.4f}"}))

# ---- performance by calendar year -----------------------------------------
yrs = pd.to_datetime(long_short_net.index).year
yearly = pd.DataFrame([performance_summary(long_short_net[yrs == y], label=str(y))
                       for y in sorted(set(yrs)) if (yrs == y).sum() >= 3]).set_index("label")
print("\n=== PERFORMANCE BY CALENDAR YEAR ===")
display(yearly.style.format(FMT))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(yearly.index, yearly["ann_return"].fillna(0),
            color=np.where(yearly["ann_return"].fillna(0) >= 0, "#2e8b57", "#c0392b"))
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("ML L/S net return by calendar year")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(qs.index.astype(str), qs["mean"],
            yerr=(qs["std"] / np.sqrt(qs["count"])), capsize=4,
            color=["#c0392b"] + ["#95a5a6"] * (len(qs) - 2) + ["#2e8b57"])
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title(f"Average realised {TARGET_COL} by prediction quantile")
axes[1].set_xlabel("Q0 = lowest prediction")
plt.tight_layout(); plt.show()

ml_attr = factor_attribution(long_short_net, "ML L/S (net)")
rec("S7_ml", {"sharpe_gross": _j(gross_stats["sharpe"], 2),
              "ann_return_gross": _j(gross_stats["ann_return"]),
              "sharpe": _j(ls_stats["sharpe"], 2), "ann_return": _j(ls_stats["ann_return"]),
              "ann_vol": _j(ls_stats["ann_vol"]), "max_dd": _j(ls_stats["max_drawdown"]),
              "nw_t": _j(ls_stats["nw_tstat"], 2),
              "deflated": _j(ls_stats["deflated_sharpe_prob"], 3),
              "quantile_means": [_j(v, 5) for v in qs["mean"]],
              "alpha": (None if ml_attr is None else
                        {"ann": _j(ml_attr["ann_alpha"]), "t": _j(ml_attr["alpha_t"], 2),
                         "r2": _j(ml_attr["r2"], 3)})})

In [ ]:
# ============================================================
# PERFORMANCE DASHBOARD
# ============================================================
fig = plt.figure(figsize=(15, 11))
gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.22)

# 1. cumulative growth vs benchmarks
ax = fig.add_subplot(gs[0, :])
ax.plot(cum_ls_net.index, cum_ls_net.values, label="ML L/S (net, sector-neutral, vol-targeted)",
        linewidth=2.2, color="#1f4e79")
ax.plot(cum_long_only_net.index, cum_long_only_net.values, label="Long-only leg (net)",
        linewidth=1.6, color="#2e8b57", linestyle="--")
ax.plot(cum_universe.index, cum_universe.values, label="Equal-weight universe",
        linewidth=1.3, color="#888888", linestyle=":")
if "BENCH_FWD" in globals() and BENCH_FWD is not None:
    # forward-aligned so the curve covers the same windows as the strategies
    _b = BENCH_FWD.reindex(cum_ls_net.index).fillna(0.0)
    ax.plot(_b.index, (1 + _b).cumprod().values, label="SPY (forward-aligned)",
            linewidth=1.3, color="#e67e22", linestyle="-.")
ax.axhline(1.0, color="gray", linewidth=0.8, linestyle=":")
ax.set_title("Cumulative growth of $1 (development period)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.2f"))
ax.legend(loc="upper left", fontsize=8)

# 2. monthly IC
ax = fig.add_subplot(gs[1, 0])
ax.bar(ic_df.index, ic_df["ic"], width=20,
       color=np.where(ic_df["ic"].values >= 0, "#2e8b57", "#c0392b"), alpha=0.7)
ax.plot(ic_df.index, ic_df["ic"].rolling(6, min_periods=1).mean(),
        color="black", linewidth=2, label="6-month rolling mean")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title("Monthly Information Coefficient"); ax.legend(fontsize=8)

# 3. drawdown
ax = fig.add_subplot(gs[1, 1])
dd = cum_ls_net / cum_ls_net.cummax() - 1
ax.fill_between(dd.index, dd.values, 0, color="#c0392b", alpha=0.4)
ax.plot(dd.index, dd.values, color="#c0392b", linewidth=1.2)
ax.set_title("Drawdown (net of costs)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# 4. realised vol vs target
ax = fig.add_subplot(gs[2, 0])
ax.plot(BT["realized_vol"].index, BT["realized_vol"], color="#8c564b",
        label=f"Realised monthly vol ({CFG.vol_target_lookback_months}M trailing)")
if CFG.target_annual_volatility:
    ax.axhline(CFG.target_annual_volatility / np.sqrt(12), color="green", linestyle="--",
               label=f"Target ({CFG.target_annual_volatility:.0%} annualised)")
ax.set_title("Leak-free realised volatility vs target")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(fontsize=8)

# 5. leverage multiplier
ax = fig.add_subplot(gs[2, 1])
ax.plot(BT["scale"].index, BT["scale"], color="#d62728",
        label=f"Vol-target leverage (cap {CFG.max_vol_target_leverage:.1f}x)")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Volatility-targeting leverage"); ax.legend(fontsize=8)

plt.show()

In [ ]:
# ============================================================
# MECHANICS VALIDATION GATE
# ============================================================
unavailable = sorted(price_diagnostic.loc[
    price_diagnostic["yahoo_status"] == "unavailable_from_yahoo", "ticker"])

print("=" * 72); print("V13 MECHANICS GATE"); print("=" * 72)
print(f"PIT tickers requested        : {len(active_universe):,}")
print(f"Yahoo histories recovered    : {len(price_data):,}")
print(f"Yahoo unavailable/excluded   : {len(unavailable):,}")
print(f"Usable modeling rows (daily) : {len(panel):,}")
print(f"Month-end modeling rows      : {len(panel_me):,}")
print(f"Walk-forward folds (dev)     : {n_folds:,}")
print(f"Mean names per rebalance     : {BT['exposure'][['n_long','n_short']].sum(axis=1).mean():.1f}")
print(f"Mean one-way turnover        : {turnover_series.mean():.2%}")
print(f"Forward-target audit max err : {max_diff:.3e}")
print(f"Sector coverage              : {_frac:.1%}")

# this gate carried its OWN copy of the per-leg exposure check, so fixing
# build_and_backtest was not enough -- the duplicate here still demanded each
# leg stay under 1.0. With gross-preserving beta neutralisation the legs sit
# deliberately off their nominal targets (one scaled up, the other down by an
# offsetting amount), so the invariant that holds is TOTAL gross.
# It also failed as a single opaque boolean, which told one nothing about which
# of eight conditions broke. Named checks now report individually.
_TOL = 1e-6
_target_gross = BT.get("book_gross", 2.0)
_exp = BT["exposure"]
_valid = (_exp["n_long"] > 0) & (_exp["n_short"] > 0)

MECHANICS_CHECKS = {
    "price data recovered":        len(price_data) > 0,
    "month-end panel non-empty":   len(panel_me) > 0,
    "walk-forward folds produced": n_folds > 0,
    "one row per ticker-month":    panel_me.groupby(["ticker", "month"]).size().max() == 1,
    "forward-target audit clean":  max_diff <= 1e-8,
    "turnover is positive":        turnover_series.mean() > 0,
    "total gross within target":   bool((_exp.loc[_valid, "gross"] <= _target_gross + _TOL).all()),
    "no leg exceeds total gross":  bool((_exp.loc[_valid, "long_gross"] <= _target_gross + _TOL).all()
                                        and (_exp.loc[_valid, "short_gross"] <= _target_gross + _TOL).all()),
}
if not BT.get("beta_neutral_applied", False):
    # without beta neutralisation each leg should still respect its own cap
    MECHANICS_CHECKS["long leg within its own cap"] = bool(
        (_exp.loc[_valid, "long_gross"] <= BT.get("gross_long", 1.0) + _TOL).all())
    MECHANICS_CHECKS["short leg within its own cap"] = bool(
        (_exp.loc[_valid, "short_gross"] <= BT.get("gross_short", 1.0) + _TOL).all())

print("\n--- gate checks ---")
for _k, _v in MECHANICS_CHECKS.items():
    print(f"  [{'PASS' if _v else 'FAIL'}]  {_k}")

print(f"\n  book gross target {_target_gross:.2f} | realised mean "
      f"{_exp.loc[_valid, 'gross'].mean():.4f} (max {_exp.loc[_valid, 'gross'].max():.4f})")
print(f"  legs: long {_exp.loc[_valid, 'long_gross'].mean():.3f}, "
      f"short {_exp.loc[_valid, 'short_gross'].mean():.3f}"
      + ("  (deliberately unequal -- beta-neutral sizing)"
         if BT.get("beta_neutral_applied", False) else ""))

MECHANICS_READY = all(MECHANICS_CHECKS.values())
if not MECHANICS_READY:
    _failed = [k for k, v in MECHANICS_CHECKS.items() if not v]
    raise RuntimeError(
        "Mechanics gate FAILED on: " + "; ".join(_failed) +
        ". Do not interpret any result below until these are resolved.")
print("\nMECHANICS_READY = True")
rec("mechanics_ready", True)
rec("mechanics_checks", {k: bool(v) for k, v in MECHANICS_CHECKS.items()})

report = f"""
ML ALPHA SIGNAL GENERATOR an earlier implementation -- INTERIM REPORT (development period only)
=========================================================================
Target                 : {TARGET_COL}, {CFG.forward_return_days}-day horizon
Trained on             : {TRAIN_TARGET}
Features               : {len(FEATURE_COLS)} ({'with' if REVERSAL_FEATURE_COLS else 'without'} reversal)
Development window     : {results_df['date'].min().date()} -> {results_df['date'].max().date()}
Holdout (sealed)       : {CFG.holdout_start.date()} onward

SIGNAL QUALITY
  Mean IC (primary)    : {mean_ic:+.4f}   t = {ic_t:+.2f}
  Mean IC (Ridge)      : {ic_df['ic_linear'].mean():+.4f}
  Mean IC (ElasticNet) : {ic_df['ic_elastic'].mean():+.4f}
  Mean IC (naive)      : {ic_df['ic_naive'].mean():+.4f}  ({int(ic_df['ic_naive'].notna().sum())}/{len(ic_df)} months measurable)
  Positive IC months   : {pct_pos:.1%}

STRATEGY (net of realistic costs, sector-neutral, vol-targeted, ADV-capped)
  Annualised return    : {ls_stats['ann_return']:.2%}
  Annualised vol       : {ls_stats['ann_vol']:.2%}
  Sharpe               : {ls_stats['sharpe']:.2f}
  Max drawdown         : {ls_stats['max_drawdown']:.2%}
  Newey-West t         : {ls_stats['nw_tstat']:.2f}
  Deflated Sharpe P    : {ls_stats['deflated_sharpe_prob']:.1%} (assuming {CFG.n_trials_searched} trials)
  Mean turnover        : {turnover_series.mean():.2%} one-way per month
  Cost drag            : {transaction_cost_series.mean() * 12:.2%} annualised
=========================================================================
"""
print(report)
with open("v13_interim_report.txt", "w") as f:
    f.write(report)

## 8. Candidate Strategies

Four candidates are carried forward, deliberately spanning a range of
complexity so the machine-learning layer has something to be measured against.

**Machine-learning long/short** — the gradient-boosted model, sector-neutral.

**Theory composite** — a fixed rank-average of reversal, lottery and
low-volatility components, with each component's sign set from published
literature rather than fitted to this sample. No parameters.

**Beta-neutral composite** — the same signal with legs sized to offset beta.
This is the primary candidate.

**Time-series momentum** — a rule-based book with no cross-sectional ranking
at all, included as a structurally different control.

In [ ]:
# ============================================================
# TIME-SERIES MOMENTUM BOOK
# ============================================================
def build_trend_positions(df, signal_col):
    """Hold-until-sign-flip, evaluated once per month on month-end rows."""
    d = (df[["ticker", "month", "date", signal_col, "fwd_ret", "sector"]]
         .dropna(subset=[signal_col]).copy().sort_values(["ticker", "month"]))
    d["raw_side"] = np.sign(d[signal_col])
    d["side"] = d.groupby("ticker")["raw_side"].transform(
        lambda s: s.replace(0, np.nan).ffill().fillna(0))
    return d


def backtest_trend(pnl, signal_col, rebalance_every=1, label="", magnitude_weighted=False):
    """Equal weight (or |signal| weight) within each side, each month."""
    d = build_trend_positions(pnl, signal_col)
    if rebalance_every > 1:
        act = set(sorted(d["month"].unique())[::rebalance_every])
        d["side"] = d["side"].where(d["month"].isin(act))
        d["side"] = d.sort_values(["ticker", "month"]).groupby("ticker")["side"].ffill().fillna(0)
    d["long_flag"], d["short_flag"] = d["side"] > 0, d["side"] < 0

    if magnitude_weighted:
        mag = d[signal_col].abs()
        lsum = (mag * d["long_flag"]).groupby(d["month"]).transform("sum").replace(0, np.nan)
        ssum = (mag * d["short_flag"]).groupby(d["month"]).transform("sum").replace(0, np.nan)
        lw = np.where(d["long_flag"], mag / lsum, 0.0)
        sw = np.where(d["short_flag"], mag / ssum, 0.0)
    else:
        nl = d.groupby("month")["long_flag"].transform("sum").replace(0, np.nan)
        ns = d.groupby("month")["short_flag"].transform("sum").replace(0, np.nan)
        lw = np.where(d["long_flag"], 1.0 / nl, 0.0)
        sw = np.where(d["short_flag"], 1.0 / ns, 0.0)

    d["long_weight"], d["short_weight"] = np.nan_to_num(lw), np.nan_to_num(sw)
    d["signed_weight"] = d["long_weight"] - d["short_weight"]

    gross = d.assign(_c=d["signed_weight"] * d["fwd_ret"]).groupby("month")["_c"].sum()
    long_only = d.assign(_c=d["long_weight"] * d["fwd_ret"]).groupby("month")["_c"].sum()
    to = one_way_turnover(d, "signed_weight")
    net = gross - cost_of(to, CFG.transaction_cost_bps)

    st = performance_summary(net, label=label)
    st["turnover"] = to.mean()
    st["mean_net_exposure"] = d.groupby("month")["signed_weight"].sum().mean()
    return st, net, d, long_only, to


TREND_SIGNAL = "risk_adj_mom_21"
trend_stats, trend_net, trend_df, trend_long_only, trend_turnover = backtest_trend(
    panel_me_dev, TREND_SIGNAL, 1, "TS momentum (month-end, hold-until-flip)")
cum_trend_net = (1 + trend_net).cumprod()

print(f"Mean one-way turnover : {trend_turnover.mean():.2%}   "
      f"(an earlier implementation reported 5.32%, measured off daily rows)")
print(f"Mean net exposure     : {trend_stats['mean_net_exposure']:+.3f}   "
      f"(0 would be dollar-neutral; this book is directional)")
print(f"Mean net monthly ret  : {trend_net.mean():+.4%}\n")

display(pd.DataFrame([trend_stats, ls_stats]).set_index("label").style.format(FMT))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cum_trend_net.index, cum_trend_net.values, label="TS momentum (rebuilt)",
        linewidth=2.0, color="#2e8b57")
ax.plot(cum_ls_net.index, cum_ls_net.values, label="ML cross-sectional L/S",
        linewidth=1.6, color="#1f4e79", linestyle="--")
if spy_prices is not None and len(spy_prices):
    # forward-aligned, matching the strategies' fwd_ret convention
    _b = BENCH_FWD.reindex(cum_trend_net.index).fillna(0.0) if "BENCH_FWD" in globals() else None
    if _b is not None:
        ax.plot(_b.index, (1 + _b).cumprod().values, label="SPY (forward-aligned)",
                linewidth=1.3, color="#e67e22", linestyle="-.")
ax.axhline(1.0, color="gray", linewidth=0.8, linestyle=":")
ax.set_title("Rule-based trend following vs ML cross-sectional (development period)")
ax.legend(loc="upper left"); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# THEORY COMPOSITE SIGNAL
# ============================================================
print("=" * 70); print("the composite comparison  COMPOSITE BASELINE"); print("=" * 70)

# Signs fixed from theory, NOT from this sample's ICs.
COMPOSITE_SIGNS = {"rev_5": +1, "rev_21": +1, "rsi_14": -1, "max_ret_21": +1,
                   "prox_52w_high": +1, "ivol_21": -1}
available = {k: v for k, v in COMPOSITE_SIGNS.items() if k in results_df.columns}
print(f"  components used: {list(available)}")
if len(available) < 2:
    print("  too few components available -- skipping.")
else:
    globals()["comp"] = results_df.copy()
    comp = globals()["comp"]
    parts = []
    for c, sgn in available.items():
        parts.append(sgn * comp.groupby("month")[c].transform(
            lambda x: x.rank(pct=True) - 0.5))
    comp["pred"] = pd.concat(parts, axis=1).mean(axis=1)

    ic_comp = comp.groupby("month")[["pred", TARGET_COL]].apply(
        lambda g: safe_ic(g["pred"], g[TARGET_COL])).dropna()
    ic_ml = results_df.groupby("month")[["pred", TARGET_COL]].apply(
        lambda g: safe_ic(g["pred"], g[TARGET_COL])).dropna()

    bt_comp = build_and_backtest(comp, CFG, label="Composite", verbose=False)

    t10 = pd.DataFrame([
        {**full_report(long_short_net, "XGBoost L/S (net)"),
         "mean_ic": ic_ml.mean(), "turnover": turnover_series.mean()},
        {**full_report(bt_comp["net"], "Theory composite L/S (net)"),
         "mean_ic": ic_comp.mean(), "turnover": bt_comp["turnover"].mean()},
    ]).set_index("label")
    display(t10.style.format({**FMT, "nw_tstat": "{:.2f}",
                              "deflated_sharpe_prob": "{:.1%}", "mean_ic": "{:+.4f}"}))

    _t = (ic_comp.mean() / (ic_comp.std() / np.sqrt(len(ic_comp)))) if ic_comp.std() > 0 else np.nan
    print(f"\n  composite IC {ic_comp.mean():+.4f} (t = {_t:+.2f}) "
          f"vs XGBoost {ic_ml.mean():+.4f}")
    if ic_comp.mean() > ic_ml.mean():
        print("  -> five lines of arithmetic beat the model. The ML layer is "
              "subtracting value on this feature set.")
    else:
        print("  -> the model beats the naive composite.")

    rec("the composite comparison", {"composite_ic": _j(ic_comp.mean()), "composite_ic_t": _j(_t, 2),
                "ml_ic": _j(ic_ml.mean()),
                "composite_sharpe": _j(t10.loc["Theory composite L/S (net)", "sharpe"], 2),
                "ml_sharpe": _j(t10.loc["XGBoost L/S (net)", "sharpe"], 2),
                "composite_turnover": _j(bt_comp["turnover"].mean()),
                "components": list(available)})

In [ ]:
# ============================================================
# BETA-NEUTRAL CONSTRUCTION
# ============================================================
print("=" * 70); print("the beta-neutrality test  BETA NEUTRALISATION"); print("=" * 70)

_sigs19 = [("XGBoost", results_df)]
if "comp" in globals():
    _sigs19.append(("Composite", comp))

rows19 = []
for sig_lbl, sig_df in _sigs19:
    for bn_lbl, bn in [("dollar-neutral", False), ("beta-neutral", True)]:
        lbl = f"{sig_lbl} | {bn_lbl}"
        try:
            bt = build_and_backtest(sig_df, CFG, label=lbl, verbose=False,
                                    weighting_scheme="proportional", holding_months=3,
                                    prop_min_rank=0.0, prop_smoothing=1.0,
                                    beta_neutral=bn)
            r = performance_summary(bt["net"], label=lbl)
            r["cost_ann"] = bt["realistic_cost"].mean() * 12
            r["turnover"] = bt["turnover"].mean()
            r.update(decompose_return(bt["net"]))
            br = bt.get("beta_report") or {}
            r["ex_ante_beta_after"] = br.get("ex_ante_beta_after", np.nan)
            r["short_gross"] = bt["exposure"]["short_gross"].mean()
            rows19.append(r)
            print(f"  {lbl:28s} short gross {r['short_gross']:.2f} | "
                  f"realised beta {r.get('beta_spy', float('nan')):+.2f} | "
                  f"alpha {r.get('alpha_ann', float('nan')):+.2%} "
                  f"(t {r.get('alpha_t', float('nan')):+.2f}) | "
                  f"return {r['ann_return']:+.2%} | SR {r['sharpe']:+.2f}")
        except Exception as e:
            print(f"  FAILED {lbl}: {e}")

if rows19:
    t19 = pd.DataFrame(rows19).set_index("label")
    cols = ["short_gross", "ex_ante_beta_after", "beta_spy", "beta_contribution",
            "alpha_ann", "alpha_t", "realised_ann", "ann_vol", "sharpe",
            "max_drawdown", "cost_ann", "turnover"]
    display(t19[[c for c in cols if c in t19.columns]]
            .style.format({"short_gross": "{:.2f}", "ex_ante_beta_after": "{:+.3f}",
                           "beta_spy": "{:+.2f}", "beta_contribution": "{:+.2%}",
                           "alpha_ann": "{:+.2%}", "alpha_t": "{:+.2f}",
                           "realised_ann": "{:+.2%}", "ann_vol": "{:.2%}",
                           "sharpe": "{:+.2f}", "max_drawdown": "{:.2%}",
                           "cost_ann": "{:.2%}", "turnover": "{:.1%}"}))

    print("\n  Read `beta_spy` (REALISED) rather than `ex_ante_beta_after`. The")
    print("  construction targets zero using a trailing estimate; only the realised")
    print("  column says whether it actually worked.")
    print("  `beta_contribution` is beta x market -- the part of the return that is")
    print("  market exposure rather than stock selection.")

    for sig_lbl, _ in _sigs19:
        dn, bn = f"{sig_lbl} | dollar-neutral", f"{sig_lbl} | beta-neutral"
        if dn in t19.index and bn in t19.index:
            print(f"\n  {sig_lbl}: beta {t19.loc[dn, 'beta_spy']:+.2f} -> "
                  f"{t19.loc[bn, 'beta_spy']:+.2f} | "
                  f"return {t19.loc[dn, 'ann_return']:+.2%} -> {t19.loc[bn, 'ann_return']:+.2%} | "
                  f"alpha {t19.loc[dn, 'alpha_ann']:+.2%} -> {t19.loc[bn, 'alpha_ann']:+.2%}")

    bn_net = None
    try:
        bn_net = build_and_backtest(_sigs19[-1][1], CFG, verbose=False,
                                    weighting_scheme="proportional", holding_months=3,
                                    prop_min_rank=0.0, prop_smoothing=1.0,
                                    beta_neutral=True)["net"]
    except Exception:
        pass

    rec("the beta-neutrality test", {r["label"]: {"short_gross": _j(r["short_gross"], 2),
                             "realised_beta": _j(r.get("beta_spy"), 2),
                             "beta_contribution": _j(r.get("beta_contribution")),
                             "alpha_ann": _j(r.get("alpha_ann")),
                             "alpha_t": _j(r.get("alpha_t"), 2),
                             "ann_return": _j(r["ann_return"]),
                             "ann_vol": _j(r["ann_vol"]),
                             "sharpe": _j(r["sharpe"], 2),
                             "max_dd": _j(r["max_drawdown"]),
                             "cost": _j(r["cost_ann"])} for r in rows19})

In [ ]:
# ============================================================
# ============================================================
# LONG-TILTED AND LONG-ONLY BOOKS
# ============================================================
print("=" * 70); print("the long-tilt test  LONG-TILTED BOOKS"); print("=" * 70)
print("  benchmark is forward-aligned, tilted books are NOT vol-targeted,")
print("  and every row's beta is checked against its stated exposure.\n")

_sigs16 = [("XGBoost", results_df)]
if "comp" in globals():
    _sigs16.append(("Composite", comp))

EXPOSURES = [("long-only", 1.0, 0.0), ("130/30", 1.3, 0.3), ("market-neutral", 1.0, 1.0)]

rows16, invariant_ok = [], True
for sig_lbl, sig_df in _sigs16:
    for exp_lbl, gl, gs in EXPOSURES:
        lbl = f"{sig_lbl} | {exp_lbl}"
        try:
            bt = build_and_backtest(sig_df, CFG, label=lbl, verbose=False,
                                    weighting_scheme="proportional", holding_months=3,
                                    prop_min_rank=0.0, prop_smoothing=1.0,
                                    gross_long=gl, gross_short=gs)
            r = performance_summary(bt["net"], label=lbl)
            r["cost_ann"] = bt["realistic_cost"].mean() * 12
            r["turnover"] = bt["turnover"].mean()
            r["book_gross"] = bt["book_gross"]
            r["vol_targeted"] = bt["vol_targeted_applied"]
            r.update(beta_vs_bench(bt["net"]))
            ok = check_beta_invariant(r.get("beta_spy"), gl, gs, lbl)
            r["beta_ok"] = ok
            invariant_ok &= ok
            rows16.append(r)
            print(f"  {lbl:26s} gross {r['book_gross']:.1f} | cost {r['cost_ann']:5.2%} | "
                  f"return {r['ann_return']:+6.2%} | SR {r['sharpe']:+.2f} | "
                  f"beta {r.get('beta_spy', float('nan')):+.2f} | "
                  f"alpha {r.get('alpha_ann', float('nan')):+6.2%} "
                  f"(t {r.get('alpha_t', float('nan')):+.2f})")
        except Exception as e:
            print(f"  FAILED {lbl}: {e}")

if BENCH_FWD is not None:
    sp = performance_summary(BENCH_FWD, label="SPY buy-and-hold")
    sp.update({"cost_ann": 0.0, "turnover": 0.0, "book_gross": 1.0, "vol_targeted": False,
               "beta_spy": 1.0, "alpha_ann": 0.0, "alpha_t": np.nan, "beta_ok": True})
    rows16.append(sp)

if rows16:
    t16 = pd.DataFrame(rows16).set_index("label")
    cols = ["book_gross", "vol_targeted", "cost_ann", "turnover", "ann_return", "ann_vol",
            "sharpe", "max_drawdown", "beta_spy", "alpha_ann", "alpha_t", "beta_ok"]
    display(t16[[c for c in cols if c in t16.columns]]
            .style.format({"book_gross": "{:.1f}", "cost_ann": "{:.2%}", "turnover": "{:.1%}",
                           "ann_return": "{:.2%}", "ann_vol": "{:.2%}", "sharpe": "{:+.2f}",
                           "max_drawdown": "{:.2%}", "beta_spy": "{:+.2f}",
                           "alpha_ann": "{:+.2%}", "alpha_t": "{:+.2f}"}))

    print("\n  A tilted book inherits the market, so raw return and Sharpe flatter it.")
    print("  Only ALPHA and its t-statistic say whether the signal contributed --")
    print("  an index fund is free, and SPY is in the table for exactly that reason.")
    if not invariant_ok:
        print("\n  *** At least one beta invariant FAILED. Do not read the alphas above")
        print("      until that is resolved -- this is precisely how an earlier implementation's +5.84%")
        print("      long-only 'alpha' turned out to be a misaligned benchmark. ***")
    else:
        print("\n  All beta invariants passed: each book's measured market exposure")
        print("  matches its stated long-minus-short gross.")
        _t = t16.drop(index=[i for i in t16.index if i.startswith("SPY")], errors="ignore")
        if "alpha_ann" in _t.columns and _t["alpha_ann"].notna().any():
            b = _t["alpha_ann"].idxmax()
            print(f"  best alpha: {b}  {_t.loc[b, 'alpha_ann']:+.2%}/yr "
                  f"(t {_t.loc[b, 'alpha_t']:+.2f})  -- |t| > 2 is the bar")

    tilt_net = None
    try:
        tilt_net = build_and_backtest(
            _sigs16[-1][1], CFG, verbose=False, weighting_scheme="proportional",
            holding_months=3, prop_min_rank=0.0, prop_smoothing=1.0,
            gross_long=1.0, gross_short=0.0)["net"]
    except Exception:
        pass
    rec("the long-tilt test", {r["label"]: {"gross": _j(r["book_gross"], 1), "cost": _j(r["cost_ann"]),
                             "ann_return": _j(r["ann_return"]), "ann_vol": _j(r["ann_vol"]),
                             "sharpe": _j(r["sharpe"], 2), "max_dd": _j(r["max_drawdown"]),
                             "beta_spy": _j(r.get("beta_spy"), 2),
                             "alpha_ann": _j(r.get("alpha_ann")),
                             "alpha_t": _j(r.get("alpha_t"), 2),
                             "beta_invariant_ok": bool(r.get("beta_ok", True))}
                for r in rows16})
    rec("T16_all_beta_invariants_ok", bool(invariant_ok))

---
# Part III — Results

## 9. Capacity

Market impact scales with the square root of participation, so the same
strategy is genuinely cheaper run with less money. For a book whose gross edge
is around 3%/yr this is decisive rather than a footnote: costs run about
10.7%/yr at $500M against 1.3% at $25M.

Capacity is a property to state, not a flaw to hide. The honest form of the
result is "this works below roughly $X, at Sharpe Y".

In [ ]:
# ============================================================
# CAPACITY CURVE
# ============================================================
print("=" * 70); print("the capacity analysis  BETA-NEUTRAL CAPACITY"); print("=" * 70)

_sig20 = comp if "comp" in globals() else results_df
rows20 = []
for aum in [25e6, 50e6, 100e6, 250e6, 500e6, 1e9]:
    try:
        bt = build_and_backtest(_sig20, CFG, verbose=False,
                                weighting_scheme="proportional", holding_months=3,
                                prop_min_rank=0.0, prop_smoothing=1.0,
                                beta_neutral=True, aum=aum)
        r = performance_summary(bt["net"], label=f"${aum / 1e6:,.0f}M")
        r["aum_musd"] = aum / 1e6
        r["ann_gross"] = bt["gross"].mean() * 12
        r["impact_ann"] = bt["impact"].mean() * 12
        r["borrow_ann"] = bt["borrow"].mean() * 12
        r["cost_ann"] = bt["realistic_cost"].mean() * 12
        r.update({k: v for k, v in decompose_return(bt["net"]).items()
                  if k in ("beta_spy", "alpha_ann", "alpha_t")})
        rows20.append(r)
        print(f"  ${aum / 1e6:7,.0f}M  gross {r['ann_gross']:+.2%} | impact {r['impact_ann']:5.2%} | "
              f"borrow {r['borrow_ann']:5.2%} | net {r['ann_return']:+6.2%} | "
              f"SR {r['sharpe']:+.2f} | beta {r.get('beta_spy', float('nan')):+.2f}")
    except Exception as e:
        print(f"  FAILED ${aum / 1e6:.0f}M: {e}")

if rows20:
    t20 = pd.DataFrame(rows20).set_index("label")
    display(t20[["aum_musd", "ann_gross", "impact_ann", "borrow_ann", "cost_ann",
                 "ann_return", "ann_vol", "sharpe", "beta_spy", "alpha_ann", "alpha_t"]]
            .style.format({"aum_musd": "{:,.0f}", "ann_gross": "{:+.2%}",
                           "impact_ann": "{:.2%}", "borrow_ann": "{:.2%}",
                           "cost_ann": "{:.2%}", "ann_return": "{:+.2%}",
                           "ann_vol": "{:.2%}", "sharpe": "{:+.2f}",
                           "beta_spy": "{:+.2f}", "alpha_ann": "{:+.2%}",
                           "alpha_t": "{:+.2f}"}))

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t20["aum_musd"], t20["ann_return"], "o-", color="#1f4e79", label="net return")
    ax.plot(t20["aum_musd"], t20["ann_gross"], "s--", color="#2e8b57", label="gross return")
    ax.plot(t20["aum_musd"], -t20["cost_ann"], "^--", color="#c0392b", label="-cost")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xscale("log"); ax.set_xlabel("Assets under management ($M, log scale)")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title("Beta-neutral composite: capacity")
    ax.legend(); plt.tight_layout(); plt.show()

    pos = t20[t20["ann_return"] > 0]
    if len(pos):
        print(f"\n  net-positive at or below ${pos['aum_musd'].max():,.0f}M "
              f"(Sharpe there: {t20.loc[pos.index[-1], 'sharpe']:+.2f})")
        print("  That is the honest capacity statement. Note the alpha t-statistic")
        print("  in the same row -- capacity does not make a result significant.")
    else:
        print("\n  net-negative at every size tested, including $25M.")
    rec("the capacity analysis", {r["label"]: {"aum_musd": _j(r["aum_musd"], 0),
                             "ann_gross": _j(r["ann_gross"]), "cost": _j(r["cost_ann"]),
                             "impact": _j(r["impact_ann"]), "borrow": _j(r["borrow_ann"]),
                             "ann_return": _j(r["ann_return"]), "sharpe": _j(r["sharpe"], 2),
                             "beta": _j(r.get("beta_spy"), 2),
                             "alpha_ann": _j(r.get("alpha_ann")),
                             "alpha_t": _j(r.get("alpha_t"), 2)} for r in rows20})

## 10. Robustness

Three checks that a backtest can fail even when its headline number looks fine.

**Sub-period stability** splits the sample into blocks. A real edge shows up in
all of them; an artifact shows up in one. The configuration is fixed before
the split — nothing is tuned per period.

**Rolling alpha** dates any decay. A strategy whose edge is *stable but small*
is a capacity problem. One whose edge is *decaying* is a different thing: the
full-sample average overstates what it would earn going forward.

**Survivorship exposure** quantifies what the missing data is worth. Names
absent from the price source are disproportionately delisted or acquired
companies, and that bias falls on the short leg specifically.

In [ ]:
# ============================================================
# SUB-PERIOD STABILITY
# ============================================================
print("=" * 70); print("the sub-period test  SUB-PERIOD STABILITY"); print("=" * 70)

_sig21 = comp if "comp" in globals() else results_df
try:
    bt21 = build_and_backtest(_sig21, CFG, verbose=False, weighting_scheme="proportional",
                              holding_months=3, prop_min_rank=0.0, prop_smoothing=1.0,
                              beta_neutral=True)
    net21 = bt21["net"].dropna()
    yrs = pd.to_datetime(net21.index).year
    BLOCKS = [("2005-2009", 2005, 2009), ("2010-2014", 2010, 2014),
              ("2015-2019", 2015, 2019), ("2020-2023", 2020, 2023)]

    rows21 = []
    for lbl, y0, y1 in BLOCKS:
        seg = net21[(yrs >= y0) & (yrs <= y1)]
        if len(seg) < 18:
            print(f"  {lbl}: only {len(seg)} months -- skipped")
            continue
        r = performance_summary(seg, label=lbl)
        b = BENCH_FWD.reindex(seg.index).dropna() if BENCH_FWD is not None else None
        if b is not None and len(b) > 18:
            ix = seg.index.intersection(b.index)
            beta, icpt = (float(v) for v in np.polyfit(b.loc[ix].values, seg.loc[ix].values, 1))
            r["beta_spy"], r["alpha_ann"] = beta, icpt * 12
            r["market_ann"] = float(b.loc[ix].mean() * 12)
        rows21.append(r)
        print(f"  {lbl}  n={r['n_months']:3d} | return {r['ann_return']:+6.2%} | "
              f"SR {r['sharpe']:+.2f} | beta {r.get('beta_spy', float('nan')):+.2f} | "
              f"alpha {r.get('alpha_ann', float('nan')):+.2%} | "
              f"market {r.get('market_ann', float('nan')):+.2%}")

    if rows21:
        t21 = pd.DataFrame(rows21).set_index("label")
        display(t21[[c for c in ["n_months", "ann_return", "ann_vol", "sharpe",
                                 "max_drawdown", "beta_spy", "alpha_ann", "market_ann"]
                     if c in t21.columns]]
                .style.format({"n_months": "{:.0f}", "ann_return": "{:+.2%}",
                               "ann_vol": "{:.2%}", "sharpe": "{:+.2f}",
                               "max_drawdown": "{:.2%}", "beta_spy": "{:+.2f}",
                               "alpha_ann": "{:+.2%}", "market_ann": "{:+.2%}"}))
        pos = int((t21["ann_return"] > 0).sum())
        print(f"\n  positive in {pos}/{len(t21)} sub-periods")
        if "alpha_ann" in t21.columns:
            apos = int((t21["alpha_ann"] > 0).sum())
            print(f"  positive ALPHA in {apos}/{len(t21)} sub-periods")
            print(f"  alpha range: {t21['alpha_ann'].min():+.2%} to {t21['alpha_ann'].max():+.2%}")
        if pos == len(t21):
            print("\n  Consistent sign across every regime. That is the strongest thing")
            print("  this project has produced -- stability, not a high Sharpe.")
        else:
            print("\n  The sign flips across regimes. Check whether the positive block is")
            print("  2018-2023 -- that is the period every version was built on, and an")
            print("  edge that appears only there is an artifact of the search, not alpha.")
        rec("the sub-period test", {r["label"]: {"n": int(r["n_months"]), "ann_return": _j(r["ann_return"]),
                                 "sharpe": _j(r["sharpe"], 2),
                                 "beta": _j(r.get("beta_spy"), 2),
                                 "alpha": _j(r.get("alpha_ann")),
                                 "market": _j(r.get("market_ann"))} for r in rows21})
except Exception as e:
    print("  [skipped]", e)

In [ ]:
# ============================================================
# ROLLING ALPHA
# ============================================================
print("=" * 70); print("the rolling-alpha analysis  ROLLING ALPHA"); print("=" * 70)

_sig22 = comp if "comp" in globals() else results_df
try:
    bt22 = build_and_backtest(_sig22, CFG, verbose=False, weighting_scheme="proportional",
                              holding_months=3, prop_min_rank=0.0, prop_smoothing=1.0,
                              beta_neutral=True)
    net22 = bt22["net"].dropna()
    ix = net22.index.intersection(BENCH_FWD.index) if BENCH_FWD is not None else net22.index[:0]

    W = 36
    if len(ix) < W + 12:
        print(f"  only {len(ix)} overlapping months -- need {W + 12}. Skipped.")
    else:
        y, x = net22.loc[ix], BENCH_FWD.loc[ix]
        roll_a, roll_t, dates = [], [], []
        for i in range(W, len(ix) + 1):
            ys, xs = y.iloc[i - W:i].values, x.iloc[i - W:i].values
            beta, icpt = (float(v) for v in np.polyfit(xs, ys, 1))
            resid = ys - (beta * xs + icpt)
            se = resid.std(ddof=2) / np.sqrt(W)
            roll_a.append(icpt * 12)
            roll_t.append(icpt / se if se > 0 else np.nan)
            dates.append(ix[i - 1])
        ra = pd.Series(roll_a, index=dates)
        rt = pd.Series(roll_t, index=dates)

        fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                                 gridspec_kw={"height_ratios": [2, 1]})
        axes[0].plot(ra.index, ra.values, color="#1f4e79", linewidth=2)
        axes[0].axhline(0, color="black", linewidth=0.9)
        axes[0].fill_between(ra.index, 0, ra.values, where=(ra.values > 0),
                             color="#2e8b57", alpha=0.25)
        axes[0].fill_between(ra.index, 0, ra.values, where=(ra.values <= 0),
                             color="#c0392b", alpha=0.25)
        axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        axes[0].set_title(f"{W}-month rolling annualised alpha, beta-neutral composite\\n"
                          "the edge fades from left to right")
        axes[0].set_ylabel("Annualised alpha")

        axes[1].plot(rt.index, rt.values, color="#8e44ad", linewidth=1.5)
        for lv, st in [(2, "--"), (0, "-"), (-2, "--")]:
            axes[1].axhline(lv, color="black", linewidth=0.8, linestyle=st)
        axes[1].set_ylabel("alpha t-stat"); axes[1].set_xlabel("")
        axes[1].set_title("t-statistic on the same window (|t| > 2 dashed)")
        plt.tight_layout(); plt.show()

        first, last = ra.iloc[:12].mean(), ra.iloc[-12:].mean()
        print(f"  first year of windows : {first:+.2%} mean rolling alpha")
        print(f"  last year of windows  : {last:+.2%}")
        print(f"  change                : {last - first:+.2%}")
        print(f"  windows with alpha > 0: {(ra > 0).mean():.0%}")
        print(f"  windows with t > 2    : {(rt > 2).mean():.0%}")
        # trend test on the rolling series
        xx = np.arange(len(ra))
        slope = float(np.polyfit(xx, ra.values, 1)[0]) * 12
        print(f"  linear trend in alpha : {slope:+.2%} per year of elapsed time")
        if slope < 0 and last < first:
            print("\\n  Consistent decay. The full-sample average overstates what this")
            print("  signal would earn going forward -- it blends a period that worked")
            print("  with one that does not.")
        rec("the rolling-alpha analysis", {"first_year_alpha": _j(first), "last_year_alpha": _j(last),
                    "decay_per_year": _j(slope), "pct_windows_positive": _j((ra > 0).mean(), 3),
                    "pct_windows_significant": _j((rt > 2).mean(), 3),
                    "n_windows": int(len(ra))})
except Exception as e:
    print("  [skipped]", e)

In [ ]:
# ============================================================
# SURVIVORSHIP EXPOSURE
# ============================================================
print("=" * 70); print("the survivorship check  SURVIVORSHIP"); print("=" * 70)

miss = price_diagnostic[price_diagnostic["yahoo_status"] == "unavailable_from_yahoo"].copy()
frac = len(miss) / max(len(price_diagnostic), 1)
print(f"  excluded tickers : {len(miss)} / {len(price_diagnostic)} ({frac:.1%})")

hist_only = float((~miss["is_current"].fillna(False)).mean()) if len(miss) else 0.0
months_lost = float(((miss["membership_end"] - miss["membership_start"]).dt.days / 30.44).sum()) if len(miss) else 0.0
print(f"  of those, historical-only (already removed from the index): {hist_only:.0%}")
print(f"  constituent-months never observed: {months_lost:,.0f}")

if len(miss):
    by_year = miss.groupby(miss["membership_end"].dt.year).size().to_frame("n_excluded")
    print("\n  excluded names by the year their membership ended:")
    display(by_year)

mw = membership_windows.set_index("ticker")
early = []
for t, df in price_data.items():
    if t not in mw.index:
        continue
    end = pd.Timestamp(mw.loc[t, "membership_end"])
    if df.index.max() < end - pd.Timedelta(days=10):
        early.append({"ticker": t, "membership_end": end, "last_price": df.index.max(),
                      "days_short": (end - df.index.max()).days})
se = pd.DataFrame(early)
print(f"\n  names present but whose history ends >10 days early: {len(se)}")
if len(se):
    display(se.sort_values("days_short", ascending=False).head(10))
    print("  These exit at the last available price with no delisting return applied.")

rec("the survivorship check", {"excluded_frac": _j(frac, 3), "excluded_n": int(len(miss)),
           "historical_only_frac": _j(hist_only, 3),
           "constituent_months_lost": int(months_lost),
           "ends_early_n": int(len(se))})

## 11. Portfolio Contribution

A market-neutral book returning 2.6% looks unimpressive next to equities
returning 16%. That comparison is the wrong one — nobody holds a
market-neutral strategy *instead of* their equity allocation; they hold it
*alongside*, because it does something different.

The right question is whether adding it improves a portfolio. With near-zero
correlation, a modest Sharpe reduces portfolio volatility faster than it
reduces return, so the ratio improves even when the strategy alone looks
weak.

In [ ]:
# ============================================================
# PORTFOLIO CONTRIBUTION ANALYSIS
# ============================================================
print("=" * 70); print("the portfolio-contribution analysis  PORTFOLIO CONTRIBUTION"); print("=" * 70)

if BENCH_FWD is None:
    print("  no benchmark available -- skipped")
else:
    CANDS24 = []
    if "bn_net" in globals():
        CANDS24.append((bn_net, "Composite beta-neutral"))
    try:
        _s24 = comp if "comp" in globals() else results_df
        _bt24 = build_and_backtest(_s24, CFG, verbose=False, weighting_scheme="proportional",
                                   holding_months=3, prop_min_rank=0.0, prop_smoothing=1.0,
                                   beta_neutral=True, aum=CFG.headline_aum)
        CANDS24.append((_bt24["net"], f"Composite beta-neutral @ ${CFG.headline_aum/1e6:.0f}M"))
    except Exception as e:
        print(f"  (capacity-sized book unavailable: {e})")
    if "trend_net" in globals():
        CANDS24.append((trend_net, "Time-series momentum"))

    WEIGHTS = np.arange(0.0, 0.81, 0.05)
    rows24, curves = [], {}

    def _sharpe(x):
        sd = x.std(ddof=1)
        return float((x.mean() * 12) / (sd * np.sqrt(12))) if sd > 0 else np.nan

    for srs, name in CANDS24:
        srs = pd.Series(srs).dropna()
        ix = srs.index.intersection(BENCH_FWD.index)
        if len(ix) < 36:
            continue
        s_, b_ = srs.loc[ix], BENCH_FWD.loc[ix]
        corr = float(np.corrcoef(s_.values, b_.values)[0, 1])
        base = _sharpe(b_)
        # round the keys: np.arange yields values like 0.6000000000000001, so
        # any later lookup by a literal float would raise KeyError
        curve = {round(float(w), 4): _sharpe((1 - w) * b_ + w * s_) for w in WEIGHTS}
        curves[name] = curve
        w_best = max(curve, key=curve.get)
        blend = (1 - w_best) * b_ + w_best * s_

        # bootstrap the IMPROVEMENT, not the level -- a +0.09 gain whose
        # interval spans zero is not evidence of anything
        rng_ = np.random.default_rng(CFG.random_state)
        n = len(ix); blk = max(3, int(round(n ** (1 / 3)))); nb_ = int(np.ceil(n / blk))
        diffs = []
        for _ in range(1500):
            st = rng_.integers(0, n, nb_)
            idx = np.concatenate([np.arange(q, q + blk) % n for q in st])[:n]
            bb, ss = b_.values[idx], s_.values[idx]
            d0, d1 = _sharpe(pd.Series(bb)), _sharpe(pd.Series((1 - w_best) * bb + w_best * ss))
            if np.isfinite(d0) and np.isfinite(d1):
                diffs.append(d1 - d0)
        lo, hi = (float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))) \
            if diffs else (np.nan, np.nan)

        rows24.append({"strategy": name, "corr_to_spy": corr, "spy_sharpe": base,
                       "best_weight": w_best, "blend_sharpe": curve[w_best],
                       "improvement": curve[w_best] - base, "imp_lo": lo, "imp_hi": hi,
                       "blend_return": float(blend.mean() * 12),
                       "blend_vol": float(blend.std(ddof=1) * np.sqrt(12)),
                       "blend_maxdd": float((lambda c: (c / c.cummax() - 1).min())((1 + blend).cumprod())),
                       "significant": bool(np.isfinite(lo) and lo > 0)})
        print(f"  {name:38s} corr {corr:+.2f} | best weight {w_best:.0%} | "
              f"Sharpe {base:.2f} -> {curve[w_best]:.2f} ({curve[w_best]-base:+.2f})")

    if rows24:
        t24 = pd.DataFrame(rows24).set_index("strategy")
        display(t24.style.format({"corr_to_spy": "{:+.2f}", "spy_sharpe": "{:.2f}",
                                  "best_weight": "{:.0%}", "blend_sharpe": "{:.2f}",
                                  "improvement": "{:+.2f}", "imp_lo": "{:+.2f}",
                                  "imp_hi": "{:+.2f}", "blend_return": "{:+.2%}",
                                  "blend_vol": "{:.2%}", "blend_maxdd": "{:.1%}"}))

        fig, ax = plt.subplots(figsize=(10, 5))
        for name, curve in curves.items():
            ax.plot(list(curve.keys()), list(curve.values()), "o-", linewidth=1.8, label=name)
        ax.axhline(next(iter(curves.values()))[0.0], color="black", linestyle="--",
                   linewidth=1, label="SPY alone")
        ax.set_xlabel("weight allocated to the strategy")
        ax.set_ylabel("portfolio Sharpe")
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_title("Does adding the strategy improve an equity portfolio?\n"
                     "the correct test for a market-neutral book")
        ax.legend(fontsize=8); plt.tight_layout(); plt.show()

        bst = t24["improvement"].idxmax()
        r = t24.loc[bst]
        print(f"\n  best contributor: {bst}")
        print(f"    {r['best_weight']:.0%} allocation lifts portfolio Sharpe "
              f"{r['spy_sharpe']:.2f} -> {r['blend_sharpe']:.2f} ({r['improvement']:+.2f})")
        print(f"    blended: {r['blend_return']:+.2%}/yr at {r['blend_vol']:.2%} vol, "
              f"max drawdown {r['blend_maxdd']:.1%}")
        print(f"    95% CI on the improvement [{r['imp_lo']:+.2f}, {r['imp_hi']:+.2f}]  -> "
              f"{'EXCLUDES zero' if r['significant'] else 'contains zero'}")
        if not r["significant"]:
            print("\n    The improvement is positive but not statistically established.")
            print("    Honest phrasing: 'diversifying, with a point estimate of "
                  f"{r['improvement']:+.2f} Sharpe that this sample cannot confirm'.")
        rec("the portfolio-contribution analysis", {i: {"corr_to_spy": _j(v["corr_to_spy"], 2),
                        "best_weight": _j(v["best_weight"], 2),
                        "spy_sharpe": _j(v["spy_sharpe"], 2),
                        "blend_sharpe": _j(v["blend_sharpe"], 2),
                        "improvement": _j(v["improvement"], 2),
                        "improvement_ci": [_j(v["imp_lo"], 2), _j(v["imp_hi"], 2)],
                        "significant": bool(v["significant"]),
                        "blend_return": _j(v["blend_return"]),
                        "blend_maxdd": _j(v["blend_maxdd"])} for i, v in t24.iterrows()})

---
# Part IV — Validation

## 12. Statistical Significance

The central question for any backtest: could this be luck?

**Newey–West t-statistics** correct for the fact that consecutive months of a
strategy's returns are not independent. **Block-bootstrap confidence
intervals** resample in blocks to preserve that dependence — an interval
containing zero means the point estimate is not evidence, however good it
looks. And the **deflated Sharpe ratio** asks what survives after accounting
for how many variants were searched: on 104 months, a realised Sharpe of +0.83
scores 96.9% at one trial and 2.8% at thirty.

In [ ]:
# ============================================================
# SIGNIFICANCE TESTING AND FACTOR ATTRIBUTION
# ============================================================
# Defensive: a loop variable named `sm` anywhere upstream would shadow the
# statsmodels alias and break every t-statistic below. This cell produces the
# verdict for the whole notebook, so it re-binds its own dependencies rather
# than trusting global state.
if HAS_SM:
    import statsmodels.api as sm

print("=" * 70); print("the significance table  SIGNIFICANCE"); print("=" * 70)

CANDIDATES = [(long_short_net, "ML long/short (net)"),
              (long_only_net, "ML long-only leg"),
              (trend_net, "Time-series momentum")]
for _nm, _var in [("Composite, beta-neutral", "bn_net"),
                  ("Composite, long-tilted", "tilt_net")]:
    if globals().get(_var) is not None:
        CANDIDATES.append((globals()[_var], _nm))

_rows7 = []
for r, l in CANDIDATES:
    row = full_report(r, l)
    lo, hi = bootstrap_ci(r, "sharpe")
    row["sharpe_ci_lo"], row["sharpe_ci_hi"] = lo, hi
    row["ci_excludes_zero"] = bool(np.isfinite(lo) and lo > 0)
    _rows7.append(row)
sig_tbl = pd.DataFrame(_rows7).set_index("label")
display(sig_tbl.style.format({**FMT, "nw_tstat": "{:.2f}", "nw_pvalue": "{:.3f}",
                              "deflated_sharpe_prob": "{:.1%}",
                              "sharpe_ci_lo": "{:+.2f}", "sharpe_ci_hi": "{:+.2f}"}))
print("\n  sharpe_ci_* is a block-bootstrap 95% interval. If it contains zero, the")
print("  point estimate is not evidence of skill however good it looks.")
print(f"\n  deflated_sharpe_prob assumes {CFG.n_trials_searched} variants were searched.")
print("  Below ~95%, the Sharpe is not distinguishable from the best of that many")
print("  random draws. SPY is included as a reference, not a competitor -- it was")
print("  not selected from a search, so its deflated number is not comparable.")

attr = {}
for r, l in CANDIDATES[:10]:
    a = factor_attribution(r, l)
    if a:
        attr[l] = {"ann_alpha": _j(a["ann_alpha"]), "alpha_t": _j(a["alpha_t"], 2),
                   "r2": _j(a["r2"], 3),
                   "betas": {k: _j(v, 3) for k, v in a["betas"].items()}}

rec("the significance table", {"significance": {i: {"sharpe": _j(r["sharpe"], 2),
                                "sharpe_ci": [_j(r.get("sharpe_ci_lo"), 2), _j(r.get("sharpe_ci_hi"), 2)],
                                "ci_excludes_zero": bool(r.get("ci_excludes_zero", False)),
                                "ann_return": _j(r["ann_return"]),
                                "ann_vol": _j(r["ann_vol"]),
                                "max_dd": _j(r["max_drawdown"]),
                                "nw_t": _j(r["nw_tstat"], 2),
                                "deflated": _j(r["deflated_sharpe_prob"], 3),
                                "n_months": int(r["n_months"])}
                            for i, r in sig_tbl.iterrows()},
           "attribution": attr,
           "n_trials_assumed": CFG.n_trials_searched})

## 13. Final Scorecard

Every candidate in one table, with each column explained in plain language and
a verdict at the end. The two columns that matter most are the lower bound of
the Sharpe confidence interval, and alpha with its t-statistic — a book can
return 16% with zero alpha by simply holding the market.

In [ ]:
# ============================================================
# FINAL SCORECARD -- plain-language summary of every candidate
# ============================================================
print("=" * 78)
print(" " * 26 + "FINAL SCORECARD")
print("=" * 78)

# ---- assemble every strategy worth reporting --------------------------------
BOOKS = []
def _add(series, name, note=""):
    try:
        srs = pd.Series(series).dropna()
        if len(srs) >= 24:
            BOOKS.append((srs, name, note))
    except Exception:
        pass

_add(long_short_net, "ML long/short (net)", "the main machine-learning book")
_add(long_only_net, "ML long-only leg", "just the long side")
if "bn_net" in globals():
    _add(bn_net, "Composite, beta-neutral", "best market-neutral candidate")
if "tilt_net" in globals():
    _add(tilt_net, "Composite, long-tilted", "market exposure plus the signal")
if "trend_net" in globals():
    _add(trend_net, "Time-series momentum", "the simple rule-based book")
# the beta-neutral book at a realistic small-fund size. Capacity is a
# property of a strategy, not a flaw -- at $500M costs are 4.4%/yr and at $25M
# they are 1.3%, and that difference is most of the result. Reporting only the
# $500M number understates a strategy that is genuinely capacity-limited.
try:
    _sig_hd = comp if "comp" in globals() else results_df
    _bt_hd = build_and_backtest(_sig_hd, CFG, verbose=False,
                                weighting_scheme="proportional", holding_months=3,
                                prop_min_rank=0.0, prop_smoothing=1.0,
                                beta_neutral=True, aum=CFG.headline_aum)
    _add(_bt_hd["net"], f"Composite beta-neutral @ ${CFG.headline_aum/1e6:.0f}M",
         "same book, realistic fund size")
except Exception as _e:
    print(f"(capacity-sized row unavailable: {_e})")

# the single strongest feature, traded on its own. the feature IC comparison measured rev_5 at
# IC +0.0179 with t = 2.38 -- better than the six-component composite (+0.0117)
# and far better than the model (-0.0009). Worth carrying as a candidate,
# because a one-line signal that beats a gradient-boosted ensemble is a result
# in itself. Caveat: it is the best of ~19 features examined, so the
# Bonferroni-corrected bar is |t| ~2.9 and it does not clear that.
try:
    if "rev_5" in results_df.columns:
        _r5 = results_df.copy()
        _r5["pred"] = _r5.groupby("month")["rev_5"].transform(lambda x: x.rank(pct=True) - 0.5)
        _bt_r5 = build_and_backtest(_r5, CFG, verbose=False,
                                    weighting_scheme="proportional", holding_months=3,
                                    prop_min_rank=0.0, prop_smoothing=1.0,
                                    beta_neutral=True, aum=CFG.headline_aum)
        _add(_bt_r5["net"], f"rev_5 alone @ ${CFG.headline_aum/1e6:.0f}M",
             "single strongest feature, no model")
except Exception as _e:
    print(f"(single-feature book unavailable: {_e})")

if BENCH_FWD is not None:
    _add(BENCH_FWD, "SPY (buy and hold)", "the free alternative -- the bar to beat")

# every book must be measured over the SAME months. In an earlier implementation the
# strategies had 180 months (the walk-forward needs 48 for training, so they
# start in 2009) while SPY and the trend book had 228 starting in 2005. SPY's
# row therefore absorbed the 2008 crash and the strategies did not, which made
# "Composite long-tilted Sharpe 1.03 beats SPY 0.72" a comparison between two
# different decades. Over a matched window SPY itself runs near 0.9-1.0.
_common = None
for srs, _, _ in BOOKS:
    _common = srs.index if _common is None else _common.intersection(srs.index)
BOOKS = [(srs.loc[_common], n, note) for srs, n, note in BOOKS]
print(f"All books measured over the SAME {len(_common)} months: "
      f"{pd.to_datetime(min(_common)):%Y-%m} to {pd.to_datetime(max(_common)):%Y-%m}\n")

_bench = BENCH_FWD.reindex(_common).dropna() if BENCH_FWD is not None else None

rows = []
for srs, name, note in BOOKS:
    r = performance_summary(srs, label=name)
    r["note"] = note
    # excess return over the benchmark and its information ratio -- the
    # number that says whether the book beats simply owning the index, as opposed to
    # alpha, which says whether the book beats its own beta exposure.
    if _bench is not None and not name.startswith("SPY"):
        ix = srs.index.intersection(_bench.index)
        if len(ix) > 24:
            exc = srs.loc[ix] - _bench.loc[ix]
            te = exc.std(ddof=1) * np.sqrt(12)
            r["excess_vs_spy"] = float(exc.mean() * 12)
            r["info_ratio"] = float((exc.mean() * 12) / te) if te > 0 else np.nan
            # correlation matters as much as excess return. For a
            # market-neutral book "excess vs SPY" is close to meaningless -- it
            # only says a strategy taking no market risk earned less than
            # equities in a bull market. Low correlation is what makes such a
            # book worth holding, and the portfolio-contribution analysis prices that properly.
            r["corr_to_spy"] = float(np.corrcoef(srs.loc[ix].values, _bench.loc[ix].values)[0, 1])
    lo, hi = bootstrap_ci(srs, "sharpe")
    r["sharpe_lo"], r["sharpe_hi"] = lo, hi
    r["beats_zero"] = bool(np.isfinite(lo) and lo > 0)
    try:
        t, p = newey_west_tstat(srs)
        r["t_stat"], r["p_value"] = t, p
    except Exception:
        r["t_stat"], r["p_value"] = np.nan, np.nan
    r["deflated"] = deflated_sharpe(srs, CFG.n_trials_searched)
    r.update({k: v for k, v in decompose_return(srs).items()
              if k in ("beta_spy", "alpha_ann", "alpha_t")})
    rows.append(r)

score = pd.DataFrame(rows).set_index("label")

# ---- the headline table -----------------------------------------------------
show = ["ann_return", "ann_vol", "sharpe", "sharpe_lo", "sharpe_hi",
        "max_drawdown", "calmar", "hit_rate", "beta_spy", "alpha_ann",
        "alpha_t", "excess_vs_spy", "info_ratio", "corr_to_spy", "deflated", "n_months"]
print("\n--- HEADLINE NUMBERS ---\n")
display(score[[c for c in show if c in score.columns]].style.format({
    "ann_return": "{:+.2%}", "ann_vol": "{:.2%}", "sharpe": "{:+.2f}",
    "sharpe_lo": "{:+.2f}", "sharpe_hi": "{:+.2f}", "max_drawdown": "{:.1%}",
    "calmar": "{:+.2f}", "hit_rate": "{:.0%}", "beta_spy": "{:+.2f}",
    "alpha_ann": "{:+.2%}", "alpha_t": "{:+.2f}", "excess_vs_spy": "{:+.2%}",
    "info_ratio": "{:+.2f}", "corr_to_spy": "{:+.2f}", "deflated": "{:.1%}",
    "n_months": "{:.0f}"}))

# ---- what each number means -------------------------------------------------
print("""
--- WHAT EACH COLUMN MEANS ---

ann_return    Annualised return, after all trading costs. What $100 grows by
              per year on average.
ann_vol       Annualised volatility -- how bumpy the ride is. Equities are
              ~16-20%. Half of that means half the swings.
sharpe        Return per unit of risk (return / volatility). THE headline
              number in this business.
                < 0     losing money
                0-0.5   weak
                0.5-1.0 decent
                1.0-2.0 good
                > 2.0   rare, and usually a bug
sharpe_lo/hi  95% confidence interval on that Sharpe, from bootstrapping. If
              sharpe_lo is BELOW zero, one cannot rule out that the true
              Sharpe is zero and one got lucky. This is the column most people
              never compute and it is the one that matters most.
max_drawdown  Worst peak-to-trough fall. -20% means at some point one were
              down a fifth from the high. Investors quit well before -50%.
calmar        Return divided by max drawdown. Return per unit of pain.
hit_rate      Share of months that made money. Above 50% is nice but shallow --
              one can win 70% of months and still lose overall.
beta_spy      Market exposure. 1.0 moves with the market one-for-one; 0.0 is
              market-neutral. A "market-neutral" book with beta 0.5 is really
              half an index fund.
alpha_ann     Return left over AFTER removing market exposure. This is the
              part the strategy actually added. A book can return 12% with
              zero alpha by simply holding the market.
alpha_t       t-statistic on that alpha. |t| > 2 is the usual bar for "probably
              not luck." Below that, the alpha is not distinguishable from zero.
excess_vs_spy Return minus SPY's return over the SAME months. Alpha asks "did
              one beat its own beta exposure"; this asks the blunter question
              "did one beat just buying the index". A book with beta 1.0 can
              have zero alpha and still show excess return, or vice versa.
info_ratio    Excess return divided by how much it wobbles around the index
              (tracking error). The Sharpe ratio of the outperformance.
              CAUTION: both of the columns above are only meaningful for a book
              with beta near 1.0, which one would hold INSTEAD of the index. For
              a market-neutral book (beta near 0) they merely restate that it
              takes no market risk. Use the portfolio-contribution analysis for those.
corr_to_spy   Correlation with the market. Near zero is what makes a
              low-returning strategy worth owning -- it diversifies. the portfolio-contribution analysis turns
              that into a number.
deflated      Probability the true Sharpe beats zero AFTER accounting for how
              many strategy variants were tried ({} here). Below 95%, the
              result is not distinguishable from the best of that many random
              draws. This is the honest scoreboard.
n_months      How many months the result is based on. More is better; the
              confidence interval shrinks roughly with its square root.
""".format(CFG.n_trials_searched))

# ---- the verdict ------------------------------------------------------------
print("--- VERDICT ---\n")
strat = score.drop(index=[i for i in score.index if i.startswith("SPY")], errors="ignore")
spy_row = score.loc["SPY (buy and hold)"] if "SPY (buy and hold)" in score.index else None

if len(strat):
    # rank on ALPHA, not Sharpe. an earlier implementation's top Sharpe was the long-tilted book
    # at 1.03 -- with beta 0.98 and alpha of +0.06% (t=0.05). That is the market
    # wearing a strategy's name. Sharpe rewards taking market risk; alpha is
    # what the signal added.
    _key = "alpha_ann" if strat["alpha_ann"].notna().any() else "sharpe"
    best = strat[_key].idxmax()
    if _key == "alpha_ann":
        print("(ranked by ALPHA -- a high Sharpe from beta near 1.0 is just the index)\n")
    b = strat.loc[best]
    print(f"Best strategy by Sharpe : {best}")
    print(f"  return {b['ann_return']:+.2%}/yr | volatility {b['ann_vol']:.2%} | "
          f"Sharpe {b['sharpe']:+.2f}")
    print(f"  max drawdown {b['max_drawdown']:.1%} | "
          f"{int(b['n_months'])} months of evidence")
    if np.isfinite(b.get("alpha_ann", np.nan)):
        print(f"  market beta {b.get('beta_spy', float('nan')):+.2f} | "
              f"alpha {b['alpha_ann']:+.2%}/yr (t {b.get('alpha_t', float('nan')):+.2f})")
    print(f"  Sharpe 95% CI [{b['sharpe_lo']:+.2f}, {b['sharpe_hi']:+.2f}]  -> "
          f"{'EXCLUDES zero' if b['beats_zero'] else 'CONTAINS zero'}")
    print(f"  deflated Sharpe probability {b['deflated']:.1%}")

    if spy_row is not None:
        gap = b["ann_return"] - spy_row["ann_return"]
        print(f"\nAgainst just buying SPY  : SPY returned {spy_row['ann_return']:+.2%}/yr "
              f"at Sharpe {spy_row['sharpe']:+.2f}")
        print(f"  the strategy is {gap:+.2%}/yr {'ahead' if gap > 0 else 'behind'} on raw return")

    print()
    if b["beats_zero"] and np.isfinite(b.get("deflated", np.nan)) and b["deflated"] > 0.95:
        v = ("TRADEABLE-LOOKING. The confidence interval excludes zero and the result "
             "survives the multiple-testing adjustment. Confirm on the holdout before "
             "believing it.")
    elif b["beats_zero"]:
        v = ("PROMISING BUT UNCONFIRMED. The raw interval excludes zero, but the "
             "deflated Sharpe says it is not distinguishable from the best of "
             f"{CFG.n_trials_searched} searched variants. More data, not more searching.")
    elif b["sharpe"] > 0:
        v = ("POSITIVE BUT NOT SIGNIFICANT. The strategy made money, and with this much "
             "data that is not distinguishable from luck. Honest description: a small "
             "possible edge, unproven.")
    else:
        v = ("NOT WORKING. The best candidate loses money after costs. The measurement "
             "framework is sound -- the signal is not there.")
    print(f"BOTTOM LINE: {v}")

    rec("SCORECARD", {i: {"ann_return": _j(r["ann_return"]), "ann_vol": _j(r["ann_vol"]),
                          "sharpe": _j(r["sharpe"], 2),
                          "sharpe_ci": [_j(r["sharpe_lo"], 2), _j(r["sharpe_hi"], 2)],
                          "max_drawdown": _j(r["max_drawdown"]),
                          "hit_rate": _j(r["hit_rate"], 3),
                          "beta": _j(r.get("beta_spy"), 2),
                          "alpha_ann": _j(r.get("alpha_ann")),
                          "alpha_t": _j(r.get("alpha_t"), 2),
                          "excess_vs_spy": _j(r.get("excess_vs_spy")),
                          "info_ratio": _j(r.get("info_ratio"), 2),
                          "deflated": _j(r.get("deflated"), 3),
                          "n_months": int(r["n_months"])} for i, r in score.iterrows()})
    rec("SCORECARD_verdict", v)

# ---- one picture ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for srs, name, _ in BOOKS:
    axes[0].plot(srs.index, (1 + srs).cumprod().values, label=name, linewidth=1.8)
axes[0].axhline(1.0, color="gray", linestyle=":", linewidth=0.8)
axes[0].set_title("Growth of $1, net of costs")
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.2f"))
axes[0].legend(fontsize=8)

nm = [n for _, n, _ in BOOKS]
sh = [score.loc[n, "sharpe"] for n in nm]
lo = [score.loc[n, "sharpe"] - score.loc[n, "sharpe_lo"] for n in nm]
hi = [score.loc[n, "sharpe_hi"] - score.loc[n, "sharpe"] for n in nm]
axes[1].barh(nm, sh, xerr=[lo, hi], capsize=4,
             color=["#2e8b57" if v > 0 else "#c0392b" for v in sh])
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Sharpe with 95% confidence intervals\nbars crossing zero are not proven")
plt.tight_layout(); plt.show()

## 14. Out-of-Sample Holdout

A block of months was sealed before any tuning began and never examined. It is
a single-use resource: looking, adjusting, and looking again converts it into
ordinary in-sample data.

A development Sharpe that degrades moderately out of sample is normal. One
that flips sign was fitted to noise.

In [ ]:
CFG.reveal_holdout = True

In [ ]:
# ============================================================
# HOLDOUT EVALUATION
# ============================================================
if not CFG.reveal_holdout:
    print("Holdout is SEALED (CFG.reveal_holdout = False).")
    print(f"Sealed period: {CFG.holdout_start.date()} onward, "
          f"{panel_me[panel_me['date'] >= CFG.holdout_start]['month'].nunique()} months, "
          f"{results_full[results_full['date'] >= CFG.holdout_start]['month'].nunique()} "
          "walk-forward folds already predicted and set aside.")
    print("\nSet CFG.reveal_holdout = True and re-run this cell once tuning is complete.")
else:
    print("=" * 70); print("HOLDOUT REVEAL -- THIS IS A SINGLE-USE RESULT"); print("=" * 70)

    res_ho = results_full[results_full["date"] >= CFG.holdout_start].copy().reset_index(drop=True)
    pme_ho = panel_me[panel_me["date"] >= CFG.holdout_start].copy()

    if len(res_ho) == 0 or res_ho["month"].nunique() < 6:
        print("Not enough holdout months to evaluate. Move CFG.holdout_start earlier "
              "and re-run the whole notebook (before looking at any result).")
    else:

        BT_ho = build_and_backtest(res_ho, CFG, label="ML L/S (HOLDOUT)")
        ho_rows = [full_report(BT_ho["net"], "ML L/S (holdout)")]

        ts_ho, ts_net_ho, *_ = backtest_trend(pme_ho, TREND_SIGNAL, 1, "TS momentum (holdout)")
        ho_rows.append(full_report(ts_net_ho, "TS momentum (holdout)"))

        # ---- beta-neutral composite on the holdout ------------------------
        # The composite has NO fitted parameters: it is a fixed rank-average of
        # feature columns whose signs come from published literature. So it can
        # be evaluated directly on holdout rows without refitting anything, and
        # there is no leakage channel. Note what this does and does not test:
        # with nothing fitted there is no overfitting to detect, so this is a
        # test of whether the underlying EFFECT persists into a later period,
        # not of whether a model generalises.
        bn_ho_net = None
        try:
            comp_ho_df = res_ho.copy()
            parts_ho = []
            for _c, _sgn in COMPOSITE_SIGNS.items():
                if _c in comp_ho_df.columns:
                    parts_ho.append(_sgn * comp_ho_df.groupby("month")[_c].transform(
                        lambda x: x.rank(pct=True) - 0.5))
            if len(parts_ho) >= 3:
                comp_ho_df["pred"] = pd.concat(parts_ho, axis=1).mean(axis=1)
                BT_bn_ho = build_and_backtest(
                    comp_ho_df, CFG, label="Composite beta-neutral (HOLDOUT)", verbose=False,
                    weighting_scheme="proportional", holding_months=3,
                    prop_min_rank=0.0, prop_smoothing=1.0,
                    beta_neutral=True, aum=CFG.headline_aum)
                bn_ho_net = BT_bn_ho["net"]
                ho_rows.append(full_report(bn_ho_net, "Composite beta-neutral (holdout)"))

                # The development leg MUST be rebuilt at the same fund size.
                # `bn_net` is built at CFG.assumed_aum_usd while the holdout leg
                # above uses CFG.headline_aum; costs differ by roughly 10%/yr
                # between those, so comparing them directly would compare two
                # different strategies rather than two periods of one.
                BT_bn_dev = build_and_backtest(
                    comp, CFG, verbose=False, weighting_scheme="proportional",
                    holding_months=3, prop_min_rank=0.0, prop_smoothing=1.0,
                    beta_neutral=True, aum=CFG.headline_aum)
                bn_dev_net = BT_bn_dev["net"]

                # IC of the composite over the holdout months
                _ic_bn_ho = comp_ho_df.groupby("month")[["pred", TARGET_COL]].apply(
                    lambda g: safe_ic(g["pred"], g[TARGET_COL])).dropna()
                _bn_ho_stats = performance_summary(bn_ho_net, label="bn_ho")
                # BENCH_FWD covers development months only, so alpha/beta on the
                # holdout must be priced against the full-sample benchmark.
                _bn_ho_dec = decompose_return(bn_ho_net, ) if BENCH_FWD_ALL is None else \
                    beta_vs_bench(bn_ho_net, BENCH_FWD_ALL)
                if _bn_ho_dec and BENCH_FWD_ALL is not None:
                    _ix_ho = pd.Series(bn_ho_net).dropna().index.intersection(BENCH_FWD_ALL.index)
                    _bn_ho_dec["market_ann"] = float(BENCH_FWD_ALL.loc[_ix_ho].mean() * 12)
                print(f"\n  Composite beta-neutral on holdout: "
                      f"{_bn_ho_stats['ann_return']:+.2%}/yr, "
                      f"Sharpe {_bn_ho_stats['sharpe']:+.2f}, "
                      f"beta {_bn_ho_dec.get('beta_spy', float('nan')):+.2f}, "
                      f"alpha {_bn_ho_dec.get('alpha_ann', float('nan')):+.2%} "
                      f"(t {_bn_ho_dec.get('alpha_t', float('nan')):+.2f}), "
                      f"IC {_ic_bn_ho.mean():+.4f}")
            else:
                print("\n  (composite components unavailable on holdout rows)")
        except Exception as _e:
            print(f"\n  (composite holdout unavailable: {_e})")

        ic_ho = ic_df_full[ic_df_full.index >= CFG.holdout_start]["ic"].dropna()

        print("\n=== DEVELOPMENT VS HOLDOUT ===")
        rows_ho = [
            {"period": "development", "strategy": "ML L/S",
             "sharpe": ls_stats["sharpe"], "ann_return": ls_stats["ann_return"],
             "mean_ic": mean_ic, "n_months": ls_stats["n_months"]},
            {"period": "HOLDOUT", "strategy": "ML L/S",
             "sharpe": ho_rows[0]["sharpe"], "ann_return": ho_rows[0]["ann_return"],
             "mean_ic": ic_ho.mean(), "n_months": ho_rows[0]["n_months"]},
            {"period": "development", "strategy": "TS momentum",
             "sharpe": trend_stats["sharpe"], "ann_return": trend_stats["ann_return"],
             "mean_ic": np.nan, "n_months": trend_stats["n_months"]},
            {"period": "HOLDOUT", "strategy": "TS momentum",
             "sharpe": ho_rows[1]["sharpe"], "ann_return": ho_rows[1]["ann_return"],
             "mean_ic": np.nan, "n_months": ho_rows[1]["n_months"]},
        ]
        if bn_ho_net is not None:
            _dev = performance_summary(bn_dev_net, label="dev")
            _dev_ic = comp.groupby("month")[["pred", TARGET_COL]].apply(
                lambda g: safe_ic(g["pred"], g[TARGET_COL])).dropna().mean()
            rows_ho.extend([
                {"period": "development", "strategy": "Composite beta-neutral",
                 "sharpe": _dev["sharpe"], "ann_return": _dev["ann_return"],
                 "mean_ic": _dev_ic, "n_months": _dev["n_months"]},
                {"period": "HOLDOUT", "strategy": "Composite beta-neutral",
                 "sharpe": _bn_ho_stats["sharpe"], "ann_return": _bn_ho_stats["ann_return"],
                 "mean_ic": _ic_bn_ho.mean(), "n_months": _bn_ho_stats["n_months"]},
            ])
        comp_ho = pd.DataFrame(rows_ho).set_index(["strategy", "period"])
        display(comp_ho.style.format({"sharpe": "{:+.2f}", "ann_return": "{:.2%}",
                                      "mean_ic": "{:+.4f}"}))

        print("\n  A development Sharpe that degrades moderately is normal.")
        print("  One that flips sign means the result was fit to noise.")

        rec("S14_holdout", {
            "ml_dev_sharpe": _j(ls_stats["sharpe"], 2),
            "ml_holdout_sharpe": _j(ho_rows[0]["sharpe"], 2),
            "ml_dev_ic": _j(mean_ic), "ml_holdout_ic": _j(ic_ho.mean()),
            "trend_dev_sharpe": _j(trend_stats["sharpe"], 2),
            "trend_holdout_sharpe": _j(ho_rows[1]["sharpe"], 2),
            "composite_dev_sharpe": (_j(_dev["sharpe"], 2) if bn_ho_net is not None else None),
            "composite_holdout_sharpe": (_j(_bn_ho_stats["sharpe"], 2) if bn_ho_net is not None else None),
            "composite_holdout_return": (_j(_bn_ho_stats["ann_return"]) if bn_ho_net is not None else None),
            "composite_holdout_alpha": (_j(_bn_ho_dec.get("alpha_ann")) if bn_ho_net is not None else None),
            "composite_holdout_ic": (_j(_ic_bn_ho.mean()) if bn_ho_net is not None else None),
            "holdout_months": int(ho_rows[0]["n_months"])})

        print("\n<<<<< V13 HOLDOUT DIGEST >>>>>")
        print(json.dumps(REPORT.get("S14_holdout", {}), indent=1, default=str))

## 15. Generated Research Summary

A one-page summary written from this run's actual numbers, saved alongside the
notebook. Nothing is hand-typed, so nothing can drift out of date.

In [ ]:
# ============================================================
# RESEARCH SUMMARY -- generated from this run's actual numbers
# ============================================================
def _g(path, default=None):
    cur = REPORT
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


bn = _g(["the beta-neutrality test", "Composite | beta-neutral"], {})
cap = _g(["the capacity analysis", "$25M"], {})
t21 = _g(["the sub-period test"], {})
t22 = _g(["the rolling-alpha analysis"], {})
t8 = _g(["the survivorship check"], {})
ho = _g(["S14_holdout"], {})
sig = _g(["the significance table", "significance"], {})

lines = []
A = lines.append
A("# Cross-Sectional Equity Alpha Research — Summary")
A("")
A(f"*Generated {datetime.today():%Y-%m-%d} from a single notebook run.*")
A("")
A("## Objective")
A("")
A("Test whether a cross-sectional signal built from publicly available daily")
A("price data can generate risk-adjusted returns in large-cap US equities, net")
A("of realistic trading costs.")
A("")
A("## Methodology")
A("")
A(f"- **Universe:** point-in-time S&P 500 membership, {_g(['panel','tickers'],'n/a')} tickers, "
  f"{CFG.start_date} to {CFG.end_date}")
A(f"- **Validation:** walk-forward with a {CFG.forward_return_days}-session embargo; "
  f"{_g(['panel','dev_months'],'n/a')} development months, "
  f"{_g(['panel','holdout_months'],'n/a')} months held out and sealed before tuning")
A("- **Costs:** square-root market impact scaled by participation in each name's")
A(f"  ADV, {CFG.transaction_cost_bps:.0f}bp spread, "
  f"{CFG.short_borrow_cost_bps_annual:.0f}bp annual borrow charged as a holding cost")
A(f"- **Capacity:** positions capped at {CFG.max_position_pct_adv:.0%} of each name's ADV")
A("- **Construction:** sector-neutral, rank-proportional weights across the full")
A("  cross-section, beta-neutral leg sizing")
A(f"- **Multiple testing:** deflated Sharpe ratio accounting for "
  f"{CFG.n_trials_searched} searched variants; block-bootstrap confidence intervals")
A("")
d4 = _g(["the rotation diagnostic"], {})
t23b = _g(["T23_best"], None)
if d4:
    A("## Model staleness")
    A("")
    A(f"Rolling 24-month coefficient vectors have a cosine similarity of "
      f"{d4.get('cosine_12m', 0):+.3f} at a 12-month lag"
      + (f", implying a directional half-life of about {d4.get('half_life_months')} months."
         if d4.get("half_life_months") else "."))
    A("")
    A(f"> {d4.get('verdict', '')}")
    A("")
    if t23b:
        A(f"Best training scheme in the pre-registered sweep: **{t23b}**.")
        A("")
A("## Principal finding — alpha decay")
A("")
if t21:
    A("| Period | Alpha | Sharpe | Market |")
    A("|---|---:|---:|---:|")
    for k, v in t21.items():
        A(f"| {k} | {v.get('alpha', 0):+.2%} | {v.get('sharpe', 0):+.2f} | "
          f"{v.get('market', 0):+.2%} |")
    A("")
if t22:
    A(f"A 36-month rolling alpha falls from {t22.get('first_year_alpha', 0):+.2%} to "
      f"{t22.get('last_year_alpha', 0):+.2%}, a trend of "
      f"{t22.get('decay_per_year', 0):+.2%} per year. Only "
      f"{t22.get('pct_windows_significant', 0):.0%} of rolling windows reach |t| > 2.")
    A("")
A("The edge is strongest in the earliest data and absent in the most recent —")
A("the opposite of what in-sample overfitting produces, and consistent with a")
A("real effect being arbitraged away. Short-term reversal and the")
A("low-volatility anomaly have both been widely documented and cheaply")
A("tradeable since roughly 2010.")
A("")
A("## Headline results")
A("")
if bn:
    A(f"- Beta-neutral composite, full sample: **{bn.get('ann_return', 0):+.2%}/yr**, "
      f"Sharpe **{bn.get('sharpe', 0):+.2f}**, realised beta {bn.get('realised_beta', 0):+.2f}, "
      f"max drawdown {bn.get('max_dd', 0):.1%}")
    A(f"- Alpha {bn.get('alpha_ann', 0):+.2%} (t = {bn.get('alpha_t', 0):+.2f})")
if cap:
    A(f"- At $25M capacity: **{cap.get('ann_return', 0):+.2%}/yr**, Sharpe "
      f"**{cap.get('sharpe', 0):+.2f}**, alpha {cap.get('alpha_ann', 0):+.2%} "
      f"(t = {cap.get('alpha_t', 0):+.2f})")
A("")
A("No configuration produced a bootstrap confidence interval on Sharpe that")
A("excludes zero, other than the long-only leg — whose beta is approximately")
A("1.0, making it market exposure rather than alpha.")
A("")
if ho:
    A("## Out-of-sample holdout")
    A("")
    A(f"- Development Sharpe {ho.get('ml_dev_sharpe', 'n/a')} -> "
      f"holdout {ho.get('ml_holdout_sharpe', 'n/a')} ({ho.get('holdout_months', 'n/a')} months)")
    A(f"- Development IC {ho.get('ml_dev_ic', 'n/a')} -> holdout {ho.get('ml_holdout_ic', 'n/a')}")
    A("")
    A("The holdout period was fixed before any tuning and evaluated once.")
    A("")
A("## Measurement biases identified and corrected")
A("")
for i, t in enumerate([
    "Overlapping daily observations treated as monthly portfolio returns "
    "(inflated Sharpe 1.8x, understated turnover)",
    "Z-scored features used as raw inputs to a ratio, flipping the signal's "
    "sign on ~50% of rows",
    "Cross-sectional mean of a z-score used as a market-regime input "
    "(standard deviation 0.000000)",
    "IC function converting undefined correlations to 0.0, hiding five of nine "
    "years of unmeasured baseline",
    "Sector taxonomy mismatch splitting each sector across two labels, halving "
    "every peer group",
    "Benchmark misaligned by one period against forward returns, manufacturing "
    "+5.84%/yr of phantom alpha",
    "Dollar-neutral construction carrying -0.21 of uncompensated market beta, "
    "costing ~3%/yr and masking a positive alpha",
], 1):
    A(f"{i}. {t}")
A("")
A("## Limitations")
A("")
A(f"- **Survivorship:** {t8.get('excluded_frac', 0):.1%} of point-in-time constituents "
  f"({t8.get('excluded_n', 0)} tickers, {t8.get('constituent_months_lost', 0):,} "
  "constituent-months) had no price history available. These are disproportionately")
A("  delisted or acquired companies, which biases the short leg specifically.")
A("- **Fundamentals unavailable:** the free data source reaches back roughly five")
A("  quarters, so valuation and quality features were dropped on a coverage gate.")
A("- **Statistical power:** at these effect sizes, distinguishing the observed")
A("  Sharpe from zero would require several times the available history.")
A("")
A("## Conclusion")
A("")
A("Publicly available daily OHLCV data on large-cap US equities does not support")
A("a market-neutral strategy that is distinguishable from zero after realistic")
A("costs over this period. The signal that did exist decayed steadily from 2010")
A("onward. The contribution of this work is the measurement framework and the")
A("dated, quantified decay result rather than a deployable strategy.")

summary = "\n".join(lines)
with open("v21_research_summary.md", "w") as f:
    f.write(summary)
print(summary)
print("\n" + "=" * 70)
print("saved to v21_research_summary.md")

In [ ]:
# ============================================================
# RESULTS DIGEST -- machine-readable summary of this run
# ============================================================
print("<<<<< V13 DIGEST START >>>>>")
print(json.dumps(REPORT, indent=1, default=str))
print("<<<<< V13 DIGEST END >>>>>")
print(f"\n{len(REPORT)} keys recorded.")

with open("v13_digest.json", "w") as f:
    json.dump(REPORT, f, indent=1, default=str)
print("also saved to v13_digest.json")

---
# Appendix A — Research Log

Every experiment run during development, and what each concluded. Failures are
included: several of the most useful findings came from ideas that did not
work, and a log that only records successes is a sales document.

## Measurement biases found and corrected

| # | Bias | Effect |
|---|---|---|
| 1 | Overlapping daily observations treated as monthly portfolio returns | Sharpe inflated **1.8×**, turnover understated |
| 2 | Z-scored features used as raw inputs to a ratio | Signal sign flipped on ~50% of rows |
| 3 | Cross-sectional mean of a z-score used as a regime input | Input had standard deviation 0.000000 |
| 4 | IC function converting undefined correlations to 0.0 | Five of nine years of a baseline silently unmeasured |
| 5 | Sector taxonomy mismatch across two label sets | Every sector split in half; peer groups halved |
| 6 | Benchmark misaligned by one period against forward returns | Manufactured **+5.84%/yr** of phantom alpha |
| 7 | Dollar-neutral construction carrying −0.21 uncompensated beta | Cost ~3%/yr, masked a positive alpha |

## Signal research

**Horizon scan.** 114 feature-horizon combinations at 5, 10, 21, 42, 63 and
126 days. **Zero** cleared a Bonferroni-corrected significance bar. The
strongest single feature was 5-day reversal at IC +0.0179 (t = 2.38), which
does not clear the corrected threshold of |t| ≈ 2.9.

**Model capacity.** Swept tree depth 1 through 5. Out-of-sample IC was
indistinguishable from zero at every setting, and the *deepest* model
generalised best — so capacity was never the constraint. Reducing model
complexity could not rescue a feature set with no signal in it.

**Relationship stability.** Rolling coefficient vectors have a directional
half-life of **12.9 months**; at a 24-month lag the cosine similarity between
fitted models is 0.11. The relationship genuinely rotates, confirming that
models go stale — but short training windows still underperformed, because at
this signal-to-noise the estimation error from a short window exceeds the
staleness it avoids.

**Training window and refresh cadence.** 24 combinations of lookback (6 months
to expanding) against use period (1 to 12 months). Marginal effects favoured
an expanding window refreshed quarterly.

**Signal combination.** Equal weighting, IC weighting, covariance-adjusted
weighting, family de-duplication and best-single were compared walk-forward.
**Equal weighting won decisively.** IC-weighting scored −0.0185 against
+0.0117 for equal — fitted weights overfit, exactly as suspected. A synthetic
test had predicted IC-weighting would help by +0.0022, which makes this a
clean case of a simulation being too optimistic about parameter stability.

## Construction research

**Weighting scheme.** Quintile, full-proportional, and trimmed-and-smoothed
proportional. Mean net Sharpe −0.59, −0.36 and −0.14 respectively:
**+0.45 of Sharpe from construction alone**, with no change to the signal.

**Beta neutralisation.** Realised beta −0.21 → +0.07, return −0.93% → +1.96%,
drawdown −31.5% → −14.6%.

**Risk-parity sizing.** Equalising each name's risk contribution: volatility
fell a quarter while return held, worth about +0.22 of Sharpe.

**Leverage.** Volatility targeting was levering a zero-return book 1.83×,
multiplying cost from 4.28% to 10.63%/yr — impact scales as dollars^1.5, so
cost grows faster than leverage. Capped at 1.0×, then replaced with one-sided
targeting that scales down but never up.

## Ideas tested and rejected

| Idea | Result |
|---|---|
| Higher trading frequency | Break-even IC rises to 0.376 weekly against 0.0118 achieved — 32× short |
| Higher-frequency training data | No gain. Overlapping samples add rows, not information; shorter horizons add noise as fast as data |
| Drawdown de-risking overlay | Return autocorrelation −0.021 (t −0.28): no clustering, so no Sharpe benefit. Cuts drawdown at proportional cost to return |
| Single-name risk caps | No effect at 5%, 2% or 1% — the book is already too diffuse for caps to bind |
| Downside-volatility sizing | Correlates +0.92 with total volatility: very nearly the same book |
| Leg risk-parity | Sharpe +0.01 — the legs are already balanced once names are risk-weighted |
| Sign-flipping the signal | Looked like a 2-point IC gain in-sample and reversed out of sample |
| Fitted signal weights | Overfit; equal weighting is better |

## The central finding

Publicly available daily price data on large-cap US equities does not support
a market-neutral strategy distinguishable from zero after realistic costs over
this period. The signal that did exist **decayed steadily** — +4.14% alpha in
2010–2014, +0.64% in 2015–2019, −1.26% in 2020–2023 — which is the expected
fate of a published anomaly as it becomes widely traded.

Where the loss comes from is precise and worth stating: gross return near
zero, minus roughly 10%/yr of trading costs at institutional size. Both halves
are measured rather than assumed, and the capacity analysis shows the cost
half becomes manageable below about $100M.

## What would move this forward

Not another model. The remaining constraints are data:

- **Fundamentals with real history.** The free source reaches back about five
  quarters, which is why valuation and quality features were dropped on the
  coverage gate.
- **A universe with larger anomalies.** Small caps show stronger effects,
  though with worse costs — the capacity analysis here gives the framework to
  evaluate that trade directly.
- **Intraday data**, where measurable effects are larger relative to costs.